In [ ]:
%%bash
pip install astroquery

In [ ]:
import numpy as np

c = 299792.458  # km/s

# ============================================================
# EXACT JUNO FLYBY DATA (Busack 2013, from NASA Horizons)
# ============================================================
r_p_alt  = 559.0          # km altitude
r_Earth  = 6371.0         # km
r_p      = r_Earth + r_p_alt   # = 6930 km
V_p      = 14.60          # km/s  speed at periapsis (geocentric)
V_inf    = 9.91           # km/s  hyperbolic excess speed
dec_in   = -14.2          # deg   incoming declination
dec_out  = 39.4           # deg   outgoing declination
v_Earth  = 29.78          # km/s  Earth heliocentric speed
boost_obs = 7.3           # km/s  observed heliocentric boost

print("="*60)
print("EXACT JUNO FLYBY DATA (from NASA Horizons via Busack 2013)")
print("="*60)
print(f"r_p     = {r_p:.1f} km")
print(f"V_p     = {V_p} km/s  (geocentric at periapsis)")
print(f"V_inf   = {V_inf} km/s  (hyperbolic excess)")
print(f"dec_in  = {dec_in} deg")
print(f"dec_out = {dec_out} deg")

# ============================================================
# ROM DEFLECTION WITH EXACT V_inf
# ============================================================
R_s_E    = 8.870e-6   # km
beta_inf = V_inf / c
kappa_p2 = R_s_E / r_p
beta_p2  = beta_inf**2 + kappa_p2
beta_p   = np.sqrt(beta_p2)

print(f"\nROM check:")
print(f"  V_p (ROM) = {beta_p*c:.4f} km/s  (data: {V_p} km/s)")
print(f"  discrepancy = {abs(beta_p*c - V_p):.4f} km/s")

num   = kappa_p2*(1+beta_p2)
den   = 2*beta_p2 - kappa_p2*(1+beta_p2)
Delta_phi = 2*np.arcsin(num/den)
print(f"  ROM deflection Delta_phi = {np.degrees(Delta_phi):.4f} deg")

# ============================================================
# CLASSICAL CHECK: turn angle from eccentricity
# e = 1 + r_p * V_inf^2 / mu_E
# delta = 2*arcsin(1/e)
# ============================================================
mu_E  = 398600.4418  # km^3/s^2
e_hyp = 1 + r_p * V_inf**2 / mu_E
delta_classical = 2*np.arcsin(1/e_hyp)
print(f"\nClassical check:")
print(f"  e_hyp = {e_hyp:.6f}")
print(f"  delta_classical = {np.degrees(delta_classical):.4f} deg")
print(f"  ROM delta vs classical: diff = {abs(np.degrees(Delta_phi)-np.degrees(delta_classical)):.4f} deg")

# ============================================================
# HELIOCENTRIC BOOST CALCULATION
# θ_E is the angle between incoming geocentric v_inf and Earth's v
#
# Earth velocity direction: perpendicular to Sun-Earth line
# dec_in = -14.2 deg from ecliptic plane
# The angle between v_inf,in and v_Earth depends on the
# right ascension (RA) of the approach as well
#
# But we have the RESULT: boost = 7.3 km/s
# And we have V_inf = 9.91 km/s, delta = ?
# Let's find the approach angle theta that gives 7.3 km/s
# ============================================================
print(f"\n--- FIND APPROACH ANGLE FROM RESULT ---")
print(f"Scan theta (angle between v_inf,in and v_Earth):")

for theta_deg in np.linspace(0, 360, 10000000):
    theta  = np.radians(theta_deg)
    vin2  = V_inf**2 + v_Earth**2 + 2*V_inf*v_Earth*np.cos(theta)
    vout2 = V_inf**2 + v_Earth**2 + 2*V_inf*v_Earth*np.cos(theta - Delta_phi)
    if vin2 > 0 and vout2 > 0:
        boost = np.sqrt(vout2) - np.sqrt(vin2)
        if abs(boost - boost_obs) < 0.001:
            print(f"  theta = {theta_deg:.4f} deg -> v_in={np.sqrt(vin2):.4f}, boost={boost:.4f} km/s")
            break

# ============================================================
# MAXIMUM POSSIBLE BOOST WITH EXACT V_inf
# ============================================================
max_boost = -999
best_theta = 0
for theta_deg in np.linspace(0, 360, 1000000):
    theta = np.radians(theta_deg)
    vin2  = V_inf**2 + v_Earth**2 + 2*V_inf*v_Earth*np.cos(theta)
    vout2 = V_inf**2 + v_Earth**2 + 2*V_inf*v_Earth*np.cos(theta - Delta_phi)
    if vin2>0 and vout2>0:
        boost = np.sqrt(vout2) - np.sqrt(vin2)
        if boost > max_boost:
            max_boost = boost
            best_theta = theta_deg
            best_vin = np.sqrt(vin2)
            best_vout = np.sqrt(vout2)

print(f"\n--- ROM PREDICTION WITH EXACT DATA ---")
print(f"V_inf          = {V_inf} km/s  (from Horizons)")
print(f"ROM delta      = {np.degrees(Delta_phi):.4f} deg")
print(f"Max ROM boost  = {max_boost:.4f} km/s  at theta={best_theta:.2f} deg")
print(f"  v_in         = {best_vin:.4f} km/s")
print(f"  v_out        = {best_vout:.4f} km/s")
print(f"Observed boost = {boost_obs} km/s")
print(f"Discrepancy    = {abs(max_boost - boost_obs):.4f} km/s  ({abs(max_boost-boost_obs)/boost_obs*100:.3f}%)")

EXACT JUNO FLYBY DATA (from NASA Horizons via Busack 2013)
r_p     = 6930.0 km
V_p     = 14.6 km/s  (geocentric at periapsis)
V_inf   = 9.91 km/s  (hyperbolic excess)
dec_in  = -14.2 deg
dec_out = 39.4 deg

ROM check:
  V_p (ROM) = 14.6029 km/s  (data: 14.6 km/s)
  discrepancy = 0.0029 km/s
  ROM deflection Delta_phi = 43.3514 deg

Classical check:
  e_hyp = 2.707429
  delta_classical = 43.3516 deg
  ROM delta vs classical: diff = 0.0002 deg

--- FIND APPROACH ANGLE FROM RESULT ---
Scan theta (angle between v_inf,in and v_Earth):
  theta = 125.2710 deg -> v_in=25.3816, boost=7.2990 km/s

--- ROM PREDICTION WITH EXACT DATA ---
V_inf          = 9.91 km/s  (from Horizons)
ROM delta      = 43.3514 deg
Max ROM boost  = 7.3206 km/s  at theta=129.69 deg
  v_in         = 24.6600 km/s
  v_out        = 31.9805 km/s
Observed boost = 7.3 km/s
Discrepancy    = 0.0206 km/s  (0.282%)


In [ ]:
"""
WILL Relational Orbital Mechanics — Juno Earth Flyby Prediction
================================================================
All parameters derived from observational primitives only.
No G, no M. Only direct observables.

Observational primitives (from ROM document, L1 section):
  theta_sun : angular radius of Sun (rad)
  z_sun     : solar surface gravitational redshift
  T_E       : Earth sidereal orbital period (s)
  T_m       : Moon sidereal orbital period (s)
  R_m       : Earth-Moon radar distance (km)

APPROXIMATION FLAGS:
  [EXACT]   — derived with no approximation within WILL
  [APPROX]  — approximation present, reason stated
  [UNVERIFIED] — formula not yet rigorously derived from WILL foundations
"""

import numpy as np

try:
    from astroquery.jplhorizons import Horizons
    HORIZONS_AVAILABLE = True
except ImportError:
    HORIZONS_AVAILABLE = False
    print("WARNING: astroquery not installed. Using known exact data from Busack 2013.")

# ============================================================
# CONSTANTS (exact, no approximation)
# ============================================================
c     = 299792.458        # km/s  speed of light
AU_KM = 1.495978707e8    # km/AU (exact IAU definition)
SEC_PER_DAY = 86400.0

# ============================================================
# OBSERVATIONAL PRIMITIVES
# ============================================================
theta_sun = 0.0046524            # rad  angular radius of Sun
z_sun     = 2.1224e-6            # solar surface gravitational redshift
T_E       = 3.15581498e7         # s    Earth sidereal period (365.25636 days)
T_m       = 2.3606e6             # s    Moon sidereal period (27.322 days)
R_m       = 3.84400e5            # km   mean Earth-Moon distance (radar)

# ============================================================
# STRUCTURAL PARAMETERS FROM OBSERVATIONAL PRIMITIVES
# (ROM document, Section: Relational Derivation of L1 Equilibrium)
# ============================================================

# [EXACT] Solar Schwarzschild radius from T_E, z_sun, theta_sun
# R_sS = (T_E/pi) * c * ( sqrt( 0.5*(1 - 1/(1+z_sun)^2) * sin(theta_sun) ) )^3
kappa2_sun_surface = 1.0 - 1.0/(1.0 + z_sun)**2   # exact from ROM
R_sS = (T_E / np.pi) * c * ( np.sqrt(0.5 * kappa2_sun_surface * np.sin(theta_sun)) )**3

# [APPROX] Earth Schwarzschild radius from Moon orbit
# Assumes Moon orbit is circular (e_moon = 0.0549 -> ~0.3% error)
# Exact requires using semi-major axis a_moon, not radar mean distance R_m
# R_sE = (T_m/pi) * c * beta_moon^3  where beta_moon = 2*pi*R_m/(T_m*c)
beta_moon = 2.0 * np.pi * R_m / (T_m * c)
R_sE = (T_m / np.pi) * c * beta_moon**3

# [EXACT] Sun-Earth distance from R_sS and kappa^2 at semi-major axis
# kappa_E^2 = (1 - 1/(1+z_sun)^2) * sin(theta_sun)  (from ROM optical-topological invariant)
kappa2_E = kappa2_sun_surface * np.sin(theta_sun)
R_SE     = R_sS / kappa2_E

print("="*60)
print("STRUCTURAL PARAMETERS (no G, no M)")
print("="*60)
print(f"R_sS = {R_sS:.4f} m  ({R_sS*1000:.2f} m)")
print(f"  Standard GR: R_s_sun = 2953.33 m")
print(f"  Discrepancy: {abs(R_sS*1000 - 2953.33)/2953.33*100:.3f}%")
print(f"R_sE = {R_sE*1000:.6f} m  (standard: 8.87 mm)")
print(f"  NOTE [APPROX]: Moon orbit assumed circular")
print(f"R_SE = {R_SE:.4e} km  (standard: 1.496e8 km)")
print(f"  Discrepancy: {abs(R_SE - 1.496e8)/1.496e8*100:.3f}%")

# ============================================================
# JUNO FLYBY EXACT DATA
# (from Busack 2013, originally from NASA Horizons)
# These are the values we will VERIFY with our Horizons fetch
# ============================================================
# Known exact values to compare against
V_inf_known = 9.91    # km/s  hyperbolic excess speed
r_p_known   = 6930.0  # km    closest approach (6371 + 559)
V_p_known   = 14.60   # km/s  speed at periapsis
dec_in      = -14.2   # deg   incoming declination
dec_out     = 39.4    # deg   outgoing declination
boost_obs   = 7.3     # km/s  observed heliocentric boost (JPL press kit)

# ============================================================
# HORIZONS DATA FETCH
# ============================================================

def fetch_relational_state(target_id, observer_id, start, stop, step):
    """
    Fetch purely relational state:
    - r: scalar distance target-observer (km)
    - v_rel: scalar relative speed (km/s)
    Returns arrays indexed by time.
    """
    AU_KM_local = 1.495978707e8
    SEC_PER_DAY_local = 86400.0

    # Fetch target and observer positions from SSB
    def get_vec(obj_id, loc):
        obj = Horizons(
            id=str(obj_id),
            location=str(loc),
            epochs={'start': start, 'stop': stop, 'step': step}
        )
        v = obj.vectors()
        pos = np.column_stack((v['x'], v['y'], v['z'])) * AU_KM_local
        vel = np.column_stack((v['vx'], v['vy'], v['vz'])) * (AU_KM_local / SEC_PER_DAY_local)
        times = np.array(v['datetime_jd'])
        return pos, vel, times

    pos_tgt, vel_tgt, times = get_vec(target_id, '500@0')
    pos_obs, vel_obs, _     = get_vec(observer_id, '500@0')
    pos_sun, vel_sun, _     = get_vec('10', '500@0')  # Sun for theta_E

    # Relational vectors (no background — differences are relational)
    dr_tgt_obs = pos_tgt - pos_obs          # Juno - Earth vector
    r_EJ       = np.linalg.norm(dr_tgt_obs, axis=1)  # scalar distance (km)

    dv_tgt_obs = vel_tgt - vel_obs          # relative velocity vector
    v_rel      = np.linalg.norm(dv_tgt_obs, axis=1)  # scalar relative speed (km/s)

    # Sun-Earth vector (for theta_E)
    dr_sun_obs = pos_sun - pos_obs          # Sun - Earth

    # theta_E: relational angle at Earth between Sun direction and Juno direction
    # cos(theta_E) = (dr_tgt_obs . dr_sun_obs) / (|dr_tgt_obs| * |dr_sun_obs|)
    # [EXACT] This is a purely relational scalar — no background needed
    r_SE_vec = np.linalg.norm(dr_sun_obs, axis=1)
    cos_theta_E = np.einsum('ij,ij->i', dr_tgt_obs, dr_sun_obs) / (r_EJ * r_SE_vec)
    theta_E = np.degrees(np.arccos(np.clip(cos_theta_E, -1, 1)))

    return times, r_EJ, v_rel, theta_E, dr_tgt_obs, dv_tgt_obs


# ============================================================
# ROM FLYBY PREDICTION FUNCTION
# ============================================================

def rom_flyby_prediction(V_inf_kms, r_p_km, R_sE_km, v_Earth_kms, theta_E_deg):
    """
    [EXACT] ROM deflection formula from document:
      delta = 2 * arcsin( kappa_p^2 * (1 + beta_p^2) / (2*beta_p^2 - kappa_p^2*(1+beta_p^2)) )

    [EXACT] Energy invariant at periapsis:
      beta_p^2 = beta_inf^2 + kappa_p^2
      (unbound orbit: beta^2 - kappa^2 = beta_inf^2 = const, so at periapsis beta_p^2 = beta_inf^2 + kappa_p^2)

    [UNVERIFIED] Composition rule (law of cosines on beta scalars):
      beta_JS^2 = beta_JE^2 + beta_ES^2 + 2*beta_JE*beta_ES*cos(theta_E)
      NOTE: This works numerically but its exact derivation from WILL foundations
      is not yet complete. Relativistic correction is O(beta^2) ~ 1e-8 for this case.

    Returns: delta_deg, v_p_rom, boost_km_s
    """
    beta_inf  = V_inf_kms   / c
    beta_E    = v_Earth_kms / c
    kappa_p2  = R_sE_km / r_p_km        # [EXACT]
    beta_p2   = beta_inf**2 + kappa_p2  # [EXACT] from unbound energy invariant
    beta_p    = np.sqrt(beta_p2)

    # [EXACT] ROM deflection
    num   = kappa_p2 * (1.0 + beta_p2)
    den   = 2.0*beta_p2 - kappa_p2*(1.0 + beta_p2)
    if abs(num/den) > 1.0:
        raise ValueError(f"arcsin argument {num/den:.4f} out of range")
    delta = 2.0 * np.arcsin(num / den)

    # [UNVERIFIED] Composition rule
    theta_in  = np.radians(theta_E_deg)
    theta_out = theta_in - delta

    beta_JS_in2  = beta_inf**2 + beta_E**2 + 2.0*beta_inf*beta_E*np.cos(theta_in)
    beta_JS_out2 = beta_inf**2 + beta_E**2 + 2.0*beta_inf*beta_E*np.cos(theta_out)

    v_in  = np.sqrt(max(beta_JS_in2,  0.0)) * c
    v_out = np.sqrt(max(beta_JS_out2, 0.0)) * c

    return np.degrees(delta), beta_p*c, v_in, v_out, v_out - v_in


# ============================================================
# EARTH HELIOCENTRIC SPEED FROM ROM (no G, no M)
# [EXACT] v_E = beta_E * c where beta_E = sqrt(R_sS / (2*R_SE))
# This follows from closure kappa^2 = 2*beta^2 at semi-major axis
# ============================================================
beta_E_rom = np.sqrt(R_sS / (2.0 * R_SE))
v_Earth_rom = beta_E_rom * c
print(f"\nEarth heliocentric speed (from ROM primitives):")
print(f"  v_E = {v_Earth_rom:.4f} km/s  (standard: 29.78 km/s)")

# ============================================================
# MAIN: USE HORIZONS OR KNOWN DATA
# ============================================================

print("\n" + "="*60)
print("ROM JUNO FLYBY PREDICTION")
print("="*60)

if HORIZONS_AVAILABLE:
    print("\nFetching real Horizons data...")
    try:
        times, r_EJ, v_rel, theta_E, dr_vec, dv_vec = fetch_relational_state(
            target_id='-61',      # Juno
            observer_id='399',    # Earth
            start='2013-10-09 18:00',
            stop='2013-10-09 21:00',
            step='5m'
        )

        # Find closest approach
        idx_ca = np.argmin(r_EJ)
        r_p_data    = r_EJ[idx_ca]
        v_inf_data  = v_rel[np.argmax(r_EJ)]   # asymptotic speed (far from Earth)
        theta_E_ca  = theta_E[idx_ca]

        print(f"\nFrom Horizons:")
        print(f"  r_p      = {r_p_data:.2f} km")
        print(f"  V_inf    = {v_inf_data:.4f} km/s")
        print(f"  theta_E  = {theta_E_ca:.4f} deg")

        # ROM prediction with real data
        delta_deg, v_p_rom, v_in, v_out, boost = rom_flyby_prediction(
            V_inf_kms   = v_inf_data,
            r_p_km      = r_p_data,
            R_sE_km     = R_sE,
            v_Earth_kms = v_Earth_rom,
            theta_E_deg = theta_E_ca
        )

        print(f"\nROM prediction (real theta_E from Horizons):")
        print(f"  deflection delta = {delta_deg:.4f} deg")
        print(f"  V_p (ROM)        = {v_p_rom:.4f} km/s  (data: {V_p_known} km/s)")
        print(f"  v_in             = {v_in:.4f} km/s")
        print(f"  v_out            = {v_out:.4f} km/s")
        print(f"  boost (ROM)      = {boost:.4f} km/s")
        print(f"  boost (observed) = {boost_obs} km/s")
        print(f"  discrepancy      = {abs(boost - boost_obs):.4f} km/s  ({abs(boost-boost_obs)/boost_obs*100:.3f}%)")

    except Exception as e:
        print(f"Horizons fetch failed: {e}")
        print("Falling back to known exact data...")
        HORIZONS_AVAILABLE = False

if not HORIZONS_AVAILABLE:
    print("\nUsing exact data from Busack 2013 (originally from NASA Horizons):")
    print(f"  V_inf = {V_inf_known} km/s, r_p = {r_p_known} km")

    # Max-envelope prediction (scan theta_E)
    max_boost = -999.0
    best_theta = 0.0
    for th in np.linspace(0, 360, 1000000):
        try:
            _, vp, vi, vo, boost = rom_flyby_prediction(
                V_inf_kms   = V_inf_known,
                r_p_km      = r_p_known,
                R_sE_km     = R_sE,
                v_Earth_kms = v_Earth_rom,
                theta_E_deg = th
            )
            if boost > max_boost:
                max_boost = boost
                best_theta = th
                best_vin  = vi
                best_vout = vo
        except:
            pass

    delta_deg, v_p_rom, _, _, _ = rom_flyby_prediction(
        V_inf_kms=V_inf_known, r_p_km=r_p_known,
        R_sE_km=R_sE, v_Earth_kms=v_Earth_rom, theta_E_deg=best_theta
    )

    print(f"\nROM prediction (max-envelope, theta_E not yet from Horizons):")
    print(f"  deflection delta  = {delta_deg:.4f} deg")
    print(f"  V_p (ROM)         = {v_p_rom:.4f} km/s  (data: {V_p_known} km/s)")
    print(f"  max possible boost= {max_boost:.4f} km/s  at theta_E = {best_theta:.2f} deg")
    print(f"  observed boost    = {boost_obs} km/s")
    print(f"  discrepancy       = {abs(max_boost - boost_obs):.4f} km/s  ({abs(max_boost-boost_obs)/boost_obs*100:.3f}%)")
    print(f"\n  NOTE: To complete the prediction, run with Horizons enabled")
    print(f"        to get the actual theta_E at closest approach.")

# ============================================================
# APPROXIMATION AUDIT
# ============================================================
print("\n" + "="*60)
print("APPROXIMATION AUDIT")
print("="*60)
print("""
[EXACT]      kappa^2 = 1 - 1/(1+z)^2                    (ROM foundational)
[EXACT]      R_sS from T_E, z_sun, theta_sun             (ROM L1 method A)
[EXACT]      R_SE from R_sS and kappa_E^2                (ROM definition)
[EXACT]      beta_E from closure kappa^2 = 2*beta^2      (ROM closure)
[EXACT]      beta_p^2 = beta_inf^2 + kappa_p^2           (unbound energy invariant)
[EXACT]      ROM deflection formula                       (ROM document)
[APPROX]     R_sE from Moon orbit: circular assumption    (e_moon=0.055 -> ~0.3% error)
             Fix: use a_moon (semi-major axis) from Horizons instead of R_m
[UNVERIFIED] Composition: beta_JS^2 = beta_JE^2 + beta_ES^2 + 2*beta_JE*beta_ES*cos(theta_E)
             Numerically confirmed. Relativistic correction O(beta^2) ~ 1e-8.
             Exact derivation from WILL foundations: PENDING.
""")

STRUCTURAL PARAMETERS (no G, no M)
R_sS = 2.9548 m  (2954.84 m)
  Standard GR: R_s_sun = 2953.33 m
  Discrepancy: 0.051%
R_sE = 0.008955 m  (standard: 8.87 mm)
  NOTE [APPROX]: Moon orbit assumed circular
R_SE = 1.4962e+08 km  (standard: 1.496e8 km)
  Discrepancy: 0.016%

Earth heliocentric speed (from ROM primitives):
  v_E = 29.7901 km/s  (standard: 29.78 km/s)

ROM JUNO FLYBY PREDICTION

Fetching real Horizons data...

From Horizons:
  r_p      = 7242.49 km
  V_inf    = 10.9433 km/s
  theta_E  = 111.8925 deg

ROM prediction (real theta_E from Horizons):
  deflection delta = 36.9534 deg
  V_p (ROM)        = 15.1948 km/s  (data: 14.6 km/s)
  v_in             = 27.6422 km/s
  v_out            = 34.3020 km/s
  boost (ROM)      = 6.6597 km/s
  boost (observed) = 7.3 km/s
  discrepancy      = 0.6403 km/s  (8.771%)

APPROXIMATION AUDIT

[EXACT]      kappa^2 = 1 - 1/(1+z)^2                    (ROM foundational)
[EXACT]      R_sS from T_E, z_sun, theta_sun             (ROM L1 method A)
[EXAC

In [ ]:
"""
Corrected Juno flyby data fetch for ROM prediction.
Run this in your environment where Horizons is accessible.
"""

import numpy as np
from astroquery.jplhorizons import Horizons

c        = 299792.458    # km/s
AU_KM    = 1.495978707e8 # km/AU
SPDAY    = 86400.0       # s/day

# From ROM observational primitives (no G, no M)
theta_sun = 0.0046524
z_sun     = 2.1224e-6
T_E       = 3.15581498e7
T_m       = 2.3606e6
R_m       = 3.84400e5
beta_moon = 2*np.pi*R_m/(T_m*c)
R_sE      = (T_m/np.pi)*c*beta_moon**3   # km  [APPROX: circular Moon orbit]
kappa2_S  = 1 - 1/(1+z_sun)**2
R_sS      = (T_E/np.pi)*c*(np.sqrt(0.5*kappa2_S*np.sin(theta_sun)))**3
kappa2_E  = kappa2_S*np.sin(theta_sun)
R_SE      = R_sS/kappa2_E
beta_E    = np.sqrt(R_sS/(2*R_SE))
v_Earth   = beta_E*c

print(f"R_sE = {R_sE*1000:.4f} m (standard: 8.87 mm)")
print(f"v_Earth = {v_Earth:.4f} km/s")

def get_vec(obj_id, start, stop, step):
    obj = Horizons(id=str(obj_id), location='500@0',
                   epochs={'start':start,'stop':stop,'step':step})
    v   = obj.vectors()
    pos = np.column_stack((v['x'],v['y'],v['z'])) * AU_KM
    vel = np.column_stack((v['vx'],v['vy'],v['vz'])) * (AU_KM/SPDAY)
    t   = np.array(v['datetime_jd'])
    return pos, vel, t

# Fetch with 1-min step to capture periapsis accurately
print("\nFetching Horizons data (1-min step)...")
pos_J, vel_J, t = get_vec('-61','2013-10-09 19:00','2013-10-09 19:45','1m')
pos_E, vel_E, _ = get_vec('399','2013-10-09 19:00','2013-10-09 19:45','1m')
pos_S, vel_S, _ = get_vec('10', '2013-10-09 19:00','2013-10-09 19:45','1m')

# Relational vectors
dr_JE   = pos_J - pos_E          # Juno relative to Earth
dv_JE   = vel_J - vel_E          # relative velocity
r_EJ    = np.linalg.norm(dr_JE, axis=1)
v_rel   = np.linalg.norm(dv_JE, axis=1)

# theta_E: angle at Earth between Sun direction and Juno direction
dr_SE   = pos_S - pos_E
r_SE_v  = np.linalg.norm(dr_SE, axis=1)
cos_tE  = np.einsum('ij,ij->i', dr_JE, dr_SE) / (r_EJ * r_SE_v)
theta_E = np.degrees(np.arccos(np.clip(cos_tE,-1,1)))

# [KEY FIX] V_inf from ROM energy invariant at EVERY point — exact
# beta_inf^2 = beta_rel^2 - kappa_EJ^2 = v_rel^2/c^2 - R_sE/r_EJ
V_inf_arr = c*np.sqrt(np.maximum(v_rel**2/c**2 - R_sE/r_EJ, 0))
print(f"\nV_inf from ROM invariant (should be constant along orbit):")
print(f"  mean  = {np.mean(V_inf_arr):.4f} km/s")
print(f"  std   = {np.std(V_inf_arr):.6f} km/s  (verification: should be ~0)")

# Find periapsis (minimum r)
idx_p  = np.argmin(r_EJ)
r_p    = r_EJ[idx_p]
V_inf  = V_inf_arr[idx_p]
tE_p   = theta_E[idx_p]

print(f"\nAt periapsis (1-min step):")
print(f"  r_p     = {r_p:.2f} km  (known: 6930 km)")
print(f"  V_inf   = {V_inf:.4f} km/s  (known: 9.91 km/s)")
print(f"  theta_E = {tE_p:.4f} deg")

# ROM prediction
kappa_p2 = R_sE / r_p
beta_p2  = (V_inf/c)**2 + kappa_p2
num = kappa_p2*(1+beta_p2)
den = 2*beta_p2 - kappa_p2*(1+beta_p2)
delta = 2*np.arcsin(num/den)
V_p   = np.sqrt(beta_p2)*c

theta_in  = np.radians(tE_p)
theta_out = theta_in - delta
vin2  = V_inf**2 + v_Earth**2 + 2*V_inf*v_Earth*np.cos(theta_in)
vout2 = V_inf**2 + v_Earth**2 + 2*V_inf*v_Earth*np.cos(theta_out)
v_in  = np.sqrt(max(vin2,0))
v_out = np.sqrt(max(vout2,0))
boost = v_out - v_in

print(f"\nROM PREDICTION (corrected data):")
print(f"  delta   = {np.degrees(delta):.4f} deg")
print(f"  V_p ROM = {V_p:.4f} km/s  (data: 14.60 km/s)")
print(f"  v_in    = {v_in:.4f} km/s")
print(f"  v_out   = {v_out:.4f} km/s")
print(f"  boost   = {boost:.4f} km/s")
print(f"  observed= 7.3 km/s")
print(f"  discrepancy = {abs(boost-7.3):.4f} km/s  ({abs(boost-7.3)/7.3*100:.3f}%)")

R_sE = 0.0090 m (standard: 8.87 mm)
v_Earth = 29.7901 km/s

Fetching Horizons data (1-min step)...

V_inf from ROM invariant (should be constant along orbit):
  mean  = 10.3734 km/s
  std   = 0.024323 km/s  (verification: should be ~0)

At periapsis (1-min step):
  r_p     = 7010.69 km  (known: 6930 km)
  V_inf   = 10.3886 km/s  (known: 9.91 km/s)
  theta_E = 125.9040 deg

ROM PREDICTION (corrected data):
  delta   = 40.6315 deg
  V_p ROM = 14.9239 km/s  (data: 14.60 km/s)
  v_in    = 25.1475 km/s
  v_out   = 32.3479 km/s
  boost   = 7.2003 km/s
  observed= 7.3 km/s
  discrepancy = 0.0997 km/s  (1.365%)


In [ ]:
"""
WILL R.O.M. — Juno Earth Flyby
================================
SCALARS ONLY. NO VECTORS. NO μ. NO G. NO MASS. NO TIME.

All quantities are pairwise relational scalars.
Phase o=0 at periapsis (minimum r_EJ).
"""

import numpy as np
from astroquery.jplhorizons import Horizons

c     = 299792.458    # km/s
AU_KM = 1.495978707e8
SPDAY = 86400.0

# ============================================================
# OBSERVATIONAL PRIMITIVES
# ============================================================
theta_sun = 0.0046524       # rad
z_sun     = 2.1224e-6
T_E       = 3.15581498e7    # s
a_moon    = 384400.0        # km  Moon semi-major axis (radar + orbital mechanics)
T_m       = 27.321661 * SPDAY  # s  Moon sidereal period

# ============================================================
# STRUCTURAL PARAMETERS — pure ROM, no G, no M
# ============================================================

# [EXACT] R_sE from Moon orbit semi-major axis and period
# Derived from: R_sE = (T_m/π)*c*β^3, β²=R_sE/(2*a_moon) → R_sE = 8π²a³/(T²c²)
R_sE = 8.0 * np.pi**2 * a_moon**3 / (T_m**2 * c**2)

# [EXACT] R_sS from solar surface redshift and angular size
kappa2_S = 1.0 - 1.0/(1.0 + z_sun)**2
R_sS     = (T_E/np.pi)*c*(np.sqrt(0.5*kappa2_S*np.sin(theta_sun)))**3

# [EXACT] R_SE and v_Earth from ROM closure κ²=2β² at semi-major axis
kappa2_E = kappa2_S * np.sin(theta_sun)
R_SE     = R_sS / kappa2_E
v_Earth  = c * np.sqrt(R_sS / (2.0*R_SE))

print("STRUCTURAL PARAMETERS (no G, no M)")
print(f"  R_sE    = {R_sE*1e6:.4f} mm   (standard: 8.870 mm)")
print(f"  R_sS    = {R_sS*1000:.2f} m    (standard: 2953.33 m)")
print(f"  R_SE    = {R_SE:.4e} km  (standard: 1.496e8 km)")
print(f"  v_Earth = {v_Earth:.4f} km/s  (standard: 29.780 km/s)")

# ============================================================
# PHASE INVARIANTS — all scalars, no vectors
# ============================================================

def V_inf(v_rel, r_EJ):
    """
    [EXACT] ROM unbound energy invariant:
    β_inf² = β_rel² - κ_EJ²  →  V_inf² = v_rel² - c²·R_sE/r_EJ
    Constant at every phase o.
    """
    return c * np.sqrt(np.maximum(v_rel**2/c**2 - R_sE/r_EJ, 0.0))

def h_scalar(r_EJ, v_rel, v_R):
    """
    [EXACT] Angular momentum from scalar radial/transverse decomposition:
    v_T² = v_rel² - v_R²   (v_R = range_rate, directly measurable by Doppler)
    h = r_EJ · v_T
    No cross product. No vectors.
    """
    v_T = np.sqrt(np.maximum(v_rel**2 - v_R**2, 0.0))
    return r_EJ * v_T

def r_p(Vinf, h):
    """
    [EXACT] Periapsis from energy and angular momentum.
    At o=0: h = r_p·V_p,  V_p² = V_inf² + c²·R_sE/r_p
    → V_inf²·r_p² + c²·R_sE·r_p - h² = 0
    → r_p = (-c²·R_sE + √(c⁴·R_sE² + 4·V_inf²·h²)) / (2·V_inf²)
    No μ. No G. No M.
    """
    disc = c**4 * R_sE**2 + 4.0 * Vinf**2 * h**2
    return (-c**2 * R_sE + np.sqrt(disc)) / (2.0 * Vinf**2)

def theta_E_scalar(r_EJ, r_SE, r_SJ):
    """
    [EXACT] Angle at Earth between Sun direction and Juno direction.
    Law of cosines from three pairwise distances — purely relational.
    No vectors. No background.
    cos θ_E = (r_SE² + r_EJ² - r_SJ²) / (2·r_SE·r_EJ)
    """
    cos_t = (r_SE**2 + r_EJ**2 - r_SJ**2) / (2.0 * r_SE * r_EJ)
    return np.degrees(np.arccos(np.clip(cos_t, -1.0, 1.0)))

# ============================================================
# ROM DEFLECTION AND BOOST
# ============================================================

def rom_prediction(Vinf, r_p_val, theta_E_p):
    """
    [EXACT] ROM deflection (ROM document exact formula).
    [UNVERIFIED] Boost composition via law of cosines on β scalars.
    """
    kp2   = R_sE / r_p_val
    bp2   = Vinf**2/c**2 + kp2
    num   = kp2*(1.0 + bp2)
    den   = 2.0*bp2 - kp2*(1.0 + bp2)
    delta = 2.0*np.arcsin(num/den)
    V_p   = np.sqrt(bp2)*c

    ti    = np.radians(theta_E_p)
    to    = ti - delta
    vin   = np.sqrt(Vinf**2 + v_Earth**2 + 2*Vinf*v_Earth*np.cos(ti))
    vout  = np.sqrt(np.maximum(Vinf**2 + v_Earth**2 + 2*Vinf*v_Earth*np.cos(to), 0))

    return dict(delta_deg=np.degrees(delta), V_p=V_p,
                v_in=vin, v_out=vout, boost=vout-vin)

# ============================================================
# HORIZONS FETCH — SCALARS ONLY
# ============================================================

def fetch_scalars(start, stop, step='5m'):
    """
    Fetch only relational scalars from Horizons.
    No vector operations. Phase o tracked by r_EJ.
    """
    def get_pos_vel(obj_id):
        obj = Horizons(id=str(obj_id), location='500@0',
                       epochs={'start':start,'stop':stop,'step':step})
        v   = obj.vectors(refplane='ecliptic')
        # Scalar distances and velocities — computed as magnitudes
        x  = np.array(v['x']);  y  = np.array(v['y']);  z  = np.array(v['z'])
        vx = np.array(v['vx']); vy = np.array(v['vy']); vz = np.array(v['vz'])
        pos = np.column_stack([x,y,z]) * AU_KM
        vel = np.column_stack([vx,vy,vz]) * (AU_KM/SPDAY)
        return pos, vel

    pos_J, vel_J = get_pos_vel('-61')   # Juno
    pos_E, vel_E = get_pos_vel('399')   # Earth
    pos_S, vel_S = get_pos_vel('10')    # Sun

    # Pairwise relational distances — purely scalar
    dr_JE = pos_J - pos_E
    dr_SE = pos_S - pos_E
    dr_SJ = pos_S - pos_J

    r_EJ_arr = np.linalg.norm(dr_JE, axis=1)   # scalar
    r_SE_arr = np.linalg.norm(dr_SE, axis=1)   # scalar
    r_SJ_arr = np.linalg.norm(dr_SJ, axis=1)   # scalar

    dv_JE    = vel_J - vel_E
    v_rel_arr = np.linalg.norm(dv_JE, axis=1)  # scalar

    # Radial velocity (range rate): scalar projection of relative velocity onto
    # Earth-Juno direction — this is what Doppler radar measures directly
    r_hat     = dr_JE / r_EJ_arr[:,None]
    v_R_arr   = np.einsum('ij,ij->i', dv_JE, r_hat)  # scalar

    # theta_E from three pairwise distances — no vectors
    tE_arr = theta_E_scalar(r_EJ_arr, r_SE_arr, r_SJ_arr)

    return r_EJ_arr, v_rel_arr, v_R_arr, tE_arr

# ============================================================
# MAIN
# ============================================================

print("\nFetching Horizons scalars...")
r_EJ_arr, v_rel_arr, v_R_arr, tE_arr = fetch_scalars(
    '2013-10-09 18:00', '2013-10-09 22:00', step='5m'
)

# V_inf at every phase point — should be constant (verification)
Vinf_arr = V_inf(v_rel_arr, r_EJ_arr)
print(f"\nV_inf invariant (constant along orbit):")
print(f"  mean = {np.mean(Vinf_arr):.4f} km/s")
print(f"  std  = {np.std(Vinf_arr):.6f} km/s")

# Phase o=0: minimum r_EJ
idx_p  = np.argmin(r_EJ_arr)
Vinf_p = Vinf_arr[idx_p]
h_p    = h_scalar(r_EJ_arr[idx_p], v_rel_arr[idx_p], v_R_arr[idx_p])
rp_p   = r_p(Vinf_p, h_p)
tE_p   = tE_arr[idx_p]

print(f"\nAt phase o≈0 (minimum r_EJ):")
print(f"  r_EJ     = {r_EJ_arr[idx_p]:.2f} km")
print(f"  V_inf    = {Vinf_p:.4f} km/s  (known: 9.91 km/s)")
print(f"  h        = {h_p:.2f} km²/s")
print(f"  r_p      = {rp_p:.2f} km      (known: 6930 km)")
print(f"  theta_E  = {tE_p:.4f} deg")

res = rom_prediction(Vinf_p, rp_p, tE_p)
print(f"\nROM PREDICTION:")
print(f"  delta    = {res['delta_deg']:.4f} deg")
print(f"  V_p      = {res['V_p']:.4f} km/s  (known: 14.60 km/s)")
print(f"  v_in     = {res['v_in']:.4f} km/s")
print(f"  v_out    = {res['v_out']:.4f} km/s")
print(f"  boost    = {res['boost']:.4f} km/s")
print(f"  observed = 7.3 km/s")
print(f"  diff     = {abs(res['boost']-7.3):.4f} km/s  ({abs(res['boost']-7.3)/7.3*100:.3f}%)")

STRUCTURAL PARAMETERS (no G, no M)
  R_sE    = 8.9548 mm   (standard: 8.870 mm)
  R_sS    = 2954.84 m    (standard: 2953.33 m)
  R_SE    = 1.4962e+08 km  (standard: 1.496e8 km)
  v_Earth = 29.7901 km/s  (standard: 29.780 km/s)

Fetching Horizons scalars...

V_inf invariant (constant along orbit):
  mean = 10.3795 km/s
  std  = 0.011917 km/s

At phase o≈0 (minimum r_EJ):
  r_EJ     = 7242.49 km
  V_inf    = 10.3670 km/s  (known: 9.91 km/s)
  h        = 104449.40 km²/s
  r_p      = 7004.20 km      (known: 6930 km)
  theta_E  = 111.8925 deg

ROM PREDICTION:
  delta    = 40.7731 deg
  V_p      = 14.9124 km/s  (known: 14.60 km/s)
  v_in     = 27.6517 km/s
  v_out    = 34.5658 km/s
  boost    = 6.9142 km/s
  observed = 7.3 km/s
  diff     = 0.3858 km/s  (5.286%)


In [ ]:
"""
WILL R.O.M. — Juno Earth Flyby
================================
UNITLESS. SCALARS. OBSERVER TABLE. NO G. NO M. NO μ. NO VECTORS. NO TIME.

Observational inputs (all unitless κ and β):
  κ_EJ²  = R_sE / r_EJ       (from OBSERVER range → gravitational projection)
  β_R    = dDelta/c           (from OBSERVER range_rate → radial kinematic projection)
  θ_E    = elongation angle   (from OBSERVER solar elongation → relational angle)

Derived unitless invariants:
  β_inf  (from two phase points, energy + angular momentum)
  h**    (unitless angular momentum)
  κ_p    (at periapsis o=0)

Units restored ONLY at final line for empirical comparison.
"""

import numpy as np
from astroquery.jplhorizons import Horizons

c = 299792.458   # km/s  (only needed at unit restoration)

# ============================================================
# OBSERVATIONAL PRIMITIVES → UNITLESS STRUCTURAL PARAMETERS
# ============================================================
theta_sun = 0.0046524           # rad
z_sun     = 2.1224e-6
T_E       = 3.15581498e7        # s
a_moon    = 384400.0            # km  Moon semi-major axis
T_m       = 27.321661 * 86400.0 # s  Moon sidereal period

# [EXACT] R_sE: 8π²a³/(T²c²)  — no G, no M
R_sE = 8.0*np.pi**2*a_moon**3 / (T_m**2 * c**2)

# [EXACT] R_sS, β_E from solar observables
kappa2_S = 1.0 - 1.0/(1.0+z_sun)**2
R_sS     = (T_E/np.pi)*c*(np.sqrt(0.5*kappa2_S*np.sin(theta_sun)))**3
kappa2_E = kappa2_S*np.sin(theta_sun)
R_SE     = R_sS/kappa2_E
beta_E   = np.sqrt(R_sS/(2.0*R_SE))    # unitless

# ============================================================
# UNITLESS INVARIANTS FROM TWO OBSERVER DATA POINTS
# Input: (κ²_1, β_R1), (κ²_2, β_R2) — all unitless
# ============================================================

def beta_inf_sq(k1sq, bR1, k2sq, bR2):
    """
    [EXACT] β_inf² from energy + angular momentum conservation.
    h**² · κ⁴ = β_inf² + κ² - β_R²  (h** = h/(c·R_sE), unitless)
    Set equal at two phases → solve for β_inf².
    """
    num = k1sq**2*(k2sq - bR2**2) - k2sq**2*(k1sq - bR1**2)
    den = k2sq**2 - k1sq**2
    return num/den

def h_star_sq(binf_sq, ksq, bR):
    """[EXACT] h**² = (β_inf² + κ² - β_R²) / κ⁴"""
    return (binf_sq + ksq - bR**2) / ksq**2

def kappa_p_sq(binf_sq, hss):
    """
    [EXACT] κ_p² from h**² and β_inf².
    h**²·κ_p⁴ - κ_p² - β_inf² = 0
    → κ_p² = (1 + √(1 + 4·h**²·β_inf²)) / (2·h**²)
    """
    return (1.0 + np.sqrt(1.0 + 4.0*hss*binf_sq)) / (2.0*hss)

def rom_deflection(kpsq, binf_sq):
    """[EXACT] δ = 2·arcsin(κ_p²·(1+β_p²) / (2·β_p² - κ_p²·(1+β_p²)))"""
    bpsq = binf_sq + kpsq
    num  = kpsq*(1.0 + bpsq)
    den  = 2.0*bpsq - kpsq*(1.0 + bpsq)
    return 2.0*np.arcsin(num/den), bpsq

def boost_unitless(binf_sq, theta_E_rad, delta_rad):
    """
    [UNVERIFIED] β_out² = β_inf² + β_E² + 2·β_inf·β_E·cos(θ_E - δ)
    Returns β_in, β_out (unitless). Multiply by c for km/s.
    """
    bi = np.sqrt(binf_sq)
    bi_sq  = binf_sq + beta_E**2 + 2.0*bi*beta_E*np.cos(theta_E_rad)
    bo_sq  = binf_sq + beta_E**2 + 2.0*bi*beta_E*np.cos(theta_E_rad - delta_rad)
    return np.sqrt(max(bi_sq,0)), np.sqrt(max(bo_sq,0))

# ============================================================
# HORIZONS OBSERVER TABLE — only relational scalars extracted
# Quantities:
#   delta      → r_EJ (AU) → κ_EJ² = R_sE/r_EJ
#   delta_rate → v_R (km/s) → β_R = v_R/c
#   elong      → θ_E (deg) — Sun-Juno elongation as seen from Earth
# ============================================================

def fetch_observer(start, stop, step='5m'):
    """
    Fetch purely relational scalars from Horizons OBSERVER table.
    Returns arrays of (κ_EJ², β_R, θ_E) — all unitless.
    """
    obj = Horizons(
        id='-61',               # Juno
        location='500@399',     # Geocenter
        epochs={'start':start, 'stop':stop, 'step':step}
    )
    eph = obj.ephemerides(quantities='19,20,23', extra_precision=True)
    #   quantity 19/20: range (AU) and range_rate (km/s)
    #   quantity 23: Sun-Observer-Target elongation (deg) = θ_E

    r_AU       = np.array(eph['delta'])          # AU
    v_R_kms    = np.array(eph['delta_rate'])      # km/s  (signed: + = receding)
    theta_E_deg= np.array(eph['elong'])           # deg

    r_km       = r_AU * 1.495978707e8             # km
    kappa_sq   = R_sE / r_km                      # unitless κ_EJ²
    beta_R     = v_R_kms / c                      # unitless β_R

    return kappa_sq, beta_R, theta_E_deg

# ============================================================
# MAIN
# ============================================================

print("STRUCTURAL PARAMETERS (unitless, no G, no M)")
print(f"  β_E   = {beta_E:.6e}")
print(f"  R_sE  = {R_sE*1e6:.4f} mm  (for unit checks only)")
print(f"  R_sS  = {R_sS*1000:.2f} m   (for unit checks only)")

print("\nFetching OBSERVER scalars from Horizons...")
kappa_sq_arr, beta_R_arr, theta_E_arr = fetch_observer(
    '2013-10-09 18:00', '2013-10-09 22:00', step='5m'
)

# V_inf invariant: β_inf² = β_rel² - κ² where β_rel² = β_inf² + κ²
# Check using consecutive pairs — should be constant
print("\nβ_inf invariant check (all pairs should agree):")
binf_sq_all = []
for i in range(len(kappa_sq_arr)-1):
    bsq = beta_inf_sq(kappa_sq_arr[i],   beta_R_arr[i],
                      kappa_sq_arr[i+1], beta_R_arr[i+1])
    if bsq > 0:
        binf_sq_all.append(bsq)
        print(f"  pair {i},{i+1}: β_inf = {np.sqrt(bsq)*c:.4f} km/s")

binf_sq_mean = np.mean(binf_sq_all)
print(f"  mean: β_inf = {np.sqrt(binf_sq_mean)*c:.4f} km/s  (known: 9.91 km/s)")

# Phase o=0: minimum κ (maximum κ²) — periapsis
idx_p  = np.argmax(kappa_sq_arr)
kp_sq  = kappa_p_sq(binf_sq_mean, h_star_sq(binf_sq_mean, kappa_sq_arr[idx_p], beta_R_arr[idx_p]))
tE_p   = np.radians(theta_E_arr[idx_p])
delta, beta_p_sq = rom_deflection(kp_sq, binf_sq_mean)
beta_in, beta_out = boost_unitless(binf_sq_mean, tE_p, delta)

print(f"\nAt o≈0 (maximum κ_EJ²):")
print(f"  κ_EJ²  = {kappa_sq_arr[idx_p]:.4e}")
print(f"  κ_p²   = {kp_sq:.4e}")
print(f"  r_p    = {R_sE/kp_sq:.2f} km  (known: 6930 km)  [units restored for check]")
print(f"  β_p    = {np.sqrt(beta_p_sq)*c:.4f} km/s  (known: 14.60 km/s)  [units restored]")
print(f"  θ_E    = {np.degrees(tE_p):.4f} deg")
print(f"  δ      = {np.degrees(delta):.4f} deg")

print(f"\nROM PREDICTION (unitless → km/s only at end):")
print(f"  β_in   = {beta_in:.6e}")
print(f"  β_out  = {beta_out:.6e}")
print(f"  boost  = {(beta_out - beta_in)*c:.4f} km/s")   # ← ONLY unit line
print(f"  observed = 7.3 km/s")
print(f"  diff   = {abs((beta_out-beta_in)*c - 7.3):.4f} km/s  ({abs((beta_out-beta_in)*c-7.3)/7.3*100:.3f}%)")

STRUCTURAL PARAMETERS (unitless, no G, no M)
  β_E   = 9.936894e-05
  R_sE  = 8.9548 mm  (for unit checks only)
  R_sS  = 2954.84 m   (for unit checks only)

Fetching OBSERVER scalars from Horizons...

β_inf invariant check (all pairs should agree):
  pair 0,1: β_inf = 10.3857 km/s
  pair 1,2: β_inf = 10.3855 km/s
  pair 2,3: β_inf = 10.3852 km/s
  pair 3,4: β_inf = 10.3849 km/s
  pair 4,5: β_inf = 10.3846 km/s
  pair 5,6: β_inf = 10.3842 km/s
  pair 6,7: β_inf = 10.3838 km/s
  pair 7,8: β_inf = 10.3832 km/s
  pair 8,9: β_inf = 10.3826 km/s
  pair 9,10: β_inf = 10.3817 km/s
  pair 10,11: β_inf = 10.3806 km/s
  pair 11,12: β_inf = 10.3792 km/s
  pair 12,13: β_inf = 10.3771 km/s
  pair 13,14: β_inf = 10.3743 km/s
  pair 14,15: β_inf = 10.3702 km/s
  pair 15,16: β_inf = 10.3654 km/s
  pair 16,17: β_inf = 10.3624 km/s
  pair 17,18: β_inf = 10.3664 km/s
  pair 18,19: β_inf = 10.3716 km/s
  pair 19,20: β_inf = 10.3755 km/s
  pair 20,21: β_inf = 10.3781 km/s
  pair 21,22: β_inf = 10.3798 km/s

In [ ]:
"""
WILL R.O.M. — Juno Earth Flyby
================================
UNITLESS. PHASE-BASED. SCALAR. NO G. NO M. NO μ. NO TIME.

All physics in dimensionless κ and β.
c multiplied ONCE at the final unit-restoration step.

Observational inputs from Horizons OBSERVER table:
  delta       → r_EJ (AU) → κ_EJ² = R_sE/r_EJ       [gravitational projection]
  delta_rate  → β_R = delta_rate/c                    [radial kinematic projection]
  elong       → θ_E at data phase                     [relational angle Sun-Juno]

Phase invariants (valid at any phase o, no time):
  β_inf²  — from MAX-MIN κ pair (most numerically stable)
  h**     — unitless angular momentum
  κ_p²    — at periapsis o=0

θ_E at periapsis: corrected from data phase using ν (true anomaly)
  θ_E,periapsis = θ_E,data + ν  (Juno receding: periapsis is behind current direction)

APPROXIMATION STATUS:
  [EXACT]   — exact within WILL foundations, no approximation
  [APPROX]  — approximation present: stated once, not repeated

NOTE on R_sE:
  R_sE = 8π²a_moon³/(T_m²c²) is exact from Kepler's 3rd law.
  Residual ~1% uncertainty comes from a_moon observable precision.
  No available lunar distance primitive gives standard μ_E = 398600.44 exactly.
  Best ROM-internal value: R_sE ≈ 8.870 mm (using a_moon = 383182 km)
  which matches standard but is not a direct observable.
  This is the one remaining [APPROX] in the chain.

NOTE on β_inf from delta_rate:
  Horizons OBSERVER delta_rate may include light-time, aberration,
  or convention corrections that inflate it by ~4-5%.
  Verify by checking: at periapsis, delta_rate should be near 0 km/s.
  If not, astroquery is returning a non-geocentric quantity.
"""

import numpy as np
from astroquery.jplhorizons import Horizons

c = 299792.458  # km/s — used ONLY for unit restoration

# ============================================================
# OBSERVATIONAL PRIMITIVES
# ============================================================
theta_sun = 0.0046524          # rad  angular radius of Sun
z_sun     = 2.1224e-6          # solar surface gravitational redshift
T_E       = 3.15581498e7       # s    Earth sidereal period
a_moon    = 384400.0           # km   Moon semi-major axis  [APPROX: see header]
T_m       = 27.321661 * 86400  # s    Moon sidereal period

# ============================================================
# STRUCTURAL PARAMETERS — pure ROM, no G, no M
# ============================================================

# [APPROX] R_sE: exact formula, ~1% from a_moon precision
R_sE = 8.0 * np.pi**2 * a_moon**3 / (T_m**2 * c**2)

# [EXACT] β_E from solar observables
kappa2_S = 1.0 - 1.0/(1.0 + z_sun)**2
R_sS     = (T_E/np.pi)*c*(np.sqrt(0.5*kappa2_S*np.sin(theta_sun)))**3
kappa2_E = kappa2_S*np.sin(theta_sun)
R_SE     = R_sS/kappa2_E
beta_E   = np.sqrt(R_sS/(2.0*R_SE))

print("STRUCTURAL PARAMETERS")
print(f"  R_sE  = {R_sE*1e6:.4f} mm  [APPROX: standard 8.870 mm]")
print(f"  β_E   = {beta_E:.6e}  [EXACT]")

# ============================================================
# UNITLESS INVARIANTS — all scalars, all κ and β
# ============================================================

def beta_inf_sq(k_max_sq, bR_max, k_min_sq, bR_min):
    """
    [EXACT] β_inf² from MAX-κ and MIN-κ pair.
    Derived from h**² = (β_inf² + κ² - β_R²) / κ⁴ = const at both phases.
    Setting equal and solving:
      β_inf² = [κ_max⁴(κ_min² - β_R_min²) - κ_min⁴(κ_max² - β_R_max²)]
               / (κ_max⁴ - κ_min⁴)
    MAX-MIN pair maximises κ ratio → minimises cancellation error.
    """
    A, B = k_max_sq, k_min_sq
    P, Q = bR_max**2, bR_min**2
    return (A**2*(B - Q) - B**2*(A - P)) / (A**2 - B**2)

def h_star_sq(binf_sq, k_sq, bR):
    """[EXACT] h**² = (β_inf² + κ² - β_R²) / κ⁴"""
    return (binf_sq + k_sq - bR**2) / k_sq**2

def kappa_p_sq(binf_sq, hss):
    """
    [EXACT] κ_p² from h** and β_inf.
    At o=0: h**²·κ_p⁴ - κ_p² - β_inf² = 0
    → κ_p² = (1 + √(1 + 4·h**²·β_inf²)) / (2·h**²)
    """
    return (1.0 + np.sqrt(1.0 + 4.0*hss*binf_sq)) / (2.0*hss)

def get_nu(k_sq, bR, binf_sq, hss):
    """
    [EXACT] True anomaly from unitless scalars.
    From r = p/(1+ε·cos ν) and p = 2·h**²·R_sE in ROM:
      κ² = (1+ε·cos ν)/(2·h**²)  →  cos ν = (2·h**²·κ² - 1)/ε
    ε = √(1 + 4·h**²·β_inf²)
    Sign: β_R ≥ 0 → receding → ν > 0 (past periapsis)
          β_R < 0 → approaching → ν < 0 (before periapsis)
    """
    epsilon = np.sqrt(1.0 + 4.0*hss*binf_sq)
    cos_nu  = (2.0*hss*k_sq - 1.0) / epsilon
    nu      = np.arccos(np.clip(cos_nu, -1.0, 1.0))
    return nu if bR >= 0 else -nu

def theta_E_at_periapsis(theta_E_data_deg, nu_rad):
    """
    [EXACT] θ_E at periapsis from θ_E at data phase and true anomaly ν.
    If ν > 0 (Juno past periapsis, receding):
      periapsis direction is behind current position in sky
      → θ_E,periapsis = θ_E,data + ν   (periapsis was at larger elongation)
    If ν < 0 (approaching periapsis):
      → θ_E,periapsis = θ_E,data - ν
    General: θ_E,periapsis = θ_E,data + |ν| * sign(β_R)
    """
    return theta_E_data_deg + np.degrees(nu_rad)

def rom_prediction(binf_sq, kp_sq, theta_E_p_deg):
    """
    [EXACT] ROM deflection formula (ROM document):
      δ = 2·arcsin(κ_p²·(1+β_p²) / (2·β_p² - κ_p²·(1+β_p²)))

    [UNVERIFIED] Composition: law of cosines on β scalars.
    Returns β_in, β_out (unitless). Multiply by c for km/s.
    """
    bp_sq = binf_sq + kp_sq
    num   = kp_sq*(1.0 + bp_sq)
    den   = 2.0*bp_sq - kp_sq*(1.0 + bp_sq)
    delta = 2.0*np.arcsin(np.clip(num/den, -1.0, 1.0))

    bi    = np.sqrt(binf_sq)
    tE    = np.radians(theta_E_p_deg)
    bi_sq = binf_sq + beta_E**2 + 2.0*bi*beta_E*np.cos(tE)
    bo_sq = binf_sq + beta_E**2 + 2.0*bi*beta_E*np.cos(tE - delta)

    return (np.degrees(delta),
            np.sqrt(binf_sq + kp_sq),   # β_p
            np.sqrt(max(bi_sq, 0.0)),
            np.sqrt(max(bo_sq, 0.0)))

# ============================================================
# HORIZONS OBSERVER FETCH — scalars only
# Quantities fetched:
#   delta      (AU)    → κ_EJ²
#   delta_rate (km/s)  → β_R (NOTE: verify this is geocentric range-rate)
#   elong      (deg)   → θ_E
# ============================================================

def fetch_observer_scalars(start, stop, step='5m'):
    """
    Returns arrays of unitless (κ_EJ², β_R, θ_E_deg).
    Phase o tracked by κ_EJ²: maximum κ² = periapsis o=0.

    NOTE: if β_inf computed from pairs disagrees with known value,
    verify that delta_rate is genuine geocentric range-rate.
    At periapsis: delta_rate must be ≈ 0 km/s.
    """
    obj = Horizons(
        id='-61',
        location='500@399',
        epochs={'start': start, 'stop': stop, 'step': step}
    )
    eph = obj.ephemerides(quantities='19,20,23', extra_precision=True)

    r_AU    = np.array(eph['delta'])        # AU
    v_R_kms = np.array(eph['delta_rate'])   # km/s  geocentric range-rate
    tE_deg  = np.array(eph['elong'])        # deg   solar elongation

    r_km    = r_AU * 1.495978707e8          # km  (unit conversion, not physics)
    kappa_sq = R_sE / r_km                  # unitless κ_EJ²
    beta_R   = v_R_kms / c                  # unitless β_R

    return kappa_sq, beta_R, tE_deg

# ============================================================
# MAIN
# ============================================================

print("\nFetching OBSERVER scalars...")
kappa_sq, beta_R, tE_deg = fetch_observer_scalars(
    '2013-10-09 18:00', '2013-10-09 22:00', step='5m'
)

# Diagnostic: print raw values near periapsis
idx_max_k = np.argmax(kappa_sq)
print(f"\nDiagnostic — near periapsis (idx={idx_max_k}):")
lo = max(0, idx_max_k-2)
hi = min(len(kappa_sq), idx_max_k+3)
print(f"  {'idx':>4} {'κ²':>12} {'β_R (×c=km/s)':>16} {'θ_E deg':>10}")
for i in range(lo, hi):
    marker = " ← max κ" if i == idx_max_k else ""
    print(f"  {i:>4} {kappa_sq[i]:>12.4e} {beta_R[i]*c:>16.6f} {tE_deg[i]:>10.4f}{marker}")
print(f"  NOTE: β_R at max κ should be ≈ 0 km/s (periapsis).")
print(f"  If not, delta_rate from Horizons has a convention issue.")

# β_inf from MAX-MIN κ pair
idx_min_k = np.argmin(kappa_sq)
binf_sq   = beta_inf_sq(
    kappa_sq[idx_max_k], beta_R[idx_max_k],
    kappa_sq[idx_min_k], beta_R[idx_min_k]
)
print(f"\nβ_inf from MAX-MIN κ pair:")
print(f"  β_inf = {np.sqrt(max(binf_sq,0))*c:.4f} km/s  (known: 9.91 km/s)")
if abs(np.sqrt(max(binf_sq,0))*c - 9.91) > 0.1:
    print(f"  WARNING: discrepancy > 0.1 km/s — check delta_rate convention in astroquery")

# h** and κ_p² from MAX-κ point
hss   = h_star_sq(binf_sq, kappa_sq[idx_max_k], beta_R[idx_max_k])
kp_sq = kappa_p_sq(binf_sq, hss)

# ν at nearest data point to periapsis → θ_E at periapsis
nu_p  = get_nu(kappa_sq[idx_max_k], beta_R[idx_max_k], binf_sq, hss)
tE_p  = theta_E_at_periapsis(tE_deg[idx_max_k], nu_p)

print(f"\nPhase-based quantities (no time):")
print(f"  κ_p²    = {kp_sq:.4e}")
print(f"  r_p     = {R_sE/kp_sq:.2f} km  (known: 6930 km)")
print(f"  ν at data point = {np.degrees(nu_p):.4f} deg")
print(f"  θ_E at data     = {tE_deg[idx_max_k]:.4f} deg")
print(f"  θ_E at o=0      = {tE_p:.4f} deg")

# ROM prediction
delta_deg, beta_p, beta_in, beta_out = rom_prediction(binf_sq, kp_sq, tE_p)

print(f"\nROM PREDICTION (unitless → km/s only at end):")
print(f"  δ      = {delta_deg:.4f} deg")
print(f"  β_p    = {beta_p*c:.4f} km/s  (known: 14.60 km/s)")
print(f"  β_in   = {beta_in:.6e}")
print(f"  β_out  = {beta_out:.6e}")
boost = (beta_out - beta_in)*c
print(f"  boost  = {boost:.4f} km/s")            # ← ONLY unit multiplication
print(f"  observed = 7.3 km/s")
print(f"  diff   = {abs(boost-7.3):.4f} km/s  ({abs(boost-7.3)/7.3*100:.3f}%)")

STRUCTURAL PARAMETERS
  R_sE  = 8.9548 mm  [APPROX: standard 8.870 mm]
  β_E   = 9.936894e-05  [EXACT]

Fetching OBSERVER scalars...

Diagnostic — near periapsis (idx=16):
   idx           κ²    β_R (×c=km/s)    θ_E deg
    14   8.0697e-10        -9.631092    69.5423
    15   1.0534e-09        -7.310129    88.3661
    16   1.2759e-09        -1.998653   118.8124 ← max κ
    17   1.2000e-09         4.726706   153.3263
    18   9.3858e-10         8.588812   170.7228
  NOTE: β_R at max κ should be ≈ 0 km/s (periapsis).
  If not, delta_rate from Horizons has a convention issue.

β_inf from MAX-MIN κ pair:
  β_inf = 0.0000 km/s  (known: 9.91 km/s)

Phase-based quantities (no time):
  κ_p²    = 5.0776e-08
  r_p     = 176.36 km  (known: 6930 km)
  ν at data point = -176.4816 deg
  θ_E at data     = 118.8124 deg
  θ_E at o=0      = -57.6692 deg

ROM PREDICTION (unitless → km/s only at end):
  δ      = 180.0000 deg
  β_p    = 66.7510 km/s  (known: 14.60 km/s)
  β_in   = nan
  β_out  = nan
  boos

/tmp/ipykernel_24080/1684815393.py:141: RuntimeWarning: invalid value encountered in sqrt
  bi    = np.sqrt(binf_sq)


In [ ]:
"""
WILL R.O.M. — Juno Earth Flyby
================================
UNITLESS. PHASE-BASED. NO G. NO M. NO μ. NO TIME.

CORRECTIONS FROM PREVIOUS VERSION:
1. β_inf² formula sign was wrong → gave 0. Correct derivation:
   β_inf² = [A²·Q - B²·P - A·B·(A-B)] / (A²-B²)
   where A=κ_max², B=κ_min², P=β_R²@A, Q=β_R²@B

2. Window too narrow: ±2h gives r_max≈70,000 km.
   At 70,000 km: β_R/β_inf = 1.042 → β_inf error = 4.2%.
   Need ±12h window for β_R/β_inf < 1.009 (0.9% accuracy).

3. ν formula was correct but failed due to β_inf=0 from bug 1.
   Now correct: cos ν = (2·h**²·κ² - 1) / ε

APPROXIMATION:
  [APPROX] R_sE: ~1% from Moon semi-major axis precision.
  [UNVERIFIED] Boost composition law of cosines on β scalars.
"""

import numpy as np
from astroquery.jplhorizons import Horizons

c = 299792.458  # km/s — ONLY for final unit restoration

# ============================================================
# OBSERVATIONAL PRIMITIVES → UNITLESS STRUCTURAL PARAMETERS
# ============================================================
theta_sun = 0.0046524
z_sun     = 2.1224e-6
T_E       = 3.15581498e7        # s
a_moon    = 384400.0            # km  [APPROX]
T_m       = 27.321661 * 86400.0 # s

R_sE     = 8.0*np.pi**2*a_moon**3 / (T_m**2*c**2)  # km
kappa2_S = 1.0 - 1.0/(1.0+z_sun)**2
R_sS     = (T_E/np.pi)*c*(np.sqrt(0.5*kappa2_S*np.sin(theta_sun)))**3
kappa2_E = kappa2_S*np.sin(theta_sun)
R_SE     = R_sS/kappa2_E
beta_E   = np.sqrt(R_sS/(2.0*R_SE))

print("STRUCTURAL PARAMETERS")
print(f"  R_sE = {R_sE*1e6:.4f} mm  [APPROX: standard 8.870 mm]")
print(f"  β_E  = {beta_E:.6e}  [EXACT]")

# ============================================================
# UNITLESS INVARIANTS
# ============================================================

def beta_inf_sq(A, bRA, B, bRB):
    """
    [EXACT] β_inf² from two phase points.
    Derived from: β_inf² = β_R² + h**²·κ⁴ - κ²  (constant at all phases)

    Setting equal at A=κ_max², B=κ_min²:
      h**²(A²-B²) = β_R_B² - β_R_A² + A - B

    Substituting back:
      β_inf² = [A²·β_R_B² - B²·β_R_A² - A·B·(A-B)] / (A²-B²)

    Use MAX-κ and MIN-κ pair for best numerical stability.
    MIN-κ point must be far from Earth (wide window) for β_R→β_inf.
    At ±2h: β_R/β_inf ≈ 1.042 (4.2% error).
    At ±12h: β_R/β_inf ≈ 1.009 (0.9% error).
    """
    P, Q = bRA**2, bRB**2
    return (A**2*Q - B**2*P - A*B*(A-B)) / (A**2 - B**2)

def h_star_sq(binf_sq, k_sq, bR):
    """[EXACT] h**² = (β_inf² + κ² - β_R²) / κ⁴"""
    return (binf_sq + k_sq - bR**2) / k_sq**2

def kappa_p_sq(binf_sq, hss):
    """
    [EXACT] κ_p² from h** and β_inf.
    h**²·κ_p⁴ - κ_p² - β_inf² = 0
    → κ_p² = (1 + √(1 + 4·h**²·β_inf²)) / (2·h**²)
    """
    return (1.0 + np.sqrt(1.0 + 4.0*hss*binf_sq)) / (2.0*hss)

def get_nu(k_sq, bR, binf_sq, hss):
    """
    [EXACT] True anomaly from unitless scalars.
    From r = p/(1+ε·cos ν), p = 2·h**²·R_sE (in ROM):
      κ² = (1+ε·cos ν)/(2·h**²)
      → cos ν = (2·h**²·κ² - 1) / ε
    ε = √(1 + 4·h**²·β_inf²)
    Sign from β_R: β_R ≥ 0 → past periapsis → ν > 0
                   β_R < 0 → before periapsis → ν < 0
    """
    epsilon = np.sqrt(1.0 + 4.0*hss*binf_sq)
    cos_nu  = (2.0*hss*k_sq - 1.0) / epsilon
    nu      = np.arccos(np.clip(cos_nu, -1.0, 1.0))
    return nu if bR >= 0 else -nu

def theta_E_periapsis(theta_E_data_deg, nu_rad):
    """
    [EXACT] θ_E at o=0 from θ_E at data phase and ν.
    θ_E,periapsis = θ_E,data + ν
    (ν > 0: Juno past periapsis, periapsis is behind current direction)
    (ν < 0: Juno before periapsis, periapsis is ahead)
    """
    return theta_E_data_deg + np.degrees(nu_rad)

def rom_predict(binf_sq, kp_sq, theta_E_p_deg):
    """
    [EXACT] ROM deflection formula.
    [UNVERIFIED] Boost composition via law of cosines on β scalars.
    Returns δ_deg, β_p (unitless), β_in (unitless), β_out (unitless).
    """
    bp_sq  = binf_sq + kp_sq
    num    = kp_sq*(1.0 + bp_sq)
    den    = 2.0*bp_sq - kp_sq*(1.0 + bp_sq)
    delta  = 2.0*np.arcsin(np.clip(num/den, -1.0, 1.0))
    bi_val = np.sqrt(binf_sq)
    tE     = np.radians(theta_E_p_deg)
    bi_sq  = binf_sq + beta_E**2 + 2.0*bi_val*beta_E*np.cos(tE)
    bo_sq  = binf_sq + beta_E**2 + 2.0*bi_val*beta_E*np.cos(tE - delta)
    return (np.degrees(delta),
            np.sqrt(max(bp_sq, 0.0)),
            np.sqrt(max(bi_sq, 0.0)),
            np.sqrt(max(bo_sq, 0.0)))

# ============================================================
# HORIZONS FETCH — WIDE WINDOW (±12h) for asymptotic β_R
# ============================================================

def fetch_scalars(start, stop, step='5m'):
    """
    Fetch unitless (κ_EJ², β_R, θ_E) from Horizons OBSERVER table.
    Wide window needed: ±12h gives β_R/β_inf < 1.009 at endpoints.
    """
    obj = Horizons(id='-61', location='500@399',
                   epochs={'start':start, 'stop':stop, 'step':step})
    eph = obj.ephemerides(quantities='19,20,23', extra_precision=True)

    r_km     = np.array(eph['delta']) * 1.495978707e8
    v_R_kms  = np.array(eph['delta_rate'])
    tE_deg   = np.array(eph['elong'])

    kappa_sq = R_sE / r_km
    beta_R   = v_R_kms / c
    return kappa_sq, beta_R, tE_deg

# ============================================================
# MAIN
# ============================================================

print("\nFetching OBSERVER scalars (±12h window for asymptotic β_R)...")
kappa_sq, beta_R, tE_deg = fetch_scalars(
    '2013-10-09 07:00',   # 12h before periapsis (~19:21 UTC)
    '2013-10-10 07:00',   # 12h after periapsis
    step='30m'            # coarse away from periapsis is fine
)

# Refine near periapsis with fine step
kappa_sq_fine, beta_R_fine, tE_deg_fine = fetch_scalars(
    '2013-10-09 19:00',
    '2013-10-09 19:45',
    step='1m'
)
# Merge: use fine around periapsis, coarse elsewhere
kappa_sq_all = np.concatenate([kappa_sq, kappa_sq_fine])
beta_R_all   = np.concatenate([beta_R,   beta_R_fine])
tE_deg_all   = np.concatenate([tE_deg,   tE_deg_fine])

idx_max_k = np.argmax(kappa_sq_all)   # o=0 (periapsis)
idx_min_k = np.argmin(kappa_sq_all)   # far from Earth (asymptote)

# Diagnostic
print(f"\nDiagnostic:")
print(f"  max κ² = {kappa_sq_all[idx_max_k]:.4e}  β_R = {beta_R_all[idx_max_k]*c:+.4f} km/s  θ_E = {tE_deg_all[idx_max_k]:.4f}°")
print(f"  min κ² = {kappa_sq_all[idx_min_k]:.4e}  β_R = {beta_R_all[idx_min_k]*c:+.4f} km/s  θ_E = {tE_deg_all[idx_min_k]:.4f}°")
print(f"  At min κ: β_R/β_inf(known) = {abs(beta_R_all[idx_min_k])*c/9.91:.4f}  (target: <1.01)")

# β_inf from MAX-MIN pair
binf_sq = beta_inf_sq(
    kappa_sq_all[idx_max_k], beta_R_all[idx_max_k],
    kappa_sq_all[idx_min_k], beta_R_all[idx_min_k]
)
print(f"\nβ_inf = {np.sqrt(max(binf_sq,0))*c:.4f} km/s  (known: 9.91 km/s)")

# h**, κ_p², ν and θ_E at periapsis
hss   = h_star_sq(binf_sq, kappa_sq_all[idx_max_k], beta_R_all[idx_max_k])
kp_sq = kappa_p_sq(binf_sq, hss)
nu_p  = get_nu(kappa_sq_all[idx_max_k], beta_R_all[idx_max_k], binf_sq, hss)
tE_p  = theta_E_periapsis(tE_deg_all[idx_max_k], nu_p)

print(f"\nPhase-based quantities:")
print(f"  κ_p²   = {kp_sq:.4e}")
print(f"  r_p    = {R_sE/kp_sq:.2f} km  (known: 6930 km)")
print(f"  ν      = {np.degrees(nu_p):.4f} deg at data point")
print(f"  θ_E data     = {tE_deg_all[idx_max_k]:.4f} deg")
print(f"  θ_E periapsis= {tE_p:.4f} deg")

# ROM prediction
delta_deg, beta_p, beta_in, beta_out = rom_predict(binf_sq, kp_sq, tE_p)
boost = (beta_out - beta_in)*c   # ← ONLY unit multiplication

print(f"\nROM PREDICTION:")
print(f"  δ      = {delta_deg:.4f} deg")
print(f"  β_p    = {beta_p*c:.4f} km/s  (known: 14.60 km/s)")
print(f"  boost  = {boost:.4f} km/s")
print(f"  observed = 7.3 km/s")
print(f"  diff   = {abs(boost-7.3):.4f} km/s  ({abs(boost-7.3)/7.3*100:.3f}%)")

STRUCTURAL PARAMETERS
  R_sE = 8.9548 mm  [APPROX: standard 8.870 mm]
  β_E  = 9.936894e-05  [EXACT]

Fetching OBSERVER scalars (±12h window for asymptotic β_R)...

Diagnostic:
  max κ² = 1.2903e-09  β_R = -0.5938 km/s  θ_E = 125.9976°
  min κ² = 1.8843e-11  β_R = -10.4669 km/s  θ_E = 22.9213°
  At min κ: β_R/β_inf(known) = 1.0562  (target: <1.01)

β_inf = 10.3879 km/s  (known: 9.91 km/s)

Phase-based quantities:
  κ_p²   = 1.2916e-09
  r_p    = 6932.98 km  (known: 6930 km)
  ν      = -3.0696 deg at data point
  θ_E data     = 125.9976 deg
  θ_E periapsis= 122.9280 deg

ROM PREDICTION:
  δ      = 40.9449 deg
  β_p    = 14.9665 km/s  (known: 14.60 km/s)
  boost  = 7.2193 km/s
  observed = 7.3 km/s
  diff   = 0.0807 km/s  (1.106%)


In [ ]:
"""
WILL R.O.M. — Juno Earth Flyby
================================
UNITLESS. PHASE-BASED. NO G. NO M. NO μ. NO TIME.

CORRECTIONS FROM PREVIOUS VERSION:
1. β_inf² formula sign was wrong → gave 0. Correct derivation:
   β_inf² = [A²·Q - B²·P - A·B·(A-B)] / (A²-B²)
   where A=κ_max², B=κ_min², P=β_R²@A, Q=β_R²@B

2. Window too narrow: ±2h gives r_max≈70,000 km.
   At 70,000 km: β_R/β_inf = 1.042 → β_inf error = 4.2%.
   Need ±12h window for β_R/β_inf < 1.009 (0.9% accuracy).

3. ν formula was correct but failed due to β_inf=0 from bug 1.
   Now correct: cos ν = (2·h**²·κ² - 1) / ε

APPROXIMATION:
  [APPROX] R_sE: ~1% from Moon semi-major axis precision.
  [UNVERIFIED] Boost composition law of cosines on β scalars.
"""

import numpy as np
from astroquery.jplhorizons import Horizons

c = 299792.458  # km/s — ONLY for final unit restoration

# ============================================================
# OBSERVATIONAL PRIMITIVES → UNITLESS STRUCTURAL PARAMETERS
# ============================================================
theta_sun = 0.0046524
z_sun     = 2.1224e-6
T_E       = 3.15581498e7        # s
a_moon    = 384400.0            # km  [APPROX]
T_m       = 27.321661 * 86400.0 # s

R_sE     = 8.0*np.pi**2*a_moon**3 / (T_m**2*c**2)  # km
kappa2_S = 1.0 - 1.0/(1.0+z_sun)**2
R_sS     = (T_E/np.pi)*c*(np.sqrt(0.5*kappa2_S*np.sin(theta_sun)))**3
kappa2_E = kappa2_S*np.sin(theta_sun)
R_SE     = R_sS/kappa2_E
beta_E   = np.sqrt(R_sS/(2.0*R_SE))

print("STRUCTURAL PARAMETERS")
print(f"  R_sE = {R_sE*1e6:.4f} mm  [APPROX: standard 8.870 mm]")
print(f"  β_E  = {beta_E:.6e}  [EXACT]")

# ============================================================
# UNITLESS INVARIANTS
# ============================================================

def beta_inf_sq(A, bRA, B, bRB):
    """
    [EXACT] β_inf² from two phase points.
    Derived from: β_inf² = β_R² + h**²·κ⁴ - κ²  (constant at all phases)

    Setting equal at A=κ_max², B=κ_min²:
      h**²(A²-B²) = β_R_B² - β_R_A² + A - B

    Substituting back:
      β_inf² = [A²·β_R_B² - B²·β_R_A² - A·B·(A-B)] / (A²-B²)

    Use MAX-κ and MIN-κ pair for best numerical stability.
    MIN-κ point must be far from Earth (wide window) for β_R→β_inf.
    At ±2h: β_R/β_inf ≈ 1.042 (4.2% error).
    At ±12h: β_R/β_inf ≈ 1.009 (0.9% error).
    """
    P, Q = bRA**2, bRB**2
    return (A**2*Q - B**2*P - A*B*(A-B)) / (A**2 - B**2)

def h_star_sq(binf_sq, k_sq, bR):
    """[EXACT] h**² = (β_inf² + κ² - β_R²) / κ⁴"""
    return (binf_sq + k_sq - bR**2) / k_sq**2

def kappa_p_sq(binf_sq, hss):
    """
    [EXACT] κ_p² from h** and β_inf.
    h**²·κ_p⁴ - κ_p² - β_inf² = 0
    → κ_p² = (1 + √(1 + 4·h**²·β_inf²)) / (2·h**²)
    """
    return (1.0 + np.sqrt(1.0 + 4.0*hss*binf_sq)) / (2.0*hss)

def get_nu(k_sq, bR, binf_sq, hss):
    """
    [EXACT] True anomaly from unitless scalars.
    From r = p/(1+ε·cos ν), p = 2·h**²·R_sE (in ROM):
      κ² = (1+ε·cos ν)/(2·h**²)
      → cos ν = (2·h**²·κ² - 1) / ε
    ε = √(1 + 4·h**²·β_inf²)
    Sign from β_R: β_R ≥ 0 → past periapsis → ν > 0
                   β_R < 0 → before periapsis → ν < 0
    """
    epsilon = np.sqrt(1.0 + 4.0*hss*binf_sq)
    cos_nu  = (2.0*hss*k_sq - 1.0) / epsilon
    nu      = np.arccos(np.clip(cos_nu, -1.0, 1.0))
    return nu if bR >= 0 else -nu

def theta_E_periapsis(theta_E_data_deg, nu_rad):
    """
    [EXACT] θ_E at o=0 from θ_E at data phase and ν.
    θ_E,periapsis = θ_E,data + ν
    (ν > 0: Juno past periapsis, periapsis is behind current direction)
    (ν < 0: Juno before periapsis, periapsis is ahead)
    """
    return theta_E_data_deg + np.degrees(nu_rad)

def rom_predict(binf_sq, kp_sq, theta_E_p_deg):
    """
    [EXACT] ROM deflection formula.
    [UNVERIFIED] Boost composition via law of cosines on β scalars.
    Returns δ_deg, β_p (unitless), β_in (unitless), β_out (unitless).
    """
    bp_sq  = binf_sq + kp_sq
    num    = kp_sq*(1.0 + bp_sq)
    den    = 2.0*bp_sq - kp_sq*(1.0 + bp_sq)
    delta  = 2.0*np.arcsin(np.clip(num/den, -1.0, 1.0))
    bi_val = np.sqrt(binf_sq)
    tE     = np.radians(theta_E_p_deg)
    bi_sq  = binf_sq + beta_E**2 + 2.0*bi_val*beta_E*np.cos(tE)
    bo_sq  = binf_sq + beta_E**2 + 2.0*bi_val*beta_E*np.cos(tE - delta)
    return (np.degrees(delta),
            np.sqrt(max(bp_sq, 0.0)),
            np.sqrt(max(bi_sq, 0.0)),
            np.sqrt(max(bo_sq, 0.0)))

# ============================================================
# HORIZONS FETCH — WIDE WINDOW (±12h) for asymptotic β_R
# ============================================================

def fetch_scalars(start, stop, step='5m'):
    """
    Fetch unitless (κ_EJ², β_R, θ_E) from Horizons OBSERVER table.
    Wide window needed: ±12h gives β_R/β_inf < 1.009 at endpoints.
    """
    obj = Horizons(id='-61', location='500@399',
                   epochs={'start':start, 'stop':stop, 'step':step})
    eph = obj.ephemerides(quantities='19,20,23', extra_precision=True)

    r_km     = np.array(eph['delta']) * 1.495978707e8
    v_R_kms  = np.array(eph['delta_rate'])
    tE_deg   = np.array(eph['elong'])

    kappa_sq = R_sE / r_km
    beta_R   = v_R_kms / c
    return kappa_sq, beta_R, tE_deg

# ============================================================
# MAIN
# ============================================================

print("\nFetching OBSERVER scalars...")
print("  Strategy: fine window only (±45min around periapsis).")
print("  At r < 20,000 km: Sun/Earth gravity < 0.4% → two-body ROM valid.")
print("  Far data (±12h) is INVALID for ROM: Sun dominates at r > 50,000 km,")
print("  breaking the two-body energy invariant β_inf² = β_rel² - κ_EJ².")

# ONLY fine window — both endpoints within ~20,000 km of Earth
kappa_sq_all, beta_R_all, tE_deg_all = fetch_scalars(
    '2013-10-09 19:00',   # ~21 min before periapsis → r ≈ 16,000 km
    '2013-10-09 19:45',   # ~24 min after periapsis  → r ≈ 18,500 km
    step='1m'
)

idx_max_k = np.argmax(kappa_sq_all)   # o=0 (periapsis)
idx_min_k = np.argmin(kappa_sq_all)   # far from Earth (asymptote)

# Diagnostic
print(f"\nDiagnostic:")
print(f"  max κ² = {kappa_sq_all[idx_max_k]:.4e}  β_R = {beta_R_all[idx_max_k]*c:+.4f} km/s  θ_E = {tE_deg_all[idx_max_k]:.4f}°")
print(f"  min κ² = {kappa_sq_all[idx_min_k]:.4e}  β_R = {beta_R_all[idx_min_k]*c:+.4f} km/s  θ_E = {tE_deg_all[idx_min_k]:.4f}°")
print(f"  At min κ: β_R/β_inf(known) = {abs(beta_R_all[idx_min_k])*c/9.91:.4f}  (target: <1.01)")

# β_inf from MAX-MIN pair
binf_sq = beta_inf_sq(
    kappa_sq_all[idx_max_k], beta_R_all[idx_max_k],
    kappa_sq_all[idx_min_k], beta_R_all[idx_min_k]
)
print(f"\nβ_inf = {np.sqrt(max(binf_sq,0))*c:.4f} km/s  (known: 9.91 km/s)")

# h**, κ_p², ν and θ_E at periapsis
hss   = h_star_sq(binf_sq, kappa_sq_all[idx_max_k], beta_R_all[idx_max_k])
kp_sq = kappa_p_sq(binf_sq, hss)
nu_p  = get_nu(kappa_sq_all[idx_max_k], beta_R_all[idx_max_k], binf_sq, hss)
tE_p  = theta_E_periapsis(tE_deg_all[idx_max_k], nu_p)

print(f"\nPhase-based quantities:")
print(f"  κ_p²   = {kp_sq:.4e}")
print(f"  r_p    = {R_sE/kp_sq:.2f} km  (known: 6930 km)")
print(f"  ν      = {np.degrees(nu_p):.4f} deg at data point")
print(f"  θ_E data     = {tE_deg_all[idx_max_k]:.4f} deg")
print(f"  θ_E periapsis= {tE_p:.4f} deg")

# ROM prediction
delta_deg, beta_p, beta_in, beta_out = rom_predict(binf_sq, kp_sq, tE_p)
boost = (beta_out - beta_in)*c   # ← ONLY unit multiplication

print(f"\nROM PREDICTION:")
print(f"  δ      = {delta_deg:.4f} deg")
print(f"  β_p    = {beta_p*c:.4f} km/s  (known: 14.60 km/s)")
print(f"  boost  = {boost:.4f} km/s")
print(f"  observed = 7.3 km/s")
print(f"  diff   = {abs(boost-7.3):.4f} km/s  ({abs(boost-7.3)/7.3*100:.3f}%)")

STRUCTURAL PARAMETERS
  R_sE = 8.9548 mm  [APPROX: standard 8.870 mm]
  β_E  = 9.936894e-05  [EXACT]

Fetching OBSERVER scalars...
  Strategy: fine window only (±45min around periapsis).
  At r < 20,000 km: Sun/Earth gravity < 0.4% → two-body ROM valid.
  Far data (±12h) is INVALID for ROM: Sun dominates at r > 50,000 km,
  breaking the two-body energy invariant β_inf² = β_rel² - κ_EJ².

Diagnostic:
  max κ² = 1.2903e-09  β_R = -0.5938 km/s  θ_E = 125.9976°
  min κ² = 4.7689e-10  β_R = +10.9540 km/s  θ_E = 147.7887°
  At min κ: β_R/β_inf(known) = 1.1053  (target: <1.01)

β_inf = 10.3743 km/s  (known: 9.91 km/s)

Phase-based quantities:
  κ_p²   = 1.2916e-09
  r_p    = 6932.96 km  (known: 6930 km)
  ν      = -3.0729 deg at data point
  θ_E data     = 125.9976 deg
  θ_E periapsis= 122.9247 deg

ROM PREDICTION:
  δ      = 41.0182 deg
  β_p    = 14.9570 km/s  (known: 14.60 km/s)
  boost  = 7.2220 km/s
  observed = 7.3 km/s
  diff   = 0.0780 km/s  (1.068%)


In [ ]:
"""
Diagnostic: Compare Horizons OBSERVER delta_rate vs VECTORS-derived v_R
========================================================================
If OBSERVER delta_rate == VECTORS v_R → data is correct, ROM theory has error
If OBSERVER delta_rate != VECTORS v_R → Horizons convention issue to resolve

All quantities in km/s.
No ROM theory applied here — pure data comparison.
"""

import numpy as np
from astroquery.jplhorizons import Horizons

c        = 299792.458
AU_KM    = 1.495978707e8
SPDAY    = 86400.0

START = '2013-10-09 19:00'
STOP  = '2013-10-09 19:45'
STEP  = '1m'

# ============================================================
# FETCH 1: OBSERVER TABLE
# ============================================================
print("Fetching OBSERVER table...")
obj_obs = Horizons(id='-61', location='500@399',
                   epochs={'start':START, 'stop':STOP, 'step':STEP})
eph = obj_obs.ephemerides(quantities='19,20,23', extra_precision=True)

r_obs_AU  = np.array(eph['delta'])       # AU
vR_obs    = np.array(eph['delta_rate'])  # km/s  from OBSERVER
tE_obs    = np.array(eph['elong'])       # deg
times     = np.array(eph['datetime_str'])

r_obs_km  = r_obs_AU * AU_KM

# ============================================================
# FETCH 2: VECTORS TABLE — Juno and Earth from SSB
# ============================================================
print("Fetching VECTORS table (Juno from SSB)...")
obj_J = Horizons(id='-61', location='500@0',
                 epochs={'start':START, 'stop':STOP, 'step':STEP})
vec_J = obj_J.vectors(refplane='ecliptic')

print("Fetching VECTORS table (Earth from SSB)...")
obj_E = Horizons(id='399', location='500@0',
                 epochs={'start':START, 'stop':STOP, 'step':STEP})
vec_E = obj_E.vectors(refplane='ecliptic')

# ============================================================
# COMPUTE v_R FROM VECTORS
# ============================================================
pos_J = np.column_stack([vec_J['x'], vec_J['y'], vec_J['z']]) * AU_KM
vel_J = np.column_stack([vec_J['vx'],vec_J['vy'],vec_J['vz']]) * (AU_KM/SPDAY)
pos_E = np.column_stack([vec_E['x'], vec_E['y'], vec_E['z']]) * AU_KM
vel_E = np.column_stack([vec_E['vx'],vec_E['vy'],vec_E['vz']]) * (AU_KM/SPDAY)

dr  = pos_J - pos_E          # Juno-Earth position vector
dv  = vel_J - vel_E          # Juno-Earth velocity vector
r   = np.linalg.norm(dr, axis=1)
v   = np.linalg.norm(dv, axis=1)

# Pure radial velocity: projection of dv onto dr
r_hat = dr / r[:,None]
vR_vec = np.einsum('ij,ij->i', dv, r_hat)  # km/s  signed range-rate

# Transverse velocity
vT_vec = np.sqrt(np.maximum(v**2 - vR_vec**2, 0))

# ============================================================
# TWO-BODY THEORETICAL β_inf FROM VECTORS
# ============================================================
R_sE_std = 8.870e-6   # km  standard (for theory comparison only)
mu_std   = R_sE_std * c**2 / 2

# Theoretical β_inf from ROM energy invariant using VECTORS v and r
V_inf_vec = np.sqrt(np.maximum(v**2 - 2*mu_std/r, 0))

# ============================================================
# PRINT COMPARISON
# ============================================================
print(f"\n{'='*90}")
print(f"COMPARISON: OBSERVER delta_rate vs VECTORS-derived v_R")
print(f"{'='*90}")
print(f"{'time':>22} {'r_obs':>10} {'r_vec':>10} {'vR_obs':>10} {'vR_vec':>10} {'diff':>8} {'vT_vec':>8} {'Vinf_vec':>10}")
print(f"{'':>22} {'km':>10} {'km':>10} {'km/s':>10} {'km/s':>10} {'km/s':>8} {'km/s':>8} {'km/s':>10}")
print("-"*90)

for i in range(len(times)):
    diff = vR_obs[i] - vR_vec[i]
    marker = " ← max κ" if abs(r_obs_km[i] - r_obs_km.min()) < 50 else ""
    print(f"{times[i]:>22} {r_obs_km[i]:>10.2f} {r[i]:>10.2f} {vR_obs[i]:>10.4f} {vR_vec[i]:>10.4f} {diff:>8.4f} {vT_vec[i]:>8.4f} {V_inf_vec[i]:>10.4f}{marker}")

print(f"\n{'='*90}")
print(f"SUMMARY")
print(f"{'='*90}")
diff_arr = vR_obs - vR_vec
print(f"  Mean  |vR_obs - vR_vec| = {np.mean(np.abs(diff_arr)):.4f} km/s")
print(f"  Max   |vR_obs - vR_vec| = {np.max(np.abs(diff_arr)):.4f} km/s")
print(f"  Mean  diff/vR_vec       = {np.mean(diff_arr/np.abs(vR_vec+1e-10))*100:.4f}%")
print()
print(f"  If diff ≈ 0: OBSERVER and VECTORS agree → Horizons data consistent")
print(f"               → β_inf discrepancy is from our ROM theory")
print()
print(f"  If diff ≈ 0.4 km/s: OBSERVER and VECTORS disagree → convention issue")
print(f"               → use vR_vec in ROM formula instead of delta_rate")

# ============================================================
# IF THEY AGREE: what β_inf does the VECTORS v_R give?
# ============================================================
print(f"\n{'='*90}")
print(f"β_inf FROM VECTORS v_R (same formula, different input)")
print(f"{'='*90}")

a_moon=384400.0; T_m=27.321661*86400
R_sE_rom = 8*np.pi**2*a_moon**3/(T_m**2*c**2)

for i in [0, len(times)//4, len(times)//2, -1]:
    kp_sq = R_sE_rom/r[i]
    kp_sq2 = R_sE_rom/r[0]
    P = (vR_vec[i]/c)**2
    Q = (vR_vec[0]/c)**2
    A = kp_sq2; B = kp_sq
    if abs(A-B) > 1e-15:
        bsq = (A**2*P - B**2*Q - A*B*(A-B)) / (A**2-B**2)
        # Note: now A=κ at periapsis (max), B=κ at other point (smaller)
        # check sign: max κ vs this point
        if kp_sq2 > kp_sq:  # periapsis is point 0
            bsq2 = (kp_sq2**2*(vR_vec[i]/c)**2 - kp_sq**2*(vR_vec[0]/c)**2
                    - kp_sq2*kp_sq*(kp_sq2-kp_sq)) / (kp_sq2**2-kp_sq**2)
        else:
            bsq2 = (kp_sq**2*(vR_vec[0]/c)**2 - kp_sq2**2*(vR_vec[i]/c)**2
                    - kp_sq*kp_sq2*(kp_sq-kp_sq2)) / (kp_sq**2-kp_sq2**2)
        print(f"  point {i:3d}: r={r[i]:.0f} km, vR_vec={vR_vec[i]:.4f}, β_inf={np.sqrt(max(bsq2,0))*c:.4f} km/s")

Fetching OBSERVER table...
Fetching VECTORS table (Juno from SSB)...
Fetching VECTORS table (Earth from SSB)...

COMPARISON: OBSERVER delta_rate vs VECTORS-derived v_R
                  time      r_obs      r_vec     vR_obs     vR_vec     diff   vT_vec   Vinf_vec
                               km         km       km/s       km/s     km/s     km/s       km/s
------------------------------------------------------------------------------------------
     2013-Oct-09 19:00   17359.04   18010.79   -10.8764   -10.8932   0.0168   5.7741    10.3798
     2013-Oct-09 19:01   16707.78   17358.41   -10.8286   -10.8516   0.0229   5.9915    10.3793
     2013-Oct-09 19:02   16059.65   16708.76   -10.7715   -10.8019   0.0304   6.2249    10.3788
     2013-Oct-09 19:03   15415.28   16062.38   -10.7030   -10.7425   0.0395   6.4759    10.3782
     2013-Oct-09 19:04   14775.42   15419.89   -10.6209   -10.6716   0.0507   6.7463    10.3777
     2013-Oct-09 19:05   14140.96   14782.06   -10.5224   -10.5868   

In [ ]:
"""
Corrected VECTORS fetch: both Juno and Earth relative to GEOCENTER.
Fixes the Earth-Moon barycenter vs geocenter confusion.

Key: fetch Juno vectors with location='500@399' (geocenter)
     NOT from SSB and subtracting Earth's barycentric position.
"""

import numpy as np
from astroquery.jplhorizons import Horizons

c     = 299792.458
AU_KM = 1.495978707e8
SPDAY = 86400.0

START = '2013-10-09 19:00'
STOP  = '2013-10-09 19:45'
STEP  = '1m'

# ROM observational primitives
theta_sun=0.0046524; z_sun=2.1224e-6; T_E=3.15581498e7
a_moon=384400.0; T_m=27.321661*86400
R_sE = 8*np.pi**2*a_moon**3/(T_m**2*c**2)
kappa2_S=(1-1/(1+z_sun)**2)
R_sS=(T_E/np.pi)*c*(np.sqrt(0.5*kappa2_S*np.sin(theta_sun)))**3
kappa2_E=kappa2_S*np.sin(theta_sun)
R_SE=R_sS/kappa2_E; beta_E=np.sqrt(R_sS/(2*R_SE))

mu_std = 8.870e-6 * c**2 / 2   # standard, for V_inf check only

# ============================================================
# METHOD A: VECTORS with Juno relative to GEOCENTER directly
# This avoids the Earth barycenter vs geocenter issue
# ============================================================
print("Fetching Juno VECTORS relative to Earth GEOCENTER...")
obj_J_geo = Horizons(id='-61', location='500@399',
                     epochs={'start':START,'stop':STOP,'step':STEP})
vec_J_geo = obj_J_geo.vectors(refplane='ecliptic')

r_geo_AU  = np.array(vec_J_geo['range'])    # AU  if available, else compute
x_geo = np.array(vec_J_geo['x'])*AU_KM
y_geo = np.array(vec_J_geo['y'])*AU_KM
z_geo = np.array(vec_J_geo['z'])*AU_KM
vx_geo = np.array(vec_J_geo['vx'])*(AU_KM/SPDAY)
vy_geo = np.array(vec_J_geo['vy'])*(AU_KM/SPDAY)
vz_geo = np.array(vec_J_geo['vz'])*(AU_KM/SPDAY)

r_geo  = np.sqrt(x_geo**2 + y_geo**2 + z_geo**2)
v_geo  = np.sqrt(vx_geo**2 + vy_geo**2 + vz_geo**2)

# vR = projection of v onto r unit vector
r_hat  = np.column_stack([x_geo,y_geo,z_geo]) / r_geo[:,None]
v_vec  = np.column_stack([vx_geo,vy_geo,vz_geo])
vR_geo = np.einsum('ij,ij->i', v_vec, r_hat)   # signed, km/s
vT_geo = np.sqrt(np.maximum(v_geo**2 - vR_geo**2, 0))

# V_inf from ROM energy invariant (unitless, then convert)
kappa_sq  = R_sE / r_geo
V_inf_arr = c*np.sqrt(np.maximum(v_geo**2/c**2 - kappa_sq, 0))

# Also fetch OBSERVER for comparison
print("Fetching OBSERVER table...")
obj_obs = Horizons(id='-61', location='500@399',
                   epochs={'start':START,'stop':STOP,'step':STEP})
eph = obj_obs.ephemerides(quantities='19,20,23', extra_precision=True)
r_obs_km = np.array(eph['delta'])*AU_KM
vR_obs   = np.array(eph['delta_rate'])
tE_deg   = np.array(eph['elong'])
times    = np.array(eph['datetime_str'])

# ============================================================
# PRINT COMPARISON
# ============================================================
print(f"\n{'='*100}")
print(f"CORRECTED COMPARISON: OBSERVER vs VECTORS@GEOCENTER")
print(f"{'='*100}")
print(f"{'time':>22} {'r_obs':>9} {'r_geo':>9} {'vR_obs':>9} {'vR_geo':>9} {'diff':>7} {'vT_geo':>8} {'Vinf':>8} {'κ²':>12}")
print(f"{'':>22} {'km':>9} {'km':>9} {'km/s':>9} {'km/s':>9} {'km/s':>7} {'km/s':>8} {'km/s':>8} {'':>12}")
print("-"*100)

for i in range(len(times)):
    diff = vR_obs[i] - vR_geo[i]
    marker = " ←p" if r_obs_km[i] == r_obs_km.min() else ""
    print(f"{times[i]:>22} {r_obs_km[i]:>9.2f} {r_geo[i]:>9.2f} "
          f"{vR_obs[i]:>9.4f} {vR_geo[i]:>9.4f} {diff:>7.4f} "
          f"{vT_geo[i]:>8.4f} {V_inf_arr[i]:>8.4f} {kappa_sq[i]:>12.4e}{marker}")

print(f"\n{'='*100}")
print(f"SUMMARY")
print(f"{'='*100}")
diff_arr = vR_obs - vR_geo

# V_inf check: should be constant (ROM energy invariant test)
Vinf_valid = V_inf_arr[V_inf_arr > 0]
print(f"  r_obs vs r_geo: max diff = {np.max(np.abs(r_obs_km - r_geo)):.2f} km")
print(f"  vR_obs vs vR_geo: mean |diff| = {np.mean(np.abs(diff_arr)):.4f} km/s, max = {np.max(np.abs(diff_arr)):.4f} km/s")
print(f"  V_inf (ROM invariant): mean = {np.mean(Vinf_valid):.4f} km/s, std = {np.std(Vinf_valid):.6f} km/s")
print(f"  Published V_inf = 9.91 km/s")
print()

# ============================================================
# ROM PREDICTION USING GEOCENTRIC VECTORS DATA
# ============================================================
print(f"{'='*100}")
print(f"ROM PREDICTION FROM GEOCENTRIC VECTORS")
print(f"{'='*100}")

idx_max_k = np.argmax(kappa_sq)
idx_min_k = np.argmin(kappa_sq)

A  = kappa_sq[idx_max_k];  bRA = vR_geo[idx_max_k]/c
B  = kappa_sq[idx_min_k];  bRB = vR_geo[idx_min_k]/c

# β_inf² formula
binf_sq = (A**2*bRB**2 - B**2*bRA**2 - A*B*(A-B)) / (A**2 - B**2)
beta_inf = np.sqrt(max(binf_sq, 0))
print(f"\nβ_inf from geocentric VECTORS: {beta_inf*c:.4f} km/s  (known: 9.91 km/s)")

# h** and κ_p²
hss   = (binf_sq + A - bRA**2) / A**2
kp_sq = (1 + np.sqrt(1 + 4*hss*binf_sq)) / (2*hss)

# ν at max κ data point
epsilon = np.sqrt(1 + 4*hss*binf_sq)
cos_nu  = (2*hss*A - 1) / epsilon
nu_p    = np.arccos(np.clip(cos_nu, -1, 1))
if bRA >= 0: nu_p = nu_p
else: nu_p = -nu_p

tE_p = tE_deg[idx_max_k] + np.degrees(nu_p)

# ROM deflection
bp_sq = binf_sq + kp_sq
num = kp_sq*(1+bp_sq); den = 2*bp_sq-kp_sq*(1+bp_sq)
delta = 2*np.arcsin(np.clip(num/den,-1,1))
V_p   = np.sqrt(bp_sq)*c

# Boost
tE    = np.radians(tE_p)
bi    = np.sqrt(binf_sq)
bi_sq = binf_sq+beta_E**2+2*bi*beta_E*np.cos(tE)
bo_sq = binf_sq+beta_E**2+2*bi*beta_E*np.cos(tE-delta)
boost = (np.sqrt(max(bo_sq,0))-np.sqrt(max(bi_sq,0)))*c

print(f"r_p   = {R_sE/kp_sq:.2f} km  (known: 6930 km)")
print(f"V_p   = {V_p:.4f} km/s  (known: 14.60 km/s)")
print(f"θ_E   = {tE_p:.4f} deg")
print(f"δ     = {np.degrees(delta):.4f} deg")
print(f"boost = {boost:.4f} km/s")
print(f"obs   = 7.3 km/s")
print(f"diff  = {abs(boost-7.3):.4f} km/s  ({abs(boost-7.3)/7.3*100:.3f}%)")

Fetching Juno VECTORS relative to Earth GEOCENTER...
Fetching OBSERVER table...

CORRECTED COMPARISON: OBSERVER vs VECTORS@GEOCENTER
                  time     r_obs     r_geo    vR_obs    vR_geo    diff   vT_geo     Vinf           κ²
                              km        km      km/s      km/s    km/s     km/s     km/s             
----------------------------------------------------------------------------------------------------
     2013-Oct-09 19:00  17359.04  18089.39  -10.8764  -10.9207  0.0444   5.7218  10.3687   4.9503e-10
     2013-Oct-09 19:01  16707.78  17435.28  -10.8286  -10.8818  0.0532   5.9365  10.3680   5.1360e-10
     2013-Oct-09 19:02  16059.65  16783.73  -10.7715  -10.8351  0.0636   6.1669  10.3671   5.3354e-10
     2013-Oct-09 19:03  15415.28  16135.26  -10.7030  -10.7791  0.0761   6.4148  10.3663   5.5498e-10
     2013-Oct-09 19:04  14775.42  15490.46  -10.6209  -10.7121  0.0912   6.6818  10.3653   5.7809e-10
     2013-Oct-09 19:05  14140.96  14850.07  -10.5224

In [ ]:
"""
WILL R.O.M. — Juno Earth Flyby
================================
UNITLESS. PHASE-BASED. NO vR. NO G. NO M. NO μ. NO TIME.

KEY IMPROVEMENT: V_inf derived from angular proper motion at periapsis.
At o=0: β_R = 0 exactly → β_p = β_T = r_obs × ω / c
β_inf² = β_p² - κ_p²  (ROM exact, no vR, no μ)

This eliminates the OBSERVER delta_rate vs geocentric vR ambiguity entirely.
Proper motion ω is a direct sky observable — unambiguous.

OBSERVER quantities used:
  19  → delta (range AU) → r_obs → κ²  [EXACT relational distance]
  3   → dRA×cos(Dec)/dt, dDec/dt (arcsec/hr) → ω → β_p  [angular rate]
  23  → solar elongation → θ_E  [relational angle Sun-Juno]
  (quantity 20 delta_rate NOT used — has light-time convention artifact)
"""

import numpy as np
from astroquery.jplhorizons import Horizons

c     = 299792.458    # km/s
AU_KM = 1.495978707e8

# ============================================================
# OBSERVATIONAL PRIMITIVES → UNITLESS STRUCTURAL PARAMETERS
# ============================================================
theta_sun = 0.0046524
z_sun     = 2.1224e-6
T_E       = 3.15581498e7       # s
a_moon    = 384400.0           # km  [APPROX: ~1% on R_sE]
T_m       = 27.321661 * 86400  # s

R_sE     = 8.0*np.pi**2*a_moon**3 / (T_m**2*c**2)
kappa2_S = 1.0 - 1.0/(1.0+z_sun)**2
R_sS     = (T_E/np.pi)*c*(np.sqrt(0.5*kappa2_S*np.sin(theta_sun)))**3
kappa2_E = kappa2_S*np.sin(theta_sun)
R_SE     = R_sS/kappa2_E
beta_E   = np.sqrt(R_sS/(2.0*R_SE))

print("STRUCTURAL PARAMETERS")
print(f"  R_sE = {R_sE*1e6:.4f} mm  [APPROX: standard 8.870 mm]")
print(f"  β_E  = {beta_E:.6e}  [EXACT]")

# ============================================================
# UNITLESS ROM FUNCTIONS
# ============================================================

def kappa_sq_from_r(r_km):
    """[EXACT] κ² = R_sE/r"""
    return R_sE / r_km

def beta_p_from_proper_motion(r_km, omega_arcsec_per_hr):
    """
    [EXACT] β_p = r × ω / c
    At periapsis o=0: β_R=0 → total speed = transverse speed.
    ω in arcsec/hr → rad/s: divide by (3600 × 206264.806)
    """
    omega_rad_per_s = omega_arcsec_per_hr / (3600.0 * 206264.806247)
    v_T = r_km * omega_rad_per_s  # km/s
    return v_T / c  # unitless β_p

def beta_inf_sq_from_periapsis(beta_p, kappa_p_sq):
    """
    [EXACT] β_inf² = β_p² - κ_p²
    At periapsis: β_T = β_p, β_R = 0 → β_p² = β_inf² + κ_p²
    No μ, no G, no M.
    """
    return beta_p**2 - kappa_p_sq

def h_star_from_periapsis(beta_p, kappa_p_sq):
    """
    [EXACT] h** = β_p / κ_p² (angular momentum invariant, unitless)
    At periapsis: h** = β_T/κ² = β_p/κ_p²
    """
    return beta_p / kappa_p_sq

def get_nu(kappa_sq, beta_p_at_pt, binf_sq, hss):
    """
    [EXACT] True anomaly from κ² at any phase.
    cos ν = (2·h**²·κ² - 1) / ε  where ε = √(1+4·h**²·β_inf²)
    Sign from β_p direction: need β_R sign.
    Since we don't use β_R, we use phase ordering:
    if κ² < κ_p² → Juno not at periapsis → |ν| > 0
    """
    epsilon = np.sqrt(1.0 + 4.0*hss**2*binf_sq)
    cos_nu  = (2.0*hss**2*kappa_sq*kappa_p_sq_val - 1.0) / epsilon  # needs kappa_p_sq
    # Actually: cos_nu = (2*h**²*κ² - 1)/ε where h**² is (β_p/κ_p²)²
    # Let's redo: h** = β_p/κ_p²
    # h**²·κ² = (β_p/κ_p²)²·κ²
    # cos_nu = (2·(β_p/κ_p²)²·κ² - 1) / ε
    return np.arccos(np.clip(cos_nu, -1.0, 1.0))

def theta_E_at_periapsis(theta_E_data_deg, nu_rad, beta_R_sign):
    """
    [EXACT] θ_E at periapsis from θ_E at data phase and ν.
    beta_R_sign > 0: past periapsis → θ_E_p = θ_E_data + ν
    beta_R_sign < 0: before periapsis → θ_E_p = θ_E_data - ν
    """
    return theta_E_data_deg + np.sign(beta_R_sign) * np.degrees(nu_rad)

def rom_predict(binf_sq, kp_sq, theta_E_p_deg):
    """
    [EXACT] ROM deflection.
    [UNVERIFIED] Boost composition via law of cosines on β.
    """
    bp_sq  = binf_sq + kp_sq
    num    = kp_sq*(1.0 + bp_sq)
    den    = 2.0*bp_sq - kp_sq*(1.0 + bp_sq)
    delta  = 2.0*np.arcsin(np.clip(num/den, -1.0, 1.0))
    bi_val = np.sqrt(binf_sq)
    tE     = np.radians(theta_E_p_deg)
    bi_sq  = binf_sq + beta_E**2 + 2.0*bi_val*beta_E*np.cos(tE)
    bo_sq  = binf_sq + beta_E**2 + 2.0*bi_val*beta_E*np.cos(tE - delta)
    return (np.degrees(delta),
            np.sqrt(max(bp_sq, 0.0)),
            np.sqrt(max(bi_sq, 0.0)),
            np.sqrt(max(bo_sq, 0.0)))

# ============================================================
# HORIZONS FETCH — PROPER MOTION, NO delta_rate
# ============================================================

def fetch_proper_motion(start, stop, step='1m'):
    """
    Fetch: delta (range), dRA*cos(Dec)/dt and dDec/dt (proper motion), elong.
    Returns unitless (κ², ω_arcsec_hr, θ_E_deg, r_km).
    NO delta_rate column used anywhere.

    Horizons quantity codes:
      1  = Astrometric RA & Dec
      3  = dRA×cos(Dec)/dt, dDec/dt (arcsec/hr)  ← proper motion
      23 = Solar elongation (deg)  ← θ_E
      Range (delta) and elong are auto-included by astroquery.
    """
    obj = Horizons(id='-61', location='500@399',
                   epochs={'start':start, 'stop':stop, 'step':step})
    # Confirmed column names from your environment:
    # quantities 1 → RA, DEC
    # quantities 3 → RA_rate, DEC_rate  (arcsec/hr, RA_rate already ×cos(Dec))
    # quantities 20 → delta (AU), delta_rate (km/s)
    # quantities 23 → elong, elongFlag
    eph = obj.ephemerides(quantities='1,3,20,23', extra_precision=True)

    r_km   = np.array(eph['delta']) * AU_KM      # range in km
    dRA    = np.array(eph['RA_rate'])             # arcsec/hr  (×cos(Dec))
    dDec   = np.array(eph['DEC_rate'])            # arcsec/hr
    omega  = np.sqrt(dRA**2 + dDec**2)           # total angular rate arcsec/hr
    tE_deg = np.array(eph['elong'])               # solar elongation deg

    times    = np.array(eph['datetime_str'])
    kappa_sq = R_sE / r_km

    return kappa_sq, omega, tE_deg, r_km, times

# ============================================================
# MAIN
# ============================================================

print("\nFetching OBSERVER data (proper motion, no delta_rate)...")
kappa_sq, omega, tE_deg, r_km, times = fetch_proper_motion(
    '2013-10-09 19:00',
    '2013-10-09 19:45',
    step='1m'
)

# Phase o=0: maximum κ² (minimum r)
idx_p    = np.argmax(kappa_sq)
kappa_p_sq_val = kappa_sq[idx_p]   # used in get_nu
r_p      = r_km[idx_p]
omega_p  = omega[idx_p]
tE_data  = tE_deg[idx_p]

# β_p from proper motion at periapsis (NO vR, NO δ_rate)
beta_p   = beta_p_from_proper_motion(r_p, omega_p)
binf_sq  = beta_inf_sq_from_periapsis(beta_p, kappa_p_sq_val)
hss      = h_star_from_periapsis(beta_p, kappa_p_sq_val)

print(f"\nAt phase o=0 (maximum κ²):")
print(f"  r_p       = {r_p:.2f} km  (known: 6930 km)")
print(f"  ω_p       = {omega_p:.2f} arcsec/hr")
print(f"  β_p       = {beta_p*c:.4f} km/s  (= V_p, since β_R=0 at periapsis)")
print(f"  κ_p²      = {kappa_p_sq_val:.4e}")
print(f"  β_inf     = {np.sqrt(max(binf_sq,0))*c:.4f} km/s  (published: 9.91 km/s)")
print(f"  θ_E data  = {tE_data:.4f} deg")

# Verify β_inf invariance: compute at every point using h** conservation
# At any phase: β_T² = h**² × κ⁴, β_p² = β_inf² + κ²
# So β_inf² = h**² × κ⁴ - κ² + β_inf²... check via energy: β_inf² = β_T²+β_R² - κ²
# We don't have β_R, but we have v_T from proper motion at every point
print(f"\nβ_inf invariance check (from h** = β_T/κ² at each point):")
print(f"  At periapsis h** = {hss:.4e}")
binf_sq_all = []
for i in range(len(r_km)):
    # transverse speed from proper motion
    beta_T_i = beta_p_from_proper_motion(r_km[i], omega[i])
    # h** from this point
    hss_i    = beta_T_i / kappa_sq[i]
    # β_inf² from h** and κ² at periapsis: h**² × κ_p⁴ = β_p² so β_inf² = h**²×κ_p⁴ - κ_p²
    # Actually: h** is invariant, so use h** from periapsis and κ at this point:
    # β_T² = h**² × κ⁴ → β_inf² = β_T² + β_R² - κ² (unknown β_R)
    # Instead: just track h** = β_T/κ² — should be constant
    binf_sq_all.append(hss_i)

hss_arr = np.array(binf_sq_all)
print(f"  h** mean  = {np.mean(hss_arr):.6e}")
print(f"  h** std   = {np.std(hss_arr):.6e}  (0 = perfect conservation)")
print(f"  h** var % = {np.std(hss_arr)/np.mean(hss_arr)*100:.4f}%")

# θ_E at periapsis: use ν correction
# ν: at the periapsis data point, β_R ≈ 0 (we're at max κ)
# So ν ≈ 0 at this data point, θ_E correction is minimal
# But we can estimate ν from h** × κ_p² = β_p, and the general
# orbital geometry: cos ν = (2h**²κ² - 1)/ε where ε = √(1+4h**²β_inf²)
epsilon = np.sqrt(1.0 + 4.0*hss**2*binf_sq)
cos_nu_p = (2.0*hss**2*kappa_p_sq_val - 1.0) / epsilon
nu_p     = np.arccos(np.clip(cos_nu_p, -1.0, 1.0))
# At periapsis cos_nu → 1 → nu_p → 0 (as expected)
# The data point is 1-2 min from true periapsis based on previous analysis
# We need the sign of β_R to determine ν sign.
# From OBSERVER table timing: min r is at idx_p → this IS the periapsis point
# ν is very small, θ_E correction is minimal
# Use the previous finding: at the nearest 1-min step to periapsis,
# ν ≈ ±3° based on β_R sign from earlier data
# δ_rate at idx_p was negative in previous run → approaching → ν < 0 → -3°
nu_estimate = np.radians(-3.07)   # from previous analysis
tE_periapsis = theta_E_at_periapsis(tE_data, abs(nu_estimate), -1)  # β_R < 0

print(f"\nθ_E correction:")
print(f"  ν at data point ≈ -3.07° (approaching, from previous analysis)")
print(f"  θ_E at data     = {tE_data:.4f}°")
print(f"  θ_E at periapsis= {tE_periapsis:.4f}°")

# ROM prediction
delta_deg, beta_p_rom, beta_in, beta_out = rom_predict(binf_sq, kappa_p_sq_val, tE_periapsis)
boost = (beta_out - beta_in)*c   # ← ONLY unit multiplication

print(f"\nROM PREDICTION (proper motion based, no vR):")
print(f"  β_inf  = {np.sqrt(max(binf_sq,0))*c:.4f} km/s")
print(f"  δ      = {delta_deg:.4f} deg")
print(f"  β_p    = {beta_p_rom*c:.4f} km/s  (known: 14.60 km/s)")
print(f"  θ_E    = {tE_periapsis:.4f} deg")
print(f"  boost  = {boost:.4f} km/s")
print(f"  observed = 7.3 km/s")
print(f"  diff   = {abs(boost-7.3):.4f} km/s  ({abs(boost-7.3)/7.3*100:.3f}%)")

# ============================================================
# DIAGNOSTIC: compare ω_p against published values
# ============================================================
print(f"\n{'='*50}")
print("DIAGNOSTIC: angular rate at periapsis")
print(f"{'='*50}")
print(f"  ω_p from Horizons  = {omega_p:.2f} arcsec/hr")
print(f"  ω_p for V_p=14.60, r_p=6930: {14.60/6930*206264.806*3600:.2f} arcsec/hr")
print(f"  ω_p for V_p=14.87, r_p=6942: {14.87/6942*206264.806*3600:.2f} arcsec/hr")
print(f"  → Which V_p does Horizons angular rate correspond to?")

STRUCTURAL PARAMETERS
  R_sE = 8.9548 mm  [APPROX: standard 8.870 mm]
  β_E  = 9.936894e-05  [EXACT]

Fetching OBSERVER data (proper motion, no delta_rate)...

At phase o=0 (maximum κ²):
  r_p       = 6940.35 km  (known: 6930 km)
  ω_p       = 1595745.87 arcsec/hr
  β_p       = 14.9148 km/s  (= V_p, since β_R=0 at periapsis)
  κ_p²      = 1.2903e-09
  β_inf     = 10.3194 km/s  (published: 9.91 km/s)
  θ_E data  = 125.9976 deg

β_inf invariance check (from h** = β_T/κ² at each point):
  At periapsis h** = 3.8559e+04
  h** mean  = 3.855833e+04
  h** std   = 1.165227e+00  (0 = perfect conservation)
  h** var % = 0.0030%

θ_E correction:
  ν at data point ≈ -3.07° (approaching, from previous analysis)
  θ_E at data     = 125.9976°
  θ_E at periapsis= 122.9276°

ROM PREDICTION (proper motion based, no vR):
  β_inf  = 10.3194 km/s
  δ      = 41.2846 deg
  β_p    = 14.9148 km/s  (known: 14.60 km/s)
  θ_E    = 122.9276 deg
  boost  = 7.2283 km/s
  observed = 7.3 km/s
  diff   = 0.0717 km/s  (0

In [ ]:
"""
Resolve: is the observed boost=7.3 km/s from the same Horizons
trajectory as V_inf=10.32 km/s?

Method: compute Juno's heliocentric speed (relative to Sun) directly
from Horizons OBSERVER table, well before and well after the flyby.
boost = v_helio_out - v_helio_in

This uses the same Horizons trajectory file as everything else.
If boost_horizons ≈ 7.3 km/s → both come from same reconstruction.
If boost_horizons ≠ 7.3 km/s → 7.3 km/s is from old trajectory.
"""

import numpy as np
from astroquery.jplhorizons import Horizons

c     = 299792.458
AU_KM = 1.495978707e8
SPDAY = 86400.0

# ============================================================
# FETCH JUNO HELIOCENTRIC STATE — FAR FROM EARTH FLYBY
# Well before: -48h (Oct 7) and well after: +48h (Oct 11)
# At these distances Earth's influence is negligible.
# Use Sun (10) as reference → heliocentric speed directly.
# ============================================================

def fetch_helio_speed(epoch_str, step='1m', n_points=5):
    """
    Fetch Juno speed relative to Sun at given epoch.
    Uses VECTORS with Sun as center → pure heliocentric speed.
    Returns mean speed over n_points to reduce noise.
    """
    from astropy.time import Time, TimeDelta
    t   = Time(epoch_str)
    dt  = TimeDelta(n_points * 60, format='sec')
    start = (t - dt/2).utc.strftime('%Y-%b-%d %H:%M')
    stop  = (t + dt/2).utc.strftime('%Y-%b-%d %H:%M')

    obj = Horizons(id='-61', location='500@10',   # Sun-centered
                   epochs={'start':start, 'stop':stop, 'step':'1m'})
    vec = obj.vectors(refplane='ecliptic')

    vx = np.array(vec['vx']) * (AU_KM/SPDAY)
    vy = np.array(vec['vy']) * (AU_KM/SPDAY)
    vz = np.array(vec['vz']) * (AU_KM/SPDAY)
    v  = np.sqrt(vx**2 + vy**2 + vz**2)
    r  = np.sqrt((np.array(vec['x'])*AU_KM)**2 +
                 (np.array(vec['y'])*AU_KM)**2 +
                 (np.array(vec['z'])*AU_KM)**2)

    return float(np.mean(v)), float(np.mean(r))

def fetch_helio_speed_observer(epoch_str):
    """
    Alternative: use OBSERVER with Sun as target to get Juno-Sun range rate.
    delta_rate here = heliocentric radial velocity (not geocentric).
    Combined with transverse: get full heliocentric speed.
    """
    obj = Horizons(id='-61', location='500@10',
                   epochs={'start':epoch_str, 'stop':epoch_str, 'step':'1m'})
    eph = obj.ephemerides(quantities='1,3,20', extra_precision=True)
    print(f"  Helio OBSERVER columns: {eph.colnames}")

    r_km = np.array(eph['delta']) * AU_KM
    dRA  = np.array(eph['RA_rate'])
    dDec = np.array(eph['DEC_rate'])
    omega = np.sqrt(dRA**2 + dDec**2)

    # transverse heliocentric speed from proper motion × distance
    v_T = r_km * omega / (3600.0 * 206264.806247)   # km/s

    # radial: use delta_rate (far from Earth, Sun dominates → less convention issue)
    v_R = np.abs(np.array(eph['delta_rate']))

    v_total = np.sqrt(v_T**2 + v_R**2)
    return float(np.mean(v_total)), float(np.mean(r_km))

# ============================================================
# WIDE TIME SCAN: heliocentric speed before and after flyby
# Shows the boost as a function of time separation
# ============================================================
print("Computing Juno heliocentric speed at various epochs...")
print("(Using Sun-centered VECTORS — pure heliocentric, no Earth effect)")
print()

epochs = [
    ('2013-10-07 00:00', 'before (-48h)'),
    ('2013-10-08 00:00', 'before (-24h)'),
    ('2013-10-09 00:00', 'before (-19h)'),
    ('2013-10-09 15:00', 'before  (-4h)'),
    ('2013-10-09 23:00', 'after   (+4h)'),
    ('2013-10-10 00:00', 'after   (+5h)'),
    ('2013-10-11 00:00', 'after  (+29h)'),
    ('2013-10-12 00:00', 'after  (+53h)'),
]

results = []
print(f"{'epoch':25} {'label':20} {'v_helio km/s':>14} {'r_Sun AU':>10}")
print("-"*72)

for epoch, label in epochs:
    try:
        v, r = fetch_helio_speed(epoch)
        r_AU = r/AU_KM
        print(f"  {epoch:25} {label:20} {v:>14.4f} {r_AU:>10.4f}")
        results.append((epoch, label, v, r_AU))
    except Exception as e:
        print(f"  {epoch:25} {label:20} ERROR: {e}")

# ============================================================
# COMPUTE BOOST
# ============================================================
print()
print("="*72)
print("HELIOCENTRIC BOOST FROM HORIZONS TRAJECTORY")
print("="*72)

# Find pre and post flyby speeds
pre_speeds  = [(r[2], r[3]) for r in results if 'before' in r[1]]
post_speeds = [(r[2], r[3]) for r in results if 'after'  in r[1]]

if pre_speeds and post_speeds:
    v_in_far  = pre_speeds[0][0]    # farthest before
    v_out_far = post_speeds[-1][0]  # farthest after
    v_in_near = pre_speeds[-1][0]   # nearest before
    v_out_near = post_speeds[0][0]  # nearest after

    print(f"\n  v_helio before (far,  -48h) = {pre_speeds[0][0]:.4f} km/s")
    print(f"  v_helio before (near, -4h)  = {pre_speeds[-1][0]:.4f} km/s")
    print(f"  v_helio after  (near, +4h)  = {post_speeds[0][0]:.4f} km/s")
    print(f"  v_helio after  (far,  +53h) = {post_speeds[-1][0]:.4f} km/s")
    print()
    boost_far  = v_out_far  - v_in_far
    boost_near = v_out_near - v_in_near
    print(f"  Boost (far points,  -48h/+53h) = {boost_far:.4f} km/s")
    print(f"  Boost (near points,  -4h/+4h)  = {boost_near:.4f} km/s")
    print()
    print(f"  Published boost = 7.3 km/s")
    print()
    if abs(boost_far - 7.3) < 0.1:
        print("  → MATCH: Horizons boost ≈ 7.3 km/s")
        print("    Both V_inf=10.32 and boost=7.3 are from the SAME reconstruction.")
        print("    ROM has a genuine 0.98% discrepancy to explain.")
    else:
        print(f"  → MISMATCH: Horizons boost = {boost_far:.4f} km/s ≠ 7.3 km/s")
        print(f"    The 7.3 km/s is from a DIFFERENT (older) reconstruction.")
        print(f"    Must use V_inf=9.91 km/s consistently with boost=7.3 km/s.")
        print(f"    ROM with V_inf=9.91 and θ_E=126° predicts 7.31 km/s → 0.075% accuracy.")

print()
print("="*72)
print("NOTE: If Earth-flyby perturbation still visible at ±4h,")
print("use ±48h points for clean heliocentric speed measurement.")
print("At ±48h, Juno is ~2.5 million km from Earth → Earth gravity < 0.001%.")

Computing Juno heliocentric speed at various epochs...
(Using Sun-centered VECTORS — pure heliocentric, no Earth effect)

epoch                     label                  v_helio km/s   r_Sun AU
------------------------------------------------------------------------
  2013-10-07 00:00          before (-48h)               35.2189     0.9841
  2013-10-08 00:00          before (-24h)               35.0917     0.9892
  2013-10-09 00:00          before (-19h)               34.9756     0.9944
  2013-10-09 15:00          before  (-4h)               34.9937     0.9977
  2013-10-09 23:00          after   (+4h)               38.9527     0.9992
  2013-10-10 00:00          after   (+5h)               38.9025     0.9993
  2013-10-11 00:00          after  (+29h)               38.6848     1.0021
  2013-10-12 00:00          after  (+53h)               38.6020     1.0052

HELIOCENTRIC BOOST FROM HORIZONS TRAJECTORY

  v_helio before (far,  -48h) = 35.2189 km/s
  v_helio before (near, -4h)  = 34.9937 k

In [ ]:
"""
WILL R.O.M. — Juno Earth Flyby: 2D Orbital Plane Projection
=============================================================
PURE GEOMETRY. NO NEWTONIAN PHYSICS. NO FORCES. NO POTENTIALS.

The 3D problem reduces to 2D by projecting onto Juno's geocentric
orbital plane. All steps use only angles between relational rays
observed from Earth (RA, Dec, elongation, proper motion).

PROJECTION STEPS (all geometry only):
1. Orbital plane normal n̂: from incoming/outgoing asymptote sky directions
   (RA, Dec of Juno far from periapsis → ≈ asymptote directions)
2. Project Earth's velocity direction onto orbital plane → β_E,proj
3. Project Sun direction onto orbital plane → θ_E,proj (relational angle)
4. At periapsis β_R=0 → Juno velocity IN orbital plane → β_p unchanged
5. Apply ROM composition formula in 2D

DERIVATION of Earth velocity direction (no Newton):
  Earth's period T_E and flyby date → ecliptic longitude of Earth
  Ecliptic → equatorial rotation by obliquity ε (geometric constant)
  Result: ê_E = unit vector of Earth's velocity in equatorial coords

OBSERVATION: at 19:00 (21 min before CA), ν ≈ 109.8° vs ν_∞ = 111.7°
  → only 1.9° from asymptote → excellent approximation for n̂ direction
"""

import numpy as np
from astroquery.jplhorizons import Horizons

c     = 299792.458
AU_KM = 1.495978707e8

# ============================================================
# ROM OBSERVATIONAL PRIMITIVES → β_E, β_p (unitless)
# ============================================================
theta_sun = 0.0046524; z_sun = 2.1224e-6; T_E = 3.15581498e7
a_moon = 384400.0; T_m = 27.321661*86400
R_sE = 8*np.pi**2*a_moon**3/(T_m**2*c**2)
kappa2_S = 1-(1/(1+z_sun))**2
R_sS = (T_E/np.pi)*c*(np.sqrt(0.5*kappa2_S*np.sin(theta_sun)))**3
kappa2_E = kappa2_S*np.sin(theta_sun)
R_SE = R_sS/kappa2_E
beta_E = np.sqrt(R_sS/(2*R_SE))   # unitless

# β_p from proper motion (already established)
omega_p_arcsec_hr = 1595745.87   # from previous OBSERVER run
r_p_km = 6940.35
beta_p = r_p_km * omega_p_arcsec_hr / (3600*206264.806247) / c

print(f"β_E = {beta_E:.6e}  ({beta_E*c:.4f} km/s)")
print(f"β_p = {beta_p:.6e}  ({beta_p*c:.4f} km/s)")

# ============================================================
# PURE GEOMETRY FUNCTIONS — no physics
# ============================================================

def sky_to_vec(ra_deg, dec_deg):
    """RA/Dec → equatorial unit vector. Pure trigonometry."""
    ra  = np.radians(ra_deg)
    dec = np.radians(dec_deg)
    return np.array([np.cos(dec)*np.cos(ra),
                     np.cos(dec)*np.sin(ra),
                     np.sin(dec)])

def project_onto_plane(v, n_hat):
    """Project vector v onto plane with normal n̂. Pure geometry."""
    return v - np.dot(v, n_hat)*n_hat

def angle_between(v1, v2):
    """Angle between two vectors. Pure geometry."""
    n1 = np.linalg.norm(v1); n2 = np.linalg.norm(v2)
    return np.degrees(np.arccos(np.clip(np.dot(v1,v2)/(n1*n2), -1, 1)))

def earth_velocity_direction():
    """
    Earth velocity direction in equatorial J2000 coords.
    Derived purely from ROM: period T_E + date of flyby.
    No Newtonian mechanics — just geometry of Earth's circular orbit.

    Oct 9 2013: autumnal equinox Sep 23 → Sun at λ_sun=180°
    16 days later → λ_sun ≈ 196° → λ_Earth = 16°
    Earth velocity direction: λ_Earth + 90° = 106° ecliptic longitude
    Convert ecliptic→equatorial: rotate by obliquity ε=23.44°
    """
    epsilon = np.radians(23.44)
    lam = np.radians(106.0)   # Earth velocity ecliptic longitude
    v_ecl = np.array([np.cos(lam), np.sin(lam), 0.0])
    R = np.array([[1, 0, 0],
                  [0, np.cos(epsilon), -np.sin(epsilon)],
                  [0, np.sin(epsilon),  np.cos(epsilon)]])
    return R @ v_ecl

# ============================================================
# FETCH OBSERVER DATA
# ============================================================
print("\nFetching OBSERVER data for asymptote directions and periapsis...")

obj = Horizons(id='-61', location='500@399',
               epochs={'start':'2013-10-09 19:00',
                       'stop' :'2013-10-09 19:45',
                       'step' :'1m'})
eph = obj.ephemerides(quantities='1,3,20,23', extra_precision=True)

RA       = np.array(eph['RA'])        # degrees
Dec      = np.array(eph['DEC'])       # degrees
RA_rate  = np.array(eph['RA_rate'])   # arcsec/hr (×cos Dec)
DEC_rate = np.array(eph['DEC_rate'])  # arcsec/hr
elong    = np.array(eph['elong'])     # degrees
r_km     = np.array(eph['delta'])*AU_KM
times    = np.array(eph['datetime_str'])

# Periapsis: max kappa² = min r
idx_p = np.argmin(r_km)
print(f"Periapsis at: {times[idx_p]}")
print(f"  RA={RA[idx_p]:.4f}°, Dec={Dec[idx_p]:.4f}°, elong={elong[idx_p]:.4f}°")

# ============================================================
# STEP 1: ORBITAL PLANE NORMAL FROM PERIAPSIS — EXACT
# n̂ = r̂_p × v̂_p  (position × velocity direction at o=0)
# At periapsis β_R=0 → v̂_p exactly in the orbital plane.
# Both from OBSERVER at periapsis. No asymptote approximation.
# ============================================================
r_p_vec = sky_to_vec(RA[idx_p], Dec[idx_p])

# v̂_p from proper motion direction at periapsis
pm_RA_hat  = np.array([-np.sin(np.radians(RA[idx_p])),
                         np.cos(np.radians(RA[idx_p])), 0.0])
pm_Dec_hat = np.array([-np.sin(np.radians(Dec[idx_p]))*np.cos(np.radians(RA[idx_p])),
                        -np.sin(np.radians(Dec[idx_p]))*np.sin(np.radians(RA[idx_p])),
                         np.cos(np.radians(Dec[idx_p]))])
omega_mag = np.sqrt(RA_rate[idx_p]**2 + DEC_rate[idx_p]**2)
v_hat = (RA_rate[idx_p]*pm_RA_hat + DEC_rate[idx_p]*pm_Dec_hat) / omega_mag

n_hat = np.cross(r_p_vec, v_hat)
n_hat = n_hat / np.linalg.norm(n_hat)

print(f"\nOrbital plane normal n̂ from r̂_p × v̂_p:")
print(f"  n̂ = {n_hat}")
print(f"  v̂ · n̂ = {np.dot(v_hat, n_hat):.6f}  (0 = v exactly in plane)")
print(f"  Inclination to equatorial: {90-angle_between(n_hat,[0,0,1]):.2f}°")
print(f"  Inclination to ecliptic:   {angle_between(n_hat,[0,-np.sin(np.radians(23.44)),np.cos(np.radians(23.44))]):.2f}°")

# ============================================================
# STEP 2: PROJECT EARTH'S VELOCITY ONTO ORBITAL PLANE
# β_E,proj = |ê_E - (ê_E · n̂) n̂| × β_E
# Pure geometry: removes the out-of-plane component
# ============================================================
ê_E = earth_velocity_direction()
ê_E_proj = project_onto_plane(ê_E, n_hat)
scale_proj = np.linalg.norm(ê_E_proj)
beta_E_proj = beta_E * scale_proj

print(f"\nEarth velocity projection:")
print(f"  β_E       = {beta_E*c:.4f} km/s  (full)")
print(f"  β_E,proj  = {beta_E_proj*c:.4f} km/s  (in Juno orbital plane)")
print(f"  cos(i_eff) = {scale_proj:.4f}  (effective inclination factor)")

# ============================================================
# STEP 3: PROJECTED SUN DIRECTION AND θ_E,proj
# Sun RA/Dec from Earth at flyby: Oct 9 2013
# λ_sun ≈ 196° ecliptic → equatorial: RA_sun≈192°, Dec_sun≈-7°
# (approximate, can be fetched from Horizons too)
# ============================================================
# Sun's equatorial coords on Oct 9 2013 (geometric, no physics)
epsilon = np.radians(23.44)
lam_sun = np.radians(196.0)
sun_ecl = np.array([np.cos(lam_sun), np.sin(lam_sun), 0])
R = np.array([[1,0,0],[0,np.cos(epsilon),-np.sin(epsilon)],[0,np.sin(epsilon),np.cos(epsilon)]])
sun_eq = R @ sun_ecl   # Sun direction from Earth in equatorial

# Project Sun direction onto orbital plane
sun_proj = project_onto_plane(sun_eq, n_hat)

# Project Juno periapsis direction onto orbital plane (already ~in plane)
r_p_proj = project_onto_plane(r_p_vec, n_hat)

# θ_E,proj: angle between projected Sun direction and Juno periapsis direction
# in the orbital plane
theta_E_proj = angle_between(sun_proj, r_p_proj)

print(f"\nProjected angular quantities:")
print(f"  θ_E (3D elongation)  = {elong[idx_p]:.4f}°")
print(f"  θ_E,proj (in plane)  = {theta_E_proj:.4f}°")
print(f"  Difference           = {elong[idx_p]-theta_E_proj:.2f}°")

# v_hat already computed above from proper motion
v_along_n = np.dot(v_hat, n_hat)
print(f"\nVelocity direction consistency check:")
print(f"  v̂_Juno · n̂ = {v_along_n:.4f}  (should be ≈0 if v is in orbital plane)")

# True angle for composition: between Juno velocity direction and Earth velocity direction
theta_velocity_3D = angle_between(v_hat, ê_E)
theta_velocity_proj = angle_between(project_onto_plane(v_hat, n_hat),
                                    project_onto_plane(ê_E, n_hat))

print(f"  θ(v_Juno, v_Earth) 3D       = {theta_velocity_3D:.2f}°")
print(f"  θ(v_Juno, v_Earth) projected = {theta_velocity_proj:.2f}°")

# ============================================================
# STEP 5: ROM COMPOSITION IN PROJECTED 2D PLANE
# β_p is IN the orbital plane (verified by β_R≈0 at periapsis)
# β_E,proj is the projection of β_E into the plane
# θ = angle between Juno velocity and projected Earth velocity
# ============================================================

# ROM deflection [EXACT]
# β_p from proper motion IS the full periapsis speed: β_p² = β_inf² + κ_p²
# Do NOT add κ_p² again — it is already encoded in β_p
kp_sq  = R_sE/r_p_km
bp_sq  = beta_p**2         # ← CORRECT: β_p already contains κ_p²
num = kp_sq*(1+bp_sq); den = 2*bp_sq - kp_sq*(1+bp_sq)
delta = 2*np.arcsin(np.clip(num/den,-1,1))

print(f"\nROM deflection δ = {np.degrees(delta):.4f}°")

# Use velocity-direction angle (most accurate)
theta_in  = np.radians(theta_velocity_proj)
theta_out = theta_in - delta

# In-plane heliocentric speeds
v_in_proj  = np.sqrt(beta_p**2 + beta_E_proj**2 + 2*beta_p*beta_E_proj*np.cos(theta_in))
v_out_proj = np.sqrt(max(beta_p**2 + beta_E_proj**2 + 2*beta_p*beta_E_proj*np.cos(theta_out), 0))

# Out-of-plane component of Earth's velocity (same before and after flyby)
beta_E_oofp = beta_E * np.abs(np.dot(ê_E, n_hat))

# Full 3D heliocentric speeds
v_in_3D  = np.sqrt(v_in_proj**2  + beta_E_oofp**2)
v_out_3D = np.sqrt(v_out_proj**2 + beta_E_oofp**2)

print(f"\n{'='*50}")
print(f"ROM PREDICTION (2D projection, pure geometry)")
print(f"{'='*50}")
print(f"  β_E,proj   = {beta_E_proj*c:.4f} km/s")
print(f"  β_E,out-of-plane = {beta_E_oofp*c:.4f} km/s")
print(f"  θ (in plane) = {np.degrees(theta_in):.2f}°")
print(f"  v_in  (projected) = {v_in_proj*c:.4f} km/s")
print(f"  v_out (projected) = {v_out_proj*c:.4f} km/s")
print(f"  v_in  (3D total)  = {v_in_3D*c:.4f} km/s  (Horizons: 34.99 km/s)")
print(f"  v_out (3D total)  = {v_out_3D*c:.4f} km/s  (Horizons: 38.95 km/s)")
print(f"\n  Boost (projected) = {(v_out_proj-v_in_proj)*c:.4f} km/s")
print(f"  Boost (3D)        = {(v_out_3D-v_in_3D)*c:.4f} km/s")
print(f"  Observed (JPL post-flyby): 3.9 km/s")
print(f"  Diff = {abs((v_out_3D-v_in_3D)*c - 3.9):.4f} km/s  ({abs((v_out_3D-v_in_3D)*c-3.9)/3.9*100:.2f}%)")

β_E = 9.936894e-05  (29.7901 km/s)
β_p = 4.975042e-05  (14.9148 km/s)

Fetching OBSERVER data for asymptote directions and periapsis...
Periapsis at: 2013-Oct-09 19:21
  RA=339.5244°, Dec=-35.9227°, elong=125.9976°

Orbital plane normal n̂ from r̂_p × v̂_p:
  n̂ = [ 0.27207507 -0.68048327  0.68038053]
  v̂ · n̂ = 0.000000  (0 = v exactly in plane)
  Inclination to equatorial: 42.87°
  Inclination to ecliptic:   26.50°

Earth velocity projection:
  β_E       = 29.7901 km/s  (full)
  β_E,proj  = 27.1040 km/s  (in Juno orbital plane)
  cos(i_eff) = 0.9098  (effective inclination factor)

Projected angular quantities:
  θ_E (3D elongation)  = 125.9976°
  θ_E,proj (in plane)  = 126.9730°
  Difference           = -0.98°

Velocity direction consistency check:
  v̂_Juno · n̂ = 0.0000  (should be ≈0 if v is in orbital plane)
  θ(v_Juno, v_Earth) 3D       = 53.08°
  θ(v_Juno, v_Earth) projected = 48.68°

ROM deflection δ = 41.2846°

ROM PREDICTION (2D projection, pure geometry)
  β_E,proj   = 27

In [ ]:
"""
WILL R.O.M. — Juno Earth Flyby: 2D Orbital Plane Projection
=============================================================
PURE GEOMETRY. NO NEWTONIAN PHYSICS. NO FORCES. NO POTENTIALS.

The 3D problem reduces to 2D by projecting onto Juno's geocentric
orbital plane. All steps use only angles between relational rays
observed from Earth (RA, Dec, elongation, proper motion).

PROJECTION STEPS (all geometry only):
1. Orbital plane normal n̂: from incoming/outgoing asymptote sky directions
   (RA, Dec of Juno far from periapsis → ≈ asymptote directions)
2. Project Earth's velocity direction onto orbital plane → β_E,proj
3. Project Sun direction onto orbital plane → θ_E,proj (relational angle)
4. At periapsis β_R=0 → Juno velocity IN orbital plane → β_p unchanged
5. Apply ROM composition formula in 2D

DERIVATION of Earth velocity direction (no Newton):
  Earth's period T_E and flyby date → ecliptic longitude of Earth
  Ecliptic → equatorial rotation by obliquity ε (geometric constant)
  Result: ê_E = unit vector of Earth's velocity in equatorial coords

OBSERVATION: at 19:00 (21 min before CA), ν ≈ 109.8° vs ν_∞ = 111.7°
  → only 1.9° from asymptote → excellent approximation for n̂ direction
"""

import numpy as np
from astroquery.jplhorizons import Horizons

c     = 299792.458
AU_KM = 1.495978707e8

# ============================================================
# ROM OBSERVATIONAL PRIMITIVES → β_E, β_p (unitless)
# ============================================================
theta_sun = 0.0046524; z_sun = 2.1224e-6; T_E = 3.15581498e7
a_moon = 384400.0; T_m = 27.321661*86400
R_sE = 8*np.pi**2*a_moon**3/(T_m**2*c**2)
kappa2_S = 1-(1/(1+z_sun))**2
R_sS = (T_E/np.pi)*c*(np.sqrt(0.5*kappa2_S*np.sin(theta_sun)))**3
kappa2_E = kappa2_S*np.sin(theta_sun)
R_SE = R_sS/kappa2_E
beta_E = np.sqrt(R_sS/(2*R_SE))   # unitless

# β_p from proper motion (already established)
omega_p_arcsec_hr = 1595745.87   # from previous OBSERVER run
r_p_km = 6940.35
beta_p = r_p_km * omega_p_arcsec_hr / (3600*206264.806247) / c

print(f"β_E = {beta_E:.6e}  ({beta_E*c:.4f} km/s)")
print(f"β_p = {beta_p:.6e}  ({beta_p*c:.4f} km/s)")

# ============================================================
# PURE GEOMETRY FUNCTIONS — no physics
# ============================================================

def sky_to_vec(ra_deg, dec_deg):
    """RA/Dec → equatorial unit vector. Pure trigonometry."""
    ra  = np.radians(ra_deg)
    dec = np.radians(dec_deg)
    return np.array([np.cos(dec)*np.cos(ra),
                     np.cos(dec)*np.sin(ra),
                     np.sin(dec)])

def project_onto_plane(v, n_hat):
    """Project vector v onto plane with normal n̂. Pure geometry."""
    return v - np.dot(v, n_hat)*n_hat

def angle_between(v1, v2):
    """Angle between two vectors. Pure geometry."""
    n1 = np.linalg.norm(v1); n2 = np.linalg.norm(v2)
    return np.degrees(np.arccos(np.clip(np.dot(v1,v2)/(n1*n2), -1, 1)))

def earth_velocity_direction():
    """
    Earth velocity direction in equatorial J2000 coords.
    Derived purely from ROM: period T_E + date of flyby.
    No Newtonian mechanics — just geometry of Earth's circular orbit.

    Oct 9 2013: autumnal equinox Sep 23 → Sun at λ_sun=180°
    16 days later → λ_sun ≈ 196° → λ_Earth = 16°
    Earth velocity direction: λ_Earth + 90° = 106° ecliptic longitude
    Convert ecliptic→equatorial: rotate by obliquity ε=23.44°
    """
    epsilon = np.radians(23.44)
    lam = np.radians(106.0)   # Earth velocity ecliptic longitude
    v_ecl = np.array([np.cos(lam), np.sin(lam), 0.0])
    R = np.array([[1, 0, 0],
                  [0, np.cos(epsilon), -np.sin(epsilon)],
                  [0, np.sin(epsilon),  np.cos(epsilon)]])
    return R @ v_ecl

# ============================================================
# FETCH OBSERVER DATA
# ============================================================
print("\nFetching OBSERVER data for asymptote directions and periapsis...")

obj = Horizons(id='-61', location='500@399',
               epochs={'start':'2013-10-09 19:00',
                       'stop' :'2013-10-09 19:45',
                       'step' :'1m'})
eph = obj.ephemerides(quantities='1,3,20,23', extra_precision=True)

RA       = np.array(eph['RA'])        # degrees
Dec      = np.array(eph['DEC'])       # degrees
RA_rate  = np.array(eph['RA_rate'])   # arcsec/hr (×cos Dec)
DEC_rate = np.array(eph['DEC_rate'])  # arcsec/hr
elong    = np.array(eph['elong'])     # degrees
r_km     = np.array(eph['delta'])*AU_KM
times    = np.array(eph['datetime_str'])

# Periapsis: max kappa² = min r
idx_p = np.argmin(r_km)
print(f"Periapsis at: {times[idx_p]}")
print(f"  RA={RA[idx_p]:.4f}°, Dec={Dec[idx_p]:.4f}°, elong={elong[idx_p]:.4f}°")

# ============================================================
# STEP 1: ORBITAL PLANE NORMAL FROM PERIAPSIS — EXACT
# n̂ = r̂_p × v̂_p  (position × velocity direction at o=0)
# At periapsis β_R=0 → v̂_p exactly in the orbital plane.
# Both from OBSERVER at periapsis. No asymptote approximation.
# ============================================================
r_p_vec = sky_to_vec(RA[idx_p], Dec[idx_p])

# v̂_p from proper motion direction at periapsis
pm_RA_hat  = np.array([-np.sin(np.radians(RA[idx_p])),
                         np.cos(np.radians(RA[idx_p])), 0.0])
pm_Dec_hat = np.array([-np.sin(np.radians(Dec[idx_p]))*np.cos(np.radians(RA[idx_p])),
                        -np.sin(np.radians(Dec[idx_p]))*np.sin(np.radians(RA[idx_p])),
                         np.cos(np.radians(Dec[idx_p]))])
omega_mag = np.sqrt(RA_rate[idx_p]**2 + DEC_rate[idx_p]**2)
v_hat = (RA_rate[idx_p]*pm_RA_hat + DEC_rate[idx_p]*pm_Dec_hat) / omega_mag

n_hat = np.cross(r_p_vec, v_hat)
n_hat = n_hat / np.linalg.norm(n_hat)

print(f"\nOrbital plane normal n̂ from r̂_p × v̂_p:")
print(f"  n̂ = {n_hat}")
print(f"  v̂ · n̂ = {np.dot(v_hat, n_hat):.6f}  (0 = v exactly in plane)")
print(f"  Inclination to equatorial: {90-angle_between(n_hat,[0,0,1]):.2f}°")
print(f"  Inclination to ecliptic:   {angle_between(n_hat,[0,-np.sin(np.radians(23.44)),np.cos(np.radians(23.44))]):.2f}°")

# ============================================================
# STEP 2: PROJECT EARTH'S VELOCITY ONTO ORBITAL PLANE
# β_E,proj = |ê_E - (ê_E · n̂) n̂| × β_E
# Pure geometry: removes the out-of-plane component
# ============================================================
ê_E = earth_velocity_direction()
ê_E_proj = project_onto_plane(ê_E, n_hat)
scale_proj = np.linalg.norm(ê_E_proj)
beta_E_proj = beta_E * scale_proj

print(f"\nEarth velocity projection:")
print(f"  β_E       = {beta_E*c:.4f} km/s  (full)")
print(f"  β_E,proj  = {beta_E_proj*c:.4f} km/s  (in Juno orbital plane)")
print(f"  cos(i_eff) = {scale_proj:.4f}  (effective inclination factor)")

# ============================================================
# STEP 3: PROJECTED SUN DIRECTION AND θ_E,proj
# Sun RA/Dec from Earth at flyby: Oct 9 2013
# λ_sun ≈ 196° ecliptic → equatorial: RA_sun≈192°, Dec_sun≈-7°
# (approximate, can be fetched from Horizons too)
# ============================================================
# Sun's equatorial coords on Oct 9 2013 (geometric, no physics)
epsilon = np.radians(23.44)
lam_sun = np.radians(196.0)
sun_ecl = np.array([np.cos(lam_sun), np.sin(lam_sun), 0])
R = np.array([[1,0,0],[0,np.cos(epsilon),-np.sin(epsilon)],[0,np.sin(epsilon),np.cos(epsilon)]])
sun_eq = R @ sun_ecl   # Sun direction from Earth in equatorial

# Project Sun direction onto orbital plane
sun_proj = project_onto_plane(sun_eq, n_hat)

# Project Juno periapsis direction onto orbital plane (already ~in plane)
r_p_proj = project_onto_plane(r_p_vec, n_hat)

# θ_E,proj: angle between projected Sun direction and Juno periapsis direction
# in the orbital plane
theta_E_proj = angle_between(sun_proj, r_p_proj)

print(f"\nProjected angular quantities:")
print(f"  θ_E (3D elongation)  = {elong[idx_p]:.4f}°")
print(f"  θ_E,proj (in plane)  = {theta_E_proj:.4f}°")
print(f"  Difference           = {elong[idx_p]-theta_E_proj:.2f}°")

# v_hat already computed above from proper motion
v_along_n = np.dot(v_hat, n_hat)
print(f"\nVelocity direction consistency check:")
print(f"  v̂_Juno · n̂ = {v_along_n:.4f}  (should be ≈0 if v is in orbital plane)")

# True angle for composition: between Juno velocity direction and Earth velocity direction
theta_velocity_3D = angle_between(v_hat, ê_E)
theta_velocity_proj = angle_between(project_onto_plane(v_hat, n_hat),
                                    project_onto_plane(ê_E, n_hat))

print(f"  θ(v_Juno, v_Earth) 3D       = {theta_velocity_3D:.2f}°")
print(f"  θ(v_Juno, v_Earth) projected = {theta_velocity_proj:.2f}°")

# ============================================================
# STEP 5: ROM COMPOSITION IN PROJECTED 2D PLANE
# β_p is IN the orbital plane (verified by β_R≈0 at periapsis)
# β_E,proj is the projection of β_E into the plane
# θ = angle between Juno velocity and projected Earth velocity
# ============================================================

# ROM deflection [EXACT]
kp_sq  = R_sE/r_p_km
bp_sq  = beta_p**2         # β_p already contains κ_p² — do NOT add again
num = kp_sq*(1+bp_sq); den = 2*bp_sq - kp_sq*(1+bp_sq)
delta = 2*np.arcsin(np.clip(num/den,-1,1))
print(f"\nROM deflection δ = {np.degrees(delta):.4f}°")

# ============================================================
# STEP 5: ROM COMPOSITION IN PROJECTED 2D PLANE
#
# The heliocentric boost happens FAR from Earth where Juno
# moves at β_inf (asymptotic speed), NOT β_p (periapsis speed).
# β_inf from ROM energy invariant: β_inf² = β_p² - κ_p²
#
# Incoming/outgoing asymptote VELOCITY DIRECTIONS in orbital plane:
# At periapsis v̂_p bisects the supplement of the deflection angle.
# Rotating v̂_p by ±δ/2 in the orbital plane gives asymptote directions.
#   v̂_in  = v̂_p rotated by -δ/2 around n̂
#   v̂_out = v̂_p rotated by +δ/2 around n̂
# Pure geometry — no physics.
# ============================================================

# β_inf from ROM energy invariant [EXACT]
beta_inf_sq = beta_p**2 - kp_sq
beta_inf    = np.sqrt(max(beta_inf_sq, 0))
print(f"β_inf = {beta_inf*c:.4f} km/s  (from β_p² - κ_p²)")

# Asymptote velocity directions — rotate v̂_p by ±δ/2 around n̂
# Rodrigues rotation: v_rot = v·cos(θ) + (n̂×v)·sin(θ)  (since v̂_p ⊥ n̂)
half_delta = delta/2
n_cross_v = np.cross(n_hat, v_hat)   # unit vector in orbital plane ⊥ to v̂_p

v_hat_in  = v_hat*np.cos(half_delta) - n_cross_v*np.sin(half_delta)
v_hat_out = v_hat*np.cos(half_delta) + n_cross_v*np.sin(half_delta)

# Project asymptote directions onto orbital plane (already in plane, for safety)
v_in_proj_dir  = project_onto_plane(v_hat_in,  n_hat)
v_out_proj_dir = project_onto_plane(v_hat_out, n_hat)
ê_E_proj_dir   = project_onto_plane(ê_E, n_hat)

# Angles in orbital plane
theta_in_proj  = angle_between(v_in_proj_dir,  ê_E_proj_dir)
theta_out_proj = angle_between(v_out_proj_dir, ê_E_proj_dir)

print(f"θ_in  (asymptote vs Earth v, in plane) = {theta_in_proj:.4f}°")
print(f"θ_out (asymptote vs Earth v, in plane) = {theta_out_proj:.4f}°")
print(f"Difference = {abs(theta_out_proj - theta_in_proj):.4f}°  (should ≈ δ = {np.degrees(delta):.4f}°)")

# Composition: β_inf (not β_p) composed with β_E_proj
bi_proj  = np.sqrt(beta_inf**2 + beta_E_proj**2 + 2*beta_inf*beta_E_proj*np.cos(np.radians(theta_in_proj)))
bo_proj  = np.sqrt(max(beta_inf**2 + beta_E_proj**2 + 2*beta_inf*beta_E_proj*np.cos(np.radians(theta_out_proj)),0))

# Out-of-plane component of Earth's velocity (unchanged by flyby)
beta_E_oofp = beta_E * np.abs(np.dot(ê_E, n_hat))

# Full 3D heliocentric speeds
v_in_3D  = np.sqrt(bi_proj**2 + beta_E_oofp**2)
v_out_3D = np.sqrt(bo_proj**2 + beta_E_oofp**2)

print(f"\n{'='*55}")
print(f"ROM PREDICTION (2D projection, pure geometry)")
print(f"{'='*55}")
print(f"  β_inf          = {beta_inf*c:.4f} km/s")
print(f"  β_E,proj       = {beta_E_proj*c:.4f} km/s")
print(f"  β_E,out-plane  = {beta_E_oofp*c:.4f} km/s")
print(f"  θ_in  (plane)  = {theta_in_proj:.4f}°")
print(f"  θ_out (plane)  = {theta_out_proj:.4f}°")
print(f"  v_in  (proj)   = {bi_proj*c:.4f} km/s")
print(f"  v_out (proj)   = {bo_proj*c:.4f} km/s")
print(f"  v_in  (3D)     = {v_in_3D*c:.4f} km/s  (Horizons: 34.99 km/s)")
print(f"  v_out (3D)     = {v_out_3D*c:.4f} km/s  (Horizons: 38.95 km/s)")
print(f"\n  Boost (proj)   = {(bo_proj-bi_proj)*c:.4f} km/s")
print(f"  Boost (3D)     = {(v_out_3D-v_in_3D)*c:.4f} km/s")   # ← only c multiplication
print(f"  Observed       = 3.9 km/s")
print(f"  Diff           = {abs((v_out_3D-v_in_3D)*c - 3.9):.4f} km/s  ({abs((v_out_3D-v_in_3D)*c-3.9)/3.9*100:.2f}%)")

β_E = 9.936894e-05  (29.7901 km/s)
β_p = 4.975042e-05  (14.9148 km/s)

Fetching OBSERVER data for asymptote directions and periapsis...
Periapsis at: 2013-Oct-09 19:21
  RA=339.5244°, Dec=-35.9227°, elong=125.9976°

Orbital plane normal n̂ from r̂_p × v̂_p:
  n̂ = [ 0.27207507 -0.68048327  0.68038053]
  v̂ · n̂ = 0.000000  (0 = v exactly in plane)
  Inclination to equatorial: 42.87°
  Inclination to ecliptic:   26.50°

Earth velocity projection:
  β_E       = 29.7901 km/s  (full)
  β_E,proj  = 27.1040 km/s  (in Juno orbital plane)
  cos(i_eff) = 0.9098  (effective inclination factor)

Projected angular quantities:
  θ_E (3D elongation)  = 125.9976°
  θ_E,proj (in plane)  = 126.9730°
  Difference           = -0.98°

Velocity direction consistency check:
  v̂_Juno · n̂ = 0.0000  (should be ≈0 if v is in orbital plane)
  θ(v_Juno, v_Earth) 3D       = 53.08°
  θ(v_Juno, v_Earth) projected = 48.68°

ROM deflection δ = 41.2846°
β_inf = 10.3194 km/s  (from β_p² - κ_p²)
θ_in  (asymptote vs Ear

In [ ]:
"""
WILL R.O.M. — Juno Earth Flyby Boost
======================================
PURE SCALARS. NO TIME. NO DIRECTION. NO VECTORS.

Primary orbit: Juno-Sun (bound, elliptical, e_JS < 1)
O_o defined on the HELIOCENTRIC orbit where κ_SJ² = 2β_JS²

Earth is the anomaly that changes θ_E between phase O_o and -O_o.

Formula:
  β_JS(o) = β_E + β_J · cos(θ_E(o))
  Δβ_JS   = β_J · (cos(θ_E(-O_o)) - cos(θ_E(O_o)))

where θ_E(o) = angle at Earth between Earth-Sun ray and Earth-Juno ray
at heliocentric phase o.
"""

import numpy as np
from astroquery.jplhorizons import Horizons

c     = 299792.458
AU_KM = 1.495978707e8
SPDAY = 86400.0

# ============================================================
# ROM PRIMITIVES
# ============================================================
theta_sun=0.0046524; z_sun=2.1224e-6; T_E=3.15581498e7
a_moon=384400.0; T_m=27.321661*86400
R_sE  = 8*np.pi**2*a_moon**3/(T_m**2*c**2)
kappa2_S = 1-(1/(1+z_sun))**2
R_sS = (T_E/np.pi)*c*(np.sqrt(0.5*kappa2_S*np.sin(theta_sun)))**3
kappa2_E = kappa2_S*np.sin(theta_sun)
R_SE = R_sS/kappa2_E
beta_E = np.sqrt(R_sS/(2*R_SE))

print("ROM PRIMITIVES")
print(f"  R_sS = {R_sS*1000:.4f} m")
print(f"  R_sE = {R_sE*1e6:.4f} mm")
print(f"  β_E  = {beta_E*c:.4f} km/s")

# ============================================================
# FETCH JUNO HELIOCENTRIC STATE FROM HORIZONS
# Use Sun-centered vectors to get r_SJ and v_JS at any phase
# Then: κ_SJ² = R_sS/r_SJ, β_JS = v_JS/c
# β_J (Juno-Earth) from OBSERVER proper motion at periapsis
# ============================================================

def fetch_helio(start, stop, step='1d'):
    """Juno heliocentric scalars from Sun-centered vectors."""
    obj = Horizons(id='-61', location='500@10',
                   epochs={'start':start, 'stop':stop, 'step':step})
    vec = obj.vectors(refplane='ecliptic')
    x  = np.array(vec['x'])*AU_KM
    y  = np.array(vec['y'])*AU_KM
    z  = np.array(vec['z'])*AU_KM
    vx = np.array(vec['vx'])*(AU_KM/SPDAY)
    vy = np.array(vec['vy'])*(AU_KM/SPDAY)
    vz = np.array(vec['vz'])*(AU_KM/SPDAY)
    r_SJ = np.sqrt(x**2+y**2+z**2)
    v_JS = np.sqrt(vx**2+vy**2+vz**2)
    t    = np.array(vec['datetime_str'])
    return r_SJ, v_JS, t

def fetch_theta_E(start, stop, step='1d'):
    """θ_E = angle at Earth between Earth-Sun and Earth-Juno rays."""
    obj = Horizons(id='-61', location='500@399',
                   epochs={'start':start, 'stop':stop, 'step':step})
    eph = obj.ephemerides(quantities='20,23', extra_precision=True)
    elong = np.array(eph['elong'])
    r_EJ  = np.array(eph['delta'])*AU_KM
    t     = np.array(eph['datetime_str'])
    return elong, r_EJ, t

# ============================================================
# STEP 1: Get Juno heliocentric orbital parameters
# Use a wide window around the flyby to map the heliocentric orbit
# ============================================================
print("\nFetching Juno heliocentric orbit...")
# Juno launched Aug 2011. Heliocentric periapsis before Earth flyby Oct 2013.
# Need wide window to capture both sides of periapsis.
r_SJ, v_JS, t_helio = fetch_helio(
    '2012-01-01', '2014-06-01', step='7d')

kappa_SJ2 = R_sS / r_SJ
beta_JS   = v_JS / c

# Energy invariant W = (κ² - β²)/2 = const → semi-major axis
# W = R_sS/(4*a_JS) → a_JS = R_sS/(4*W)
# β_JS² = κ_SJ²/2 at circular equilibrium (O_o where κ²=2β²)
# Use energy invariant at any point:
# β² = κ²/2 - 2W → W = (κ² - 2β²)/4...
# Actually: W = β²/2 (from ROM, β is at semi-major axis)
# But here β_JS varies. Use: κ² - β² = 2W = const
W_arr = (kappa_SJ2 - beta_JS**2) / 2
W_mean = np.mean(W_arr)
print(f"  W (heliocentric energy invariant) = {W_mean:.6e}  (std={np.std(W_arr):.2e}, {np.std(W_arr)/abs(W_mean)*100:.1f}%)")
if np.std(W_arr)/abs(W_mean) > 0.05:
    print(f"  WARNING: W not conserved (>{5:.0f}% variation)")
    print(f"  Horizons trajectory contains maneuvers — W unreliable")
    print(f"  Using median W instead of mean")
    W_mean = np.median(W_arr)

beta_JS_sma = np.sqrt(2*W_mean)   # β at semi-major axis (κ²=2β²)
kappa_SJ_sma2 = 4*W_mean          # κ² at semi-major axis
r_SJ_sma = R_sS / kappa_SJ_sma2  # semi-major axis

print(f"  β_JS at semi-major axis = {beta_JS_sma*c:.4f} km/s")
print(f"  a_JS (semi-major axis)  = {r_SJ_sma/AU_KM:.4f} AU")

# Eccentricity from periapsis
# Need κ_p_JS² and β_p_JS at heliocentric periapsis
# Periapsis: max κ_SJ² (min r_SJ)
idx_p_helio = np.argmax(kappa_SJ2)
kappa_p_SJ2 = kappa_SJ2[idx_p_helio]
beta_p_SJ   = beta_JS[idx_p_helio]
r_p_SJ      = r_SJ[idx_p_helio]

e_JS = 2*beta_p_SJ**2/kappa_p_SJ2 - 1
O_o  = np.degrees(np.arccos(-e_JS))

print(f"\nHeliocentric orbit parameters:")
print(f"  r_p_SJ  = {r_p_SJ/AU_KM:.4f} AU  (perihelion)")
print(f"  β_p_SJ  = {beta_p_SJ*c:.4f} km/s")
print(f"  e_JS    = {e_JS:.6f}")
print(f"  O_o     = {O_o:.4f}°  (balance point where κ_SJ²=2β_JS²)")

# ============================================================
# STEP 2: Find κ² at O_o for heliocentric orbit
# At O_o: κ_SJ² = 2β_JS² and W = β_JS²/2
# From W: β_JS²(O_o) = W_mean, κ_SJ²(O_o) = 2*W_mean
# ============================================================
kappa_Oo_SJ2 = 2*W_mean     # = κ² at O_o
r_Oo_SJ      = R_sS/kappa_Oo_SJ2
print(f"\n  κ²(O_o)  = {kappa_Oo_SJ2:.4e}  (= 2β² = 2W)")
print(f"  r_SJ(O_o)= {r_Oo_SJ/AU_KM:.4f} AU  (heliocentric balance point)")

# ============================================================
# STEP 3: Find two epochs where κ_SJ² ≈ κ²(O_o)
# One on each side of periapsis
# ============================================================
# Approaching side (before periapsis)
kappa_app  = kappa_SJ2[:idx_p]
t_app      = t_helio[:idx_p]
print(f"\nDataset: {len(t_helio)} epochs, periapsis at idx={idx_p} ({t_helio[idx_p]})")
print(f"  Approaching side: {len(kappa_app)} points")
if len(kappa_app) > 0:
    print(f"  κ_SJ² range approaching: {kappa_app.min():.4e} to {kappa_app.max():.4e}")
print(f"  Target κ²(O_o) = {kappa_Oo_SJ2:.4e}")

t_Oo_app = None
t_Oo_rec = None

if len(kappa_app) > 0 and kappa_Oo_SJ2 <= kappa_app.max():
    idx_Oo_app = np.argmin(np.abs(kappa_app - kappa_Oo_SJ2))
    t_Oo_app   = t_app[idx_Oo_app]
    print(f"\nO_o (approaching): epoch ~ {t_Oo_app}")
    print(f"  κ_SJ² = {kappa_app[idx_Oo_app]:.4e}  (target: {kappa_Oo_SJ2:.4e})")
else:
    print(f"\nO_o (approaching): NOT in window — need earlier epochs")

# Receding side (after periapsis)
kappa_rec  = kappa_SJ2[idx_p+1:]
t_rec      = t_helio[idx_p+1:]
print(f"  Receding side: {len(kappa_rec)} points")

if len(kappa_rec) > 0 and kappa_Oo_SJ2 <= kappa_rec.max():
    idx_Oo_rec = np.argmin(np.abs(kappa_rec - kappa_Oo_SJ2))
    t_Oo_rec   = t_rec[idx_Oo_rec]
    print(f"\n-O_o (receding): epoch ~ {t_Oo_rec}")
    print(f"  κ_SJ² = {kappa_rec[idx_Oo_rec]:.4e}  (target: {kappa_Oo_SJ2:.4e})")
else:
    print(f"\n-O_o (receding): NOT in window — need later epochs")

# ============================================================
# STEP 4: Get θ_E at both O_o phases
# ============================================================
if t_Oo_app is None or t_Oo_rec is None:
    print("\nCannot compute boost — O_o phases not found in dataset.")
    print("Extend the heliocentric window and rerun.")
else:
    print(f"\nFetching θ_E at O_o phases...")
    elong_app, r_EJ_app, t_ea = fetch_theta_E(
        t_Oo_app, t_Oo_app, step='1d')
    elong_rec, r_EJ_rec, t_er = fetch_theta_E(
        t_Oo_rec, t_Oo_rec, step='1d')

    theta_E_Oo  = float(np.mean(elong_app))
    theta_E_mOo = float(np.mean(elong_rec))

    print(f"\nθ_E at O_o  = {theta_E_Oo:.4f}°")
    print(f"θ_E at -O_o = {theta_E_mOo:.4f}°")

    beta_J = beta_JS_sma

    beta_JS_Oo  = beta_E + beta_J*np.cos(np.radians(theta_E_Oo))
    beta_JS_mOo = beta_E + beta_J*np.cos(np.radians(theta_E_mOo))

    boost = (beta_JS_mOo - beta_JS_Oo)*c

    print(f"\n{'='*50}")
    print(f"ROM PREDICTION")
    print(f"{'='*50}")
    print(f"  β_J           = {beta_J*c:.4f} km/s")
    print(f"  β_E           = {beta_E*c:.4f} km/s")
    print(f"  θ_E(O_o)      = {theta_E_Oo:.4f}°")
    print(f"  θ_E(-O_o)     = {theta_E_mOo:.4f}°")
    print(f"  β_JS(O_o)     = {beta_JS_Oo*c:.4f} km/s")
    print(f"  β_JS(-O_o)    = {beta_JS_mOo*c:.4f} km/s")
    print(f"  Δβ_JS (boost) = {boost:.4f} km/s")
    print(f"  Observed      = 3.9 km/s")
    print(f"  Diff          = {abs(boost-3.9):.4f} km/s  ({abs(boost-3.9)/3.9*100:.2f}%)")

ROM PRIMITIVES
  R_sS = 2954.8427 m
  R_sE = 8.9548 mm
  β_E  = 29.7901 km/s

Fetching Juno heliocentric orbit...
  W (heliocentric energy invariant) = 2.675494e-09  (std=6.89e-10, 25.7%)
  Horizons trajectory contains maneuvers — W unreliable
  Using median W instead of mean
  β_JS at semi-major axis = 23.2821 km/s
  a_JS (semi-major axis)  = 1.6375 AU

Heliocentric orbit parameters:
  r_p_SJ  = 0.8815 AU  (perihelion)
  β_p_SJ  = 38.0711 km/s
  e_JS    = 0.439500
  O_o     = 116.0720°  (balance point where κ_SJ²=2β_JS²)

  κ²(O_o)  = 6.0312e-09  (= 2β² = 2W)
  r_SJ(O_o)= 3.2750 AU  (heliocentric balance point)

Dataset: 127 epochs, periapsis at idx=0 (A.D. 2012-Jan-01 00:00:00.0000)
  Approaching side: 0 points
  Target κ²(O_o) = 6.0312e-09

O_o (approaching): NOT in window — need earlier epochs
  Receding side: 126 points

-O_o (receding): epoch ~ A.D. 2014-Jun-01 00:00:00.0000
  κ_SJ² = 6.8373e-09  (target: 6.0312e-09)

Cannot compute boost — O_o phases not found in dataset.
Extend

In [ ]:
"""
WILL ROM N-Body: Anton's Physics Implemented
=============================================
Earth as relational origin. Sun-Earth axis = o=0.
No G. No M. No μ. No vectors. No time. No delta_rate.

Anton's formulas exactly as derived:

  β_Eo(o) = β_E · √(1+e_E²+2e_E·cos(o)) / √(1-e_E²)

  r_O(o)  = (1-e_J²) / (1 + e_J·cos(o-ω_E))       [normalised r_SJ/a_J]

  r_SJ(o) = a_J · r_O(o)

  β_Jo(o) = β_J · √(1+e_J²+2e_J·cos(o-ω_E)) / √(1-e_J²)

  β_Jo_total²(o) = β_Jo²(o) + κ_EJ²(o)·cos(θ_E(o))

  κ_EJ²(o) = R_sE / r_EJ(o)

  θ_E(o) = angle at Earth between Earth-Sun ray and Earth-Juno ray
           → directly from OBSERVER elong column

Boost = β_Jo_total(-O_o) - β_Jo_total(O_o)
where O_o = phase of heliocentric balance point κ_SJ² = 2β_JS²
"""

import numpy as np
from astroquery.jplhorizons import Horizons

c     = 299792.458
AU_KM = 1.495978707e8
SPDAY = 86400.0

# ============================================================
# ROM PRIMITIVES
# ============================================================
theta_sun = 0.0046524; z_sun = 2.1224e-6; T_E = 3.15581498e7
a_moon = 384400.0; T_m = 27.321661*86400
R_sE  = 8*np.pi**2*a_moon**3/(T_m**2*c**2)
kappa2_S = 1-(1/(1+z_sun))**2
R_sS  = (T_E/np.pi)*c*(np.sqrt(0.5*kappa2_S*np.sin(theta_sun)))**3
kappa2_E_val = kappa2_S*np.sin(theta_sun)
R_SE  = R_sS/kappa2_E_val
beta_E = np.sqrt(R_sS/(2*R_SE))
e_E   = 0.0167

print("ROM PRIMITIVES")
print(f"  R_sS   = {R_sS*1000:.4f} m")
print(f"  R_sE   = {R_sE*1e6:.4f} mm")
print(f"  β_E    = {beta_E*c:.4f} km/s")

# ============================================================
# STEP 1: OBSERVER DATA AT FLYBY
# Fetch θ_E(o), r_EJ(o), ω_p at each phase
# ============================================================
print("\nFetching OBSERVER data (flyby window)...")
obj = Horizons(id='-61', location='500@399',
               epochs={'start':'2013-10-09 19:00',
                       'stop' :'2013-10-09 19:45',
                       'step' :'1m'})
eph = obj.ephemerides(quantities='1,3,20,23', extra_precision=True)

r_EJ_km  = np.array(eph['delta'])*AU_KM
theta_E  = np.array(eph['elong'])
RA_rate  = np.array(eph['RA_rate'])
DEC_rate = np.array(eph['DEC_rate'])
omega_pm = np.sqrt(RA_rate**2 + DEC_rate**2)
times    = np.array(eph['datetime_str'])

kappa_EJ2 = R_sE / r_EJ_km

# β_p and β_J from proper motion at periapsis
idx_p   = np.argmax(kappa_EJ2)
r_p     = r_EJ_km[idx_p]
beta_p  = r_p * omega_pm[idx_p] / (3600*206264.806247) / c
kp_sq   = kappa_EJ2[idx_p]
beta_J_geo = np.sqrt(max(beta_p**2 - kp_sq, 0))   # Juno-Earth at asymptote

print(f"\nPeriapsis (o=0):")
print(f"  r_p      = {r_p:.2f} km")
print(f"  β_p      = {beta_p*c:.4f} km/s")
print(f"  β_J(geo) = {beta_J_geo*c:.4f} km/s  (Juno-Earth at large r)")
print(f"  θ_E(p)   = {theta_E[idx_p]:.4f}°")

# ============================================================
# STEP 2: JUNO HELIOCENTRIC ORBIT
# Fetch heliocentric state at clean epoch (away from maneuvers)
# ============================================================
print("\nFetching Juno heliocentric state (Sep 2013)...")
obj_h = Horizons(id='-61', location='500@10',
                 epochs={'start':'2013-09-01',
                         'stop' :'2013-09-10',
                         'step' :'1d'})
vec = obj_h.vectors(refplane='ecliptic')
vx  = np.array(vec['vx'])*(AU_KM/SPDAY)
vy  = np.array(vec['vy'])*(AU_KM/SPDAY)
vz  = np.array(vec['vz'])*(AU_KM/SPDAY)
x   = np.array(vec['x'])*AU_KM
y   = np.array(vec['y'])*AU_KM
z   = np.array(vec['z'])*AU_KM
r_SJ_arr = np.sqrt(x**2+y**2+z**2)
v_JS_arr = np.sqrt(vx**2+vy**2+vz**2)

kappa_SJ2_arr = R_sS/r_SJ_arr
beta_JS_arr   = v_JS_arr/c
W_arr = (kappa_SJ2_arr - beta_JS_arr**2)/2

# W should be constant — take median (maneuver-robust)
W_J = float(np.median(W_arr))
print(f"  W_J   = {W_J:.6e}  (std/mean = {np.std(W_arr)/abs(W_J)*100:.2f}%)")

# Heliocentric orbit parameters from W
a_J        = R_sS/(4*W_J)             # semi-major axis
beta_J_sma = np.sqrt(2*W_J)           # β_J at O_o (semi-major axis, κ²=2β²)

# Eccentricity: need periapsis r_p_helio
# r_p_helio from min r_SJ in dataset
idx_p_helio   = np.argmin(r_SJ_arr)
r_p_helio     = float(r_SJ_arr[idx_p_helio])
kappa_p_SJ2   = R_sS/r_p_helio
beta_p_helio2 = kappa_p_SJ2 - 2*W_J
e_J           = 2*beta_p_helio2/kappa_p_SJ2 - 1
O_o_deg       = np.degrees(np.arccos(np.clip(-e_J, -1, 1)))

print(f"  a_J   = {a_J/AU_KM:.4f} AU")
print(f"  e_J   = {e_J:.6f}")
print(f"  O_o   = {O_o_deg:.4f}°")
print(f"  β_J(O_o) = {beta_J_sma*c:.4f} km/s")

# ============================================================
# STEP 3: EARTH ORBITAL PHASE ω_E AT FLYBY
# From Sun position as seen from Earth → Earth heliocentric longitude
# ============================================================
print("\nFetching Sun position from Earth at flyby...")
obj_s = Horizons(id='10', location='500@399',
                 epochs={'start':'2013-10-09 19:21',
                         'stop' :'2013-10-09 19:22',
                         'step' :'1m'})
eph_s  = obj_s.ephemerides(quantities='1', extra_precision=True)
RA_sun = float(eph_s['RA'][0])
omega_E = np.radians((RA_sun + 180) % 360)   # Earth heliocentric longitude
print(f"  ω_E = {np.degrees(omega_E):.4f}°")

# ============================================================
# STEP 4: ANTON'S FORMULAS AT EACH PHASE
#
# The orbital phase index i maps to heliocentric phase o via:
# o = true anomaly of Juno in heliocentric orbit at each observation
# We use the phase ordering from the OBSERVER data (sorted by r_EJ)
# and compute β_Jo using cos(o-ω_E) where o increases from 0 at periapsis
# ============================================================

print(f"\n{'='*65}")
print("ANTON'S FORMULAS: β_Jo_total² = β_Jo² + κ_EJ²·cos(θ_E)")
print("="*65)
print(f"{'idx':>4} {'r_EJ':>9} {'β_Jo':>8} {'κ²cosθ×c²':>11} {'β_tot':>8} {'θ_E°':>8}")

# Map OBSERVER index to heliocentric true anomaly o
# At idx_p: o=0 (periapsis). Index increases with phase.
# Use the phase variable proportional to index relative to periapsis
n_pts = len(times)
beta_tot_arr = np.zeros(n_pts)

for i in range(n_pts):
    # Heliocentric true anomaly: approximate from index offset from periapsis
    # Δo per step estimated from angular frequency
    # Use orbit equation: r_SJ(o) = a_J*(1-e_J²)/(1+e_J*cos(o-ω_E))
    # We know r_SJ at flyby ≈ R_SE (Juno near Earth at 1 AU)
    # For the ±45min window, the heliocentric o changes very little
    # Use: o(i) ≈ ν_flyby + (i - idx_p) * β_J_sma*c / (a_J/AU_KM) * (1/60) * π/10800
    # Better: use orbit equation directly
    # r_SJ(i) ≈ R_SE (all points in window near Earth)
    # cos(o_i - ω_E) = (a_J*(1-e_J²)/R_SE - 1)/e_J  ← constant across window
    cos_o_eff = (a_J*(1-e_J**2)/R_SE - 1)/e_J
    cos_o_eff = np.clip(cos_o_eff, -1, 1)
    # Sign: before periapsis (approaching) → negative phase
    # After periapsis (receding) → positive phase
    sign = -1 if i < idx_p else 1
    o_eff = sign * np.arccos(cos_o_eff)

    # β_Jo(o): Anton's formula with cos(o-ω_E)
    cos_arg   = np.cos(o_eff)   # = cos(o_eff - ω_E) since ω_E absorbed in o_eff definition
    denom     = np.sqrt(1-e_J**2)
    beta_Jo_i = beta_J_sma * np.sqrt(max(1 + e_J**2 + 2*e_J*cos_arg, 0)) / denom

    # Earth correction: κ_EJ²(o)·cos(θ_E(o))
    correction = kappa_EJ2[i] * np.cos(np.radians(theta_E[i]))

    # Total
    beta_tot2 = beta_Jo_i**2 + correction
    beta_tot  = np.sqrt(max(beta_tot2, 0))*c
    beta_tot_arr[i] = beta_tot

    if i % 10 == 0 or i == idx_p:
        mark = " ← peri" if i == idx_p else ""
        print(f"{i:>4} {r_EJ_km[i]:>9.1f} {beta_Jo_i*c:>8.4f} {correction*c**2:>11.4f} {beta_tot:>8.4f} {theta_E[i]:>8.4f}{mark}")

# ============================================================
# STEP 5: BOOST
# Compare β_Jo_total at first and last data points
# (both far from Earth on approaching/receding sides)
# ============================================================
v_in  = beta_tot_arr[0]
v_out = beta_tot_arr[-1]
boost = v_out - v_in

print(f"\n{'='*55}")
print("RESULT")
print("="*55)
print(f"  β_JS at first point (approaching) = {v_in:.4f} km/s")
print(f"  β_JS at last point  (receding)    = {v_out:.4f} km/s")
print(f"  Boost = {boost:.4f} km/s")
print(f"  Observed = 3.9 km/s")
print(f"  Diff = {abs(boost-3.9):.4f} km/s  ({abs(boost-3.9)/3.9*100:.2f}%)")

# ============================================================
# STEP 6: VERIFICATION — β_Jo_total should follow the orbit
# Print the full profile showing how β_JS changes across the flyby
# ============================================================
print(f"\nFull β_JS profile across flyby:")
print(f"{'idx':>4} {'r_EJ km':>9} {'β_JS km/s':>11} {'θ_E°':>8}")
for i in range(n_pts):
    print(f"{i:>4} {r_EJ_km[i]:>9.1f} {beta_tot_arr[i]:>11.4f} {theta_E[i]:>8.4f}")

ROM PRIMITIVES
  R_sS   = 2954.8427 m
  R_sE   = 8.9548 mm
  β_E    = 29.7901 km/s

Fetching OBSERVER data (flyby window)...

Periapsis (o=0):
  r_p      = 6940.35 km
  β_p      = 14.9148 km/s
  β_J(geo) = 10.3194 km/s  (Juno-Earth at large r)
  θ_E(p)   = 125.9976°

Fetching Juno heliocentric state (Sep 2013)...
  W_J   = 3.139604e-09  (std/mean = 0.00%)
  a_J   = 1.5728 AU
  e_J   = 0.439511
  O_o   = 116.0727°
  β_J(O_o) = 23.7560 km/s

Fetching Sun position from Earth at flyby...
  ω_E = 15.1787°

ANTON'S FORMULAS: β_Jo_total² = β_Jo² + κ_EJ²·cos(θ_E)
 idx      r_EJ     β_Jo   κ²cosθ×c²    β_tot     θ_E°
   0   17359.0  34.7929     29.0249  35.2076  51.2419
  10   11096.9  34.7929     25.3492  35.1553  69.5423
  20    7018.4  34.7929    -55.2658  33.9894 118.8124
  21    6940.4  34.7929    -68.1570  33.7993 125.9976 ← peri
  30    9540.8  34.7929    -83.2524  33.5752 170.7228
  40   15522.0  34.7929    -46.4232  34.1193 153.5509

RESULT
  β_JS at first point (approaching) = 35.2076

In [ ]:
"""
WILL ROM N-Body: Anton's Physics Implemented
=============================================
Earth as relational origin. Sun-Earth axis = o=0.
No G. No M. No μ. No vectors. No time. No delta_rate.

Anton's formulas exactly as derived:

  β_Eo(o) = β_E · √(1+e_E²+2e_E·cos(o)) / √(1-e_E²)

  r_O(o)  = (1-e_J²) / (1 + e_J·cos(o-ω_E))       [normalised r_SJ/a_J]

  r_SJ(o) = a_J · r_O(o)

  β_Jo(o) = β_J · √(1+e_J²+2e_J·cos(o-ω_E)) / √(1-e_J²)

  β_Jo_total²(o) = β_Jo²(o) + κ_EJ²(o)·cos(θ_E(o))

  κ_EJ²(o) = R_sE / r_EJ(o)

  θ_E(o) = angle at Earth between Earth-Sun ray and Earth-Juno ray
           → directly from OBSERVER elong column

Boost = β_Jo_total(-O_o) - β_Jo_total(O_o)
where O_o = phase of heliocentric balance point κ_SJ² = 2β_JS²
"""

import numpy as np
from astroquery.jplhorizons import Horizons

c     = 299792.458
AU_KM = 1.495978707e8
SPDAY = 86400.0

# ============================================================
# ROM PRIMITIVES
# ============================================================
theta_sun = 0.0046524; z_sun = 2.1224e-6; T_E = 3.15581498e7
a_moon = 384400.0; T_m = 27.321661*86400
R_sE  = 8*np.pi**2*a_moon**3/(T_m**2*c**2)
kappa2_S = 1-(1/(1+z_sun))**2
R_sS  = (T_E/np.pi)*c*(np.sqrt(0.5*kappa2_S*np.sin(theta_sun)))**3
kappa2_E_val = kappa2_S*np.sin(theta_sun)
R_SE  = R_sS/kappa2_E_val
beta_E = np.sqrt(R_sS/(2*R_SE))
e_E   = 0.0167

print("ROM PRIMITIVES")
print(f"  R_sS   = {R_sS*1000:.4f} m")
print(f"  R_sE   = {R_sE*1e6:.4f} mm")
print(f"  β_E    = {beta_E*c:.4f} km/s")

# ============================================================
# STEP 1: OBSERVER DATA AT FLYBY
# Fetch θ_E(o), r_EJ(o), ω_p at each phase
# ============================================================
print("\nFetching OBSERVER data (flyby window)...")
obj = Horizons(id='-61', location='500@399',
               epochs={'start':'2013-10-09 19:00',
                       'stop' :'2013-10-09 19:45',
                       'step' :'1m'})
eph = obj.ephemerides(quantities='1,3,20,23', extra_precision=True)

r_EJ_km  = np.array(eph['delta'])*AU_KM
theta_E  = np.array(eph['elong'])
RA_rate  = np.array(eph['RA_rate'])
DEC_rate = np.array(eph['DEC_rate'])
omega_pm = np.sqrt(RA_rate**2 + DEC_rate**2)
times    = np.array(eph['datetime_str'])

kappa_EJ2 = R_sE / r_EJ_km

# β_p and β_J from proper motion at periapsis
idx_p   = np.argmax(kappa_EJ2)
r_p     = r_EJ_km[idx_p]
beta_p  = r_p * omega_pm[idx_p] / (3600*206264.806247) / c
kp_sq   = kappa_EJ2[idx_p]
beta_J_geo = np.sqrt(max(beta_p**2 - kp_sq, 0))   # Juno-Earth at asymptote

print(f"\nPeriapsis (o=0):")
print(f"  r_p      = {r_p:.2f} km")
print(f"  β_p      = {beta_p*c:.4f} km/s")
print(f"  β_J(geo) = {beta_J_geo*c:.4f} km/s  (Juno-Earth at large r)")
print(f"  θ_E(p)   = {theta_E[idx_p]:.4f}°")

# ============================================================
# STEP 2: JUNO HELIOCENTRIC ORBIT
# Fetch heliocentric state at clean epoch (away from maneuvers)
# ============================================================
print("\nFetching Juno heliocentric state (Sep 2013)...")
obj_h = Horizons(id='-61', location='500@10',
                 epochs={'start':'2013-09-01',
                         'stop' :'2013-09-10',
                         'step' :'1d'})
vec = obj_h.vectors(refplane='ecliptic')
vx  = np.array(vec['vx'])*(AU_KM/SPDAY)
vy  = np.array(vec['vy'])*(AU_KM/SPDAY)
vz  = np.array(vec['vz'])*(AU_KM/SPDAY)
x   = np.array(vec['x'])*AU_KM
y   = np.array(vec['y'])*AU_KM
z   = np.array(vec['z'])*AU_KM
r_SJ_arr = np.sqrt(x**2+y**2+z**2)
v_JS_arr = np.sqrt(vx**2+vy**2+vz**2)

kappa_SJ2_arr = R_sS/r_SJ_arr
beta_JS_arr   = v_JS_arr/c
W_arr = (kappa_SJ2_arr - beta_JS_arr**2)/2

# W should be constant — take median (maneuver-robust)
W_J = float(np.median(W_arr))
print(f"  W_J   = {W_J:.6e}  (std/mean = {np.std(W_arr)/abs(W_J)*100:.2f}%)")

# Heliocentric orbit parameters from W
a_J        = R_sS/(4*W_J)             # semi-major axis
beta_J_sma = np.sqrt(2*W_J)           # β_J at O_o (semi-major axis, κ²=2β²)

# Eccentricity: need periapsis r_p_helio
# r_p_helio from min r_SJ in dataset
idx_p_helio   = np.argmin(r_SJ_arr)
r_p_helio     = float(r_SJ_arr[idx_p_helio])
kappa_p_SJ2   = R_sS/r_p_helio
beta_p_helio2 = kappa_p_SJ2 - 2*W_J
e_J           = 2*beta_p_helio2/kappa_p_SJ2 - 1
O_o_deg       = np.degrees(np.arccos(np.clip(-e_J, -1, 1)))

print(f"  a_J   = {a_J/AU_KM:.4f} AU")
print(f"  e_J   = {e_J:.6f}")
print(f"  O_o   = {O_o_deg:.4f}°")
print(f"  β_J(O_o) = {beta_J_sma*c:.4f} km/s")

# ============================================================
# STEP 3: EARTH ORBITAL PHASE ω_E AT FLYBY
# From Sun position as seen from Earth → Earth heliocentric longitude
# ============================================================
print("\nFetching Sun position from Earth at flyby...")
obj_s = Horizons(id='10', location='500@399',
                 epochs={'start':'2013-10-09 19:21',
                         'stop' :'2013-10-09 19:22',
                         'step' :'1m'})
eph_s  = obj_s.ephemerides(quantities='1', extra_precision=True)
RA_sun = float(eph_s['RA'][0])
omega_E = np.radians((RA_sun + 180) % 360)   # Earth heliocentric longitude
print(f"  ω_E = {np.degrees(omega_E):.4f}°")

# ============================================================
# STEP 4: FETCH θ_E AND β_Jo_total AT O_o PHASES
# O_o before flyby: ~130 days before Oct 9, 2013 → June 2, 2013
# -O_o after flyby: ~130 days after  Oct 9, 2013 → Feb 15, 2014
# Use 3-day window around each epoch, 1-day step
# ============================================================

# Refine O_o epoch: solve r_SJ = a_J exactly
# ν_Oo = arccos(-e_J), mean anomaly M_Oo, time from periapsis t_Oo
nu_Oo  = np.arccos(np.clip(-e_J, -1, 1))
E_Oo   = 2*np.arctan(np.sqrt((1-e_J)/(1+e_J)) * np.tan(nu_Oo/2))
M_Oo   = E_Oo - e_J*np.sin(E_Oo)
T_J    = 2*np.pi*np.sqrt((a_J)**3 / (R_sS*c**2/2))   # from ROM: μ=R_s*c²/2
t_Oo_s = M_Oo/(2*np.pi) * T_J   # seconds from periapsis

import datetime
flyby_dt   = datetime.datetime(2013, 10, 9, 19, 21)
t_Oo_days  = t_Oo_s / 86400.0
epoch_Oo   = flyby_dt - datetime.timedelta(days=t_Oo_days)
epoch_mOo  = flyby_dt + datetime.timedelta(days=t_Oo_days)

print(f"\nO_o epochs:")
print(f"  t_Oo  = {t_Oo_days:.2f} days from periapsis")
print(f"  O_o   = {epoch_Oo.strftime('%Y-%m-%d')}  (approaching)")
print(f"  -O_o  = {epoch_mOo.strftime('%Y-%m-%d')}  (receding)")

def fetch_obs(epoch_dt, window_days=3):
    start = (epoch_dt - datetime.timedelta(days=window_days//2)).strftime('%Y-%m-%d')
    stop  = (epoch_dt + datetime.timedelta(days=window_days//2)).strftime('%Y-%m-%d')
    obj = Horizons(id='-61', location='500@399',
                   epochs={'start':start, 'stop':stop, 'step':'1d'})
    eph = obj.ephemerides(quantities='1,3,20,23', extra_precision=True)
    r_EJ  = np.array(eph['delta'])*AU_KM
    tE    = np.array(eph['elong'])
    omega = np.sqrt(np.array(eph['RA_rate'])**2 + np.array(eph['DEC_rate'])**2)
    return r_EJ, tE, omega

print("\nFetching OBSERVER data at O_o phases...")
r_EJ_Oo,  tE_Oo,  om_Oo  = fetch_obs(epoch_Oo)
r_EJ_mOo, tE_mOo, om_mOo = fetch_obs(epoch_mOo)

# Use median values (stable across the 3-day window)
r_EJ_Oo_val  = float(np.median(r_EJ_Oo))
r_EJ_mOo_val = float(np.median(r_EJ_mOo))
tE_Oo_val    = float(np.median(tE_Oo))
tE_mOo_val   = float(np.median(tE_mOo))

print(f"\nAt O_o  ({epoch_Oo.strftime('%Y-%m-%d')}):")
print(f"  r_EJ   = {r_EJ_Oo_val:.0f} km")
print(f"  θ_E    = {tE_Oo_val:.4f}°")
print(f"\nAt -O_o ({epoch_mOo.strftime('%Y-%m-%d')}):")
print(f"  r_EJ   = {r_EJ_mOo_val:.0f} km")
print(f"  θ_E    = {tE_mOo_val:.4f}°")

# ============================================================
# STEP 5: ANTON'S FORMULAS AT O_o PHASES
#
# At O_o: r_SJ = a_J → cos(o_Oo - ω_E) = -e_J (from orbit equation)
# β_Jo(O_o) = β_J · √(1+e_J²+2e_J·(-e_J)) / √(1-e_J²)
#           = β_J · √(1-e_J²) / √(1-e_J²)
#           = β_J
# So at the balance point, β_Jo = β_J exactly. Clean.
#
# κ_EJ²(O_o) = R_sE / r_EJ(O_o)
# Correction  = κ_EJ² · cos(θ_E)
# β_total²    = β_J² + κ_EJ²·cos(θ_E)
# ============================================================

# Verify β_Jo at O_o
cos_at_Oo = -e_J   # cos(o_Oo - ω_E) at balance point
beta_Jo_Oo_check = beta_J_sma * np.sqrt(max(1+e_J**2+2*e_J*cos_at_Oo, 0)) / np.sqrt(1-e_J**2)
print(f"\nVerification: β_Jo at O_o = {beta_Jo_Oo_check*c:.4f} km/s  (should = β_J = {beta_J_sma*c:.4f})")

# At O_o:
kappa_EJ2_Oo  = R_sE / r_EJ_Oo_val
correction_Oo = kappa_EJ2_Oo * np.cos(np.radians(tE_Oo_val))
beta_tot2_Oo  = beta_J_sma**2 + correction_Oo
beta_tot_Oo   = np.sqrt(max(beta_tot2_Oo, 0))*c

# At -O_o:
kappa_EJ2_mOo  = R_sE / r_EJ_mOo_val
correction_mOo = kappa_EJ2_mOo * np.cos(np.radians(tE_mOo_val))
beta_tot2_mOo  = beta_J_sma**2 + correction_mOo
beta_tot_mOo   = np.sqrt(max(beta_tot2_mOo, 0))*c

boost = beta_tot_mOo - beta_tot_Oo

print(f"\n{'='*55}")
print("RESULT: ANTON'S FORMULA AT O_o PHASES")
print("="*55)
print(f"  β_J           = {beta_J_sma*c:.4f} km/s  (at balance point)")
print(f"  β_E           = {beta_E*c:.4f} km/s")
print()
print(f"  At O_o:")
print(f"    r_EJ        = {r_EJ_Oo_val:.0f} km")
print(f"    κ_EJ²       = {kappa_EJ2_Oo:.4e}")
print(f"    θ_E         = {tE_Oo_val:.4f}°")
print(f"    correction  = κ_EJ²·cos(θ_E) = {correction_Oo*c**2:.6e}")
print(f"    β_JS(O_o)   = {beta_tot_Oo:.4f} km/s")
print()
print(f"  At -O_o:")
print(f"    r_EJ        = {r_EJ_mOo_val:.0f} km")
print(f"    κ_EJ²       = {kappa_EJ2_mOo:.4e}")
print(f"    θ_E         = {tE_mOo_val:.4f}°")
print(f"    correction  = κ_EJ²·cos(θ_E) = {correction_mOo*c**2:.6e}")
print(f"    β_JS(-O_o)  = {beta_tot_mOo:.4f} km/s")
print()
print(f"  Δβ_JS (boost) = {boost:.4f} km/s")
print(f"  Observed      = 3.9 km/s")
print(f"  Diff          = {abs(boost-3.9):.4f} km/s  ({abs(boost-3.9)/3.9*100:.2f}%)")

ROM PRIMITIVES
  R_sS   = 2954.8427 m
  R_sE   = 8.9548 mm
  β_E    = 29.7901 km/s

Fetching OBSERVER data (flyby window)...

Periapsis (o=0):
  r_p      = 6940.35 km
  β_p      = 14.9148 km/s
  β_J(geo) = 10.3194 km/s  (Juno-Earth at large r)
  θ_E(p)   = 125.9976°

Fetching Juno heliocentric state (Sep 2013)...
  W_J   = 3.139604e-09  (std/mean = 0.00%)
  a_J   = 1.5728 AU
  e_J   = 0.439511
  O_o   = 116.0727°
  β_J(O_o) = 23.7560 km/s

Fetching Sun position from Earth at flyby...
  ω_E = 15.1787°

O_o epochs:
  t_Oo  = 129.68 days from periapsis
  O_o   = 2013-06-02  (approaching)
  -O_o  = 2014-02-16  (receding)

Fetching OBSERVER data at O_o phases...

At O_o  (2013-06-02):
  r_EJ   = 80121766 km
  θ_E    = 113.1502°

At -O_o (2014-02-16):
  r_EJ   = 218836489 km
  θ_E    = 110.6390°

Verification: β_Jo at O_o = 23.7560 km/s  (should = β_J = 23.7560)

RESULT: ANTON'S FORMULA AT O_o PHASES
  β_J           = 23.7560 km/s  (at balance point)
  β_E           = 29.7901 km/s

  At O_o:

In [ ]:
"""
WILL ROM N-Body: Juno Flyby Boost
===================================
Anton's physics — integrated over the encounter.

The boost accumulates through the asymmetry of Earth's motion:
on approach Earth runs away from Juno, after periapsis Earth
chases Juno. The cos(o-ω_E) shift encodes this asymmetry.

β_Jo_total²(o) = β_Jo²(o) + κ_EJ²(o)·cos(θ_E(o))

where:
  β_Jo(o) = β_J · √(1+e_J²+2e_J·cos(o-ω_E)) / √(1-e_J²)
  κ_EJ²(o) = R_sE / r_EJ(o)
  θ_E(o) = OBSERVER elong — angle at Earth between Earth-Sun and Earth-Juno

Integration limits: O_o and -O_o (heliocentric balance phases)
At these phases Earth's influence vanishes → κ_EJ² negligible → clean boundary.

Boost = β_Jo_total(-O_o) - β_Jo_total(O_o)
"""

import numpy as np
from astroquery.jplhorizons import Horizons
import datetime

c     = 299792.458
AU_KM = 1.495978707e8
SPDAY = 86400.0

# ============================================================
# ROM PRIMITIVES
# ============================================================
theta_sun=0.0046524; z_sun=2.1224e-6; T_E=3.15581498e7
a_moon=384400.0; T_m=27.321661*86400
R_sE  = 8*np.pi**2*a_moon**3/(T_m**2*c**2)
kappa2_S = 1-(1/(1+z_sun))**2
R_sS  = (T_E/np.pi)*c*(np.sqrt(0.5*kappa2_S*np.sin(theta_sun)))**3
kappa2_E_val = kappa2_S*np.sin(theta_sun)
R_SE  = R_sS/kappa2_E_val
beta_E = np.sqrt(R_sS/(2*R_SE))
e_E   = 0.0167

print("ROM PRIMITIVES")
print(f"  R_sS = {R_sS*1000:.4f} m")
print(f"  R_sE = {R_sE*1e6:.4f} mm")
print(f"  β_E  = {beta_E*c:.4f} km/s")

# ============================================================
# STEP 1: JUNO HELIOCENTRIC ORBIT (pre-flyby, clean epoch)
# ============================================================
print("\nFetching Juno heliocentric orbit (Sep 2013)...")
obj_h = Horizons(id='-61', location='500@10',
                 epochs={'start':'2013-09-01',
                         'stop' :'2013-09-10',
                         'step' :'1d'})
vec = obj_h.vectors(refplane='ecliptic')
vx = np.array(vec['vx'])*(AU_KM/SPDAY)
vy = np.array(vec['vy'])*(AU_KM/SPDAY)
vz = np.array(vec['vz'])*(AU_KM/SPDAY)
x  = np.array(vec['x'])*AU_KM
y  = np.array(vec['y'])*AU_KM
z  = np.array(vec['z'])*AU_KM
r_SJ_arr = np.sqrt(x**2+y**2+z**2)
v_JS_arr = np.sqrt(vx**2+vy**2+vz**2)

kappa_SJ2_arr = R_sS/r_SJ_arr
beta_JS_arr   = v_JS_arr/c
W_arr = (kappa_SJ2_arr - beta_JS_arr**2)/2
W_J   = float(np.median(W_arr))

a_J        = R_sS/(4*W_J)
beta_J_sma = np.sqrt(2*W_J)         # β_J at O_o

# Eccentricity from periapsis
idx_p_h   = np.argmin(r_SJ_arr)
r_p_helio = float(r_SJ_arr[idx_p_h])
kp_SJ2    = R_sS/r_p_helio
e_J       = 2*(kp_SJ2 - 2*W_J)/kp_SJ2 - 1
# Cleaner: e_J = 1 - r_p/a
e_J       = 1 - r_p_helio/a_J
O_o_deg   = np.degrees(np.arccos(np.clip(-e_J, -1, 1)))

print(f"  W_J      = {W_J:.6e}")
print(f"  a_J      = {a_J/AU_KM:.4f} AU")
print(f"  e_J      = {e_J:.6f}")
print(f"  O_o      = {O_o_deg:.4f}°")
print(f"  β_J(O_o) = {beta_J_sma*c:.4f} km/s")

# ============================================================
# STEP 2: O_o EPOCH CALCULATION
# ============================================================
nu_Oo  = np.arccos(np.clip(-e_J, -1, 1))
E_Oo   = 2*np.arctan(np.sqrt((1-e_J)/(1+e_J))*np.tan(nu_Oo/2))
M_Oo   = E_Oo - e_J*np.sin(E_Oo)
T_J    = 2*np.pi*np.sqrt(a_J**3/(R_sS*c**2/2))
t_Oo_days = M_Oo/(2*np.pi)*T_J/SPDAY

flyby  = datetime.datetime(2013, 10, 9, 19, 21)
ep_Oo  = flyby - datetime.timedelta(days=t_Oo_days)
ep_mOo = flyby + datetime.timedelta(days=t_Oo_days)

print(f"\n  O_o  = {ep_Oo.strftime('%Y-%m-%d')}  ({t_Oo_days:.1f} days before flyby)")
print(f"  -O_o = {ep_mOo.strftime('%Y-%m-%d')}  ({t_Oo_days:.1f} days after flyby)")

# ============================================================
# STEP 3: FETCH OBSERVER DATA OVER FULL ENCOUNTER
# From O_o to -O_o: ~260 day window, 1-day step
# θ_E(o) and r_EJ(o) at every phase
# ============================================================
print(f"\nFetching OBSERVER data over full encounter ({ep_Oo.strftime('%Y-%m-%d')} to {ep_mOo.strftime('%Y-%m-%d')})...")

obj_enc = Horizons(id='-61', location='500@399',
                   epochs={'start': ep_Oo.strftime('%Y-%m-%d'),
                           'stop' : ep_mOo.strftime('%Y-%m-%d'),
                           'step' : '1d'})
eph_enc = obj_enc.ephemerides(quantities='1,3,20,23', extra_precision=True)

r_EJ_enc   = np.array(eph_enc['delta'])*AU_KM
theta_E_enc= np.array(eph_enc['elong'])
times_enc  = np.array(eph_enc['datetime_str'])
n_pts      = len(times_enc)
print(f"  {n_pts} data points")

# ============================================================
# STEP 4: EARTH ORBITAL PHASE ω_E AT EACH PHASE
# From Sun RA as seen from Earth at each epoch
# ============================================================
print(f"Fetching ω_E(o) at each phase...")
obj_sun = Horizons(id='10', location='500@399',
                   epochs={'start': ep_Oo.strftime('%Y-%m-%d'),
                           'stop' : ep_mOo.strftime('%Y-%m-%d'),
                           'step' : '1d'})
eph_sun = obj_sun.ephemerides(quantities='1', extra_precision=True)
RA_sun_arr = np.array(eph_sun['RA'])
omega_E_arr = np.radians((RA_sun_arr + 180) % 360)   # Earth heliocentric longitude

# ============================================================
# STEP 5: COMPUTE β_Jo_total²(o) AT EVERY PHASE
#
# β_Jo(o)        = β_J · √(1+e_J²+2e_J·cos(o-ω_E(o))) / √(1-e_J²)
# κ_EJ²(o)       = R_sE / r_EJ(o)
# β_Jo_total²(o) = β_Jo²(o) + κ_EJ²(o)·cos(θ_E(o))
#
# o here is the heliocentric phase of Juno.
# We derive it from r_EJ and the orbit geometry:
# r_SJ(o) = a_J·(1-e_J²)/(1+e_J·cos(o-ω_E))
# Since r_SJ ≈ r_EJ + r_SE·cos(angle) ≈ complex...
# Better: use the heliocentric orbit equation to find cos(o-ω_E)
# from known r_SJ at each epoch.
# r_SJ from Horizons heliocentric fetch at each epoch.
# ============================================================

print(f"Fetching r_SJ at each phase...")
obj_helio_enc = Horizons(id='-61', location='500@10',
                         epochs={'start': ep_Oo.strftime('%Y-%m-%d'),
                                 'stop' : ep_mOo.strftime('%Y-%m-%d'),
                                 'step' : '1d'})
vec_enc = obj_helio_enc.vectors(refplane='ecliptic')
x_enc   = np.array(vec_enc['x'])*AU_KM
y_enc   = np.array(vec_enc['y'])*AU_KM
z_enc   = np.array(vec_enc['z'])*AU_KM
r_SJ_enc = np.sqrt(x_enc**2+y_enc**2+z_enc**2)

# cos(o-ω_E) from orbit equation: r_SJ = a_J*(1-e_J²)/(1+e_J*cos(o-ω_E))
cos_o_enc = ((a_J*(1-e_J**2)/r_SJ_enc) - 1) / e_J
cos_o_enc = np.clip(cos_o_enc, -1, 1)

# β_Jo(o) at each phase
denom     = np.sqrt(1-e_J**2)
beta_Jo_enc = beta_J_sma * np.sqrt(np.maximum(1+e_J**2+2*e_J*cos_o_enc, 0)) / denom

# κ_EJ²(o) at each phase
kappa_EJ2_enc = R_sE / r_EJ_enc

# Correction term
cos_tE_enc    = np.cos(np.radians(theta_E_enc))
correction_enc= kappa_EJ2_enc * cos_tE_enc

# β_Jo_total²(o)
beta_tot2_enc = beta_Jo_enc**2 + correction_enc
beta_tot_enc  = np.sqrt(np.maximum(beta_tot2_enc, 0))*c

# ============================================================
# STEP 6: BOOST = β_tot(-O_o) - β_tot(O_o)
# ============================================================
beta_at_Oo  = beta_tot_enc[0]    # first point = O_o
beta_at_mOo = beta_tot_enc[-1]   # last point  = -O_o
boost = beta_at_mOo - beta_at_Oo

# ============================================================
# PRINT RESULTS
# ============================================================
print(f"\n{'='*65}")
print(f"ENCOUNTER PROFILE (every 20 days)")
print(f"{'='*65}")
print(f"{'date':>12} {'r_EJ Mkm':>10} {'r_SJ AU':>8} {'β_Jo':>8} {'κ²cosθ×c²':>12} {'β_tot':>8} {'θ_E°':>8}")
for i in range(0, n_pts, 20):
    print(f"{times_enc[i][:10]:>12} {r_EJ_enc[i]/1e6:>10.2f} {r_SJ_enc[i]/AU_KM:>8.4f} "
          f"{beta_Jo_enc[i]*c:>8.4f} {correction_enc[i]*c**2:>12.6f} "
          f"{beta_tot_enc[i]:>8.4f} {theta_E_enc[i]:>8.4f}")

# Highlight flyby epoch
idx_flyby = np.argmin(r_EJ_enc)
print(f"\nAt flyby (min r_EJ):")
print(f"  date     = {times_enc[idx_flyby][:10]}")
print(f"  r_EJ     = {r_EJ_enc[idx_flyby]/1e6:.4f} Mkm")
print(f"  β_Jo     = {beta_Jo_enc[idx_flyby]*c:.4f} km/s")
print(f"  κ²cosθ   = {correction_enc[idx_flyby]*c**2:.6f}")
print(f"  β_tot    = {beta_tot_enc[idx_flyby]:.4f} km/s")
print(f"  θ_E      = {theta_E_enc[idx_flyby]:.4f}°")

print(f"\n{'='*55}")
print(f"RESULT")
print(f"{'='*55}")
print(f"  β_JS(O_o)  = {beta_at_Oo:.6f} km/s  (at {times_enc[0][:10]})")
print(f"  β_JS(-O_o) = {beta_at_mOo:.6f} km/s  (at {times_enc[-1][:10]})")
print(f"  Boost      = {boost:.4f} km/s")
print(f"  Observed   = 3.9 km/s")
print(f"  Diff       = {abs(boost-3.9):.4f} km/s  ({abs(boost-3.9)/3.9*100:.2f}%)")

ROM PRIMITIVES
  R_sS = 2954.8427 m
  R_sE = 8.9548 mm
  β_E  = 29.7901 km/s

Fetching Juno heliocentric orbit (Sep 2013)...
  W_J      = 3.139604e-09
  a_J      = 1.5728 AU
  e_J      = 0.439511
  O_o      = 116.0727°
  β_J(O_o) = 23.7560 km/s

  O_o  = 2013-06-02  (129.7 days before flyby)
  -O_o = 2014-02-16  (129.7 days after flyby)

Fetching OBSERVER data over full encounter (2013-06-02 to 2014-02-16)...
  260 data points
Fetching ω_E(o) at each phase...
Fetching r_SJ at each phase...

ENCOUNTER PROFILE (every 20 days)
        date   r_EJ Mkm  r_SJ AU     β_Jo    κ²cosθ×c²    β_tot     θ_E°
  2013-Jun-0      80.12   1.3200  27.9385    -0.003949  27.9384 113.1502
  2013-Jun-2      80.35   1.1860  30.5367    -0.000784  30.5367  94.4901
  2013-Jul-1      75.83   1.0585  33.3586     0.001750  33.3586  80.5108
  2013-Aug-0      64.70   0.9527  36.0425     0.004387  36.0426  69.3519
  2013-Aug-2      47.71   0.8901  37.8155     0.008700  37.8156  58.9544
  2013-Sep-1      27.97   0.8895

In [ ]:
"""
WILL ROM N-Body: Juno Flyby Boost
===================================
Orbital invariant: β_J² = κ_SJ²(o) - β_JS²(o) = const

The flyby changes β_J²:
  Δ(β_J²) = ∫ κ_EJ²(o)·cos(θ_E(o)) · do  over close encounter

Post-flyby heliocentric speed at any phase:
  β_JS_post²(o) = κ_SJ²(o) - β_J_post²

Boost at flyby heliocentric phase (r_SJ ≈ R_SE):
  Δβ_JS = √(κ_SJ²(R_SE) - β_J_post²) - √(κ_SJ²(R_SE) - β_J_pre²)
"""

import numpy as np
from astroquery.jplhorizons import Horizons
import datetime

c     = 299792.458
AU_KM = 1.495978707e8
SPDAY = 86400.0

# ============================================================
# ROM PRIMITIVES
# ============================================================
theta_sun=0.0046524; z_sun=2.1224e-6; T_E=3.15581498e7
a_moon=384400.0; T_m=27.321661*86400
R_sE  = 8*np.pi**2*a_moon**3/(T_m**2*c**2)
kappa2_S = 1-(1/(1+z_sun))**2
R_sS  = (T_E/np.pi)*c*(np.sqrt(0.5*kappa2_S*np.sin(theta_sun)))**3
kappa2_E_val = kappa2_S*np.sin(theta_sun)
R_SE  = R_sS/kappa2_E_val
beta_E = np.sqrt(R_sS/(2*R_SE))

print("ROM PRIMITIVES")
print(f"  R_sS = {R_sS*1000:.4f} m")
print(f"  R_sE = {R_sE*1e6:.4f} mm")
print(f"  β_E  = {beta_E*c:.4f} km/s")

# ============================================================
# STEP 1: PRE-FLYBY ORBITAL INVARIANT β_J²
# β_J² = κ_SJ²(o) - β_JS²(o) = const  at any phase
# Use clean pre-flyby epoch (Sep 2013)
# ============================================================
print("\nFetching pre-flyby heliocentric state (Sep 2013)...")
obj_pre = Horizons(id='-61', location='500@10',
                   epochs={'start':'2013-09-01',
                           'stop' :'2013-09-10',
                           'step' :'1d'})
vec_pre = obj_pre.vectors(refplane='ecliptic')
vx = np.array(vec_pre['vx'])*(AU_KM/SPDAY)
vy = np.array(vec_pre['vy'])*(AU_KM/SPDAY)
vz = np.array(vec_pre['vz'])*(AU_KM/SPDAY)
x  = np.array(vec_pre['x'])*AU_KM
y  = np.array(vec_pre['y'])*AU_KM
z  = np.array(vec_pre['z'])*AU_KM
r_SJ_pre = np.sqrt(x**2+y**2+z**2)
v_JS_pre = np.sqrt(vx**2+vy**2+vz**2)

kappa_SJ2_pre = R_sS/r_SJ_pre
beta_JS_pre   = v_JS_pre/c
beta_J2_pre   = kappa_SJ2_pre - beta_JS_pre**2

beta_J2_mean  = float(np.median(beta_J2_pre))
print(f"  β_J²  = {beta_J2_mean:.6e}  (std/mean={np.std(beta_J2_pre)/abs(beta_J2_mean)*100:.3f}%)")
print(f"  β_J   = {np.sqrt(beta_J2_mean)*c:.4f} km/s  (pre-flyby invariant)")
print(f"  a_J   = {R_sS/(2*beta_J2_mean)/AU_KM:.4f} AU  (= R_sS/(2β_J²))")

# ============================================================
# STEP 2: CLOSE ENCOUNTER DATA
# Fine window around flyby: θ_E(o), κ_EJ²(o), κ_SJ²(o)
# 1-minute step to resolve the encounter properly
# ============================================================
print("\nFetching close encounter data (19:00-19:45, 1-min step)...")
obj_enc = Horizons(id='-61', location='500@399',
                   epochs={'start':'2013-10-09 19:00',
                           'stop' :'2013-10-09 19:45',
                           'step' :'1m'})
eph_enc = obj_enc.ephemerides(quantities='1,3,20,23', extra_precision=True)

r_EJ_enc   = np.array(eph_enc['delta'])*AU_KM
theta_E_enc= np.array(eph_enc['elong'])
RA_rate    = np.array(eph_enc['RA_rate'])
DEC_rate   = np.array(eph_enc['DEC_rate'])
times_enc  = np.array(eph_enc['datetime_str'])
n_pts      = len(times_enc)

kappa_EJ2_enc = R_sE / r_EJ_enc

# Heliocentric r_SJ during encounter (Juno near Earth → r_SJ ≈ R_SE)
# Fetch precisely
obj_h_enc = Horizons(id='-61', location='500@10',
                     epochs={'start':'2013-10-09 19:00',
                             'stop' :'2013-10-09 19:45',
                             'step' :'1m'})
vec_h_enc = obj_h_enc.vectors(refplane='ecliptic')
x_h  = np.array(vec_h_enc['x'])*AU_KM
y_h  = np.array(vec_h_enc['y'])*AU_KM
z_h  = np.array(vec_h_enc['z'])*AU_KM
r_SJ_enc = np.sqrt(x_h**2+y_h**2+z_h**2)
kappa_SJ2_enc = R_sS/r_SJ_enc

print(f"  {n_pts} data points")
print(f"  r_SJ range: {r_SJ_enc.min()/AU_KM:.4f} to {r_SJ_enc.max()/AU_KM:.4f} AU")
print(f"  r_EJ range: {r_EJ_enc.min():.0f} to {r_EJ_enc.max():.0f} km")

# ============================================================
# STEP 3: ACCUMULATE Δ(β_J²) OVER ENCOUNTER
#
# In ROM: β_J² = κ_SJ² - β_JS²
# Earth perturbs this invariant through κ_EJ²·cos(θ_E)
#
# At each phase step: dβ_J² = κ_EJ²·cos(θ_E) · dφ
#
# The phase element dφ corresponds to the heliocentric phase change.
# In the encounter: Earth orbital phase changes at rate ω_E = β_E·c/R_SE
# For each 1-minute step: dφ ≈ ω_E·dt = β_E·c/R_SE · dt
#
# But we want this in unitless ROM form:
# dφ = (β_E/R_SE) · c·dt = (β_E/R_SE) · dr_arc
# In the encounter window, the dominant phase variable is
# the angle Juno sweeps around Earth.
#
# The heliocentric orbital phase change per step:
# dθ_helio = β_Jo·c / r_SJ · dt  (angular rate at each step)
# Each 1-min step: dt = 60s
#
# Δ(β_J²) = Σ κ_EJ²(i)·cos(θ_E(i)) · dθ_helio(i)
# ============================================================

# Heliocentric angular rate: β_Jo·c / r_SJ at each step
# β_Jo from energy invariant: β_JS² = κ_SJ² - β_J²_pre
beta_JS_enc_sq = kappa_SJ2_enc - beta_J2_mean
beta_JS_enc    = np.sqrt(np.maximum(beta_JS_enc_sq, 0))

# Transverse heliocentric component from proper motion
# (needed for angular rate)
# β_Jo at each step from the orbit equation:
# Use actual heliocentric speed from vectors
obj_h_v = Horizons(id='-61', location='500@10',
                   epochs={'start':'2013-10-09 19:00',
                           'stop' :'2013-10-09 19:45',
                           'step' :'1m'})
vec_h_v = obj_h_v.vectors(refplane='ecliptic')
vx_h = np.array(vec_h_v['vx'])*(AU_KM/SPDAY)
vy_h = np.array(vec_h_v['vy'])*(AU_KM/SPDAY)
vz_h = np.array(vec_h_v['vz'])*(AU_KM/SPDAY)
v_JS_enc = np.sqrt(vx_h**2+vy_h**2+vz_h**2)  # heliocentric speed at each step

# Radial heliocentric velocity (range rate Sun-Juno)
r_hat_x = x_h/r_SJ_enc; r_hat_y = y_h/r_SJ_enc; r_hat_z = z_h/r_SJ_enc
v_R_SJ  = vx_h*r_hat_x + vy_h*r_hat_y + vz_h*r_hat_z
v_T_SJ  = np.sqrt(np.maximum(v_JS_enc**2 - v_R_SJ**2, 0))  # transverse heliocentric

# Angular rate around Sun: ω_SJ = v_T_SJ / r_SJ
omega_SJ = v_T_SJ / r_SJ_enc   # rad/s

# Phase step: dφ = ω_SJ · dt (1 min = 60 s)
dt = 60.0  # seconds
dphi = omega_SJ * dt  # radians per step

# Correction term at each step
cos_tE     = np.cos(np.radians(theta_E_enc))
correction = kappa_EJ2_enc * cos_tE   # unitless

# Accumulate Δ(β_J²)
delta_beta_J2 = np.sum(correction * dphi)

beta_J2_post = beta_J2_mean + delta_beta_J2

print(f"\n{'='*55}")
print(f"Δ(β_J²) ACCUMULATED OVER ENCOUNTER")
print(f"{'='*55}")
print(f"  β_J²_pre    = {beta_J2_mean:.6e}")
print(f"  Δ(β_J²)     = {delta_beta_J2:.6e}")
print(f"  β_J²_post   = {beta_J2_post:.6e}")
print(f"  Rel change  = {delta_beta_J2/beta_J2_mean*100:.4f}%")

# ============================================================
# STEP 4: BOOST AT FLYBY HELIOCENTRIC PHASE
# At r_SJ = R_SE (flyby): κ_SJ² = R_sS/R_SE = kappa2_E_val
# β_JS_pre² = κ_SJ²(R_SE) - β_J²_pre
# β_JS_post² = κ_SJ²(R_SE) - β_J²_post
# ============================================================
kappa_SJ2_flyby = R_sS/R_SE  # = kappa2_E_val

beta_JS_pre_sq  = kappa_SJ2_flyby - beta_J2_mean
beta_JS_post_sq = kappa_SJ2_flyby - beta_J2_post

beta_JS_pre_val  = np.sqrt(max(beta_JS_pre_sq,  0))*c
beta_JS_post_val = np.sqrt(max(beta_JS_post_sq, 0))*c
boost = beta_JS_post_val - beta_JS_pre_val

print(f"\n{'='*55}")
print(f"ROM PREDICTION: BOOST AT r_SJ = R_SE")
print(f"{'='*55}")
print(f"  κ_SJ²(R_SE)  = {kappa_SJ2_flyby:.6e}")
print(f"  β_JS_pre     = {beta_JS_pre_val:.4f} km/s")
print(f"  β_JS_post    = {beta_JS_post_val:.4f} km/s")
print(f"  Boost        = {boost:.4f} km/s")
print(f"  Observed     = 3.9 km/s")
print(f"  Diff         = {abs(boost-3.9):.4f} km/s  ({abs(boost-3.9)/3.9*100:.2f}%)")

# ============================================================
# DIAGNOSTIC: correction profile across encounter
# ============================================================
print(f"\nCorrection profile (κ_EJ²·cos(θ_E)·dφ at each step):")
print(f"{'step':>5} {'r_EJ km':>10} {'θ_E°':>8} {'κ²cosθ':>12} {'dφ':>10} {'contrib':>12}")
idx_p = np.argmin(r_EJ_enc)
for i in [0, 5, 10, idx_p-2, idx_p, idx_p+2, 30, 40, 45]:
    if 0 <= i < n_pts:
        mark = " ← peri" if i == idx_p else ""
        print(f"{i:>5} {r_EJ_enc[i]:>10.1f} {theta_E_enc[i]:>8.4f} "
              f"{correction[i]:>12.6e} {dphi[i]:>10.6f} {correction[i]*dphi[i]:>12.6e}{mark}")
print(f"\nTotal Δ(β_J²) = {delta_beta_J2:.6e}  (sum of contributions)")

ROM PRIMITIVES
  R_sS = 2954.8427 m
  R_sE = 8.9548 mm
  β_E  = 29.7901 km/s

Fetching pre-flyby heliocentric state (Sep 2013)...
  β_J²  = 6.279208e-09  (std/mean=0.002%)
  β_J   = 23.7560 km/s  (pre-flyby invariant)
  a_J   = 1.5728 AU  (= R_sS/(2β_J²))

Fetching close encounter data (19:00-19:45, 1-min step)...
  46 data points
  r_SJ range: 0.9986 to 0.9988 AU
  r_EJ range: 6940 to 18777 km

Δ(β_J²) ACCUMULATED OVER ENCOUNTER
  β_J²_pre    = 6.279208e-09
  Δ(β_J²)     = -2.664253e-13
  β_J²_post   = 6.278942e-09
  Rel change  = -0.0042%

ROM PREDICTION: BOOST AT r_SJ = R_SE
  κ_SJ²(R_SE)  = 1.974837e-08
  β_JS_pre     = 34.7929 km/s
  β_JS_post    = 34.7933 km/s
  Boost        = 0.0003 km/s
  Observed     = 3.9 km/s
  Diff         = 3.8997 km/s  (99.99%)

Correction profile (κ_EJ²·cos(θ_E)·dφ at each step):
 step    r_EJ km     θ_E°       κ²cosθ         dφ      contrib
    0    17359.0  51.2419 3.229452e-10   0.000014 4.506008e-15
    5    14141.0  58.3521 3.322670e-10   0.000014 4